# D1_heldout_probe__L4x4 — Pilot C

Held-out information-ladder probe (pre-registered: docs/PLAN_D1_heldout_probe.md; spec docs/D_SPECS.md on paper/main). Re-checks claims 1 and 3 on pilot D's held-out tasks: selects 24 of them by the frozen rule (issue statements only, no sympy), captures the agent's contexts with the real harness loop, then for 31b, e4b and 12b scores 15 depth-1 cells per task and draws 11 depth-2 samples per task. **Version switch:** the configuration cell has `DEPTH2 = True` (main version). Set it to `False` for the fallback version (depth 1 only); change nothing else. The last cell writes the final tables and figures.

- **Accelerator:** GPU L4 x4
- **Internet:** Off
- **Inputs to attach:** competition `gemma-4-developer-agent`; dataset `metric/gemma-4-developer-agent-wheelhouse`; models `google/gemma-4` → Other → `gemma-4-e4b-it-qat-w4a16-ct`, `gemma-4-12b-it-qat-w4a16-ct`, `gemma-4-31b-it-qat-w4a16-ct` (version 2; e2b is unused)
- **Earlier output needed:** **the output of `notebooks/D0b_heldout_export__CPU.ipynb`** (pilot D, branch `pilot/d-heldout`; Add input → Your Work → Notebooks → `D0b_heldout_export__CPU`; it provides `heldout_export/ and D0_summary.json`). Run those notebooks first.
- **Expected run time:** ≈ 1.55 L4x4 h main version (range 1.3–2.0; hard stop 2.2 h); ≈ 1.15 h fallback version, `DEPTH2 = False` (range 1.0–1.5; hard stop 1.6 h)
- **Bring back:** `/kaggle/working/D1_heldout_probe.json` and D_selection.json, D1_heldout_probe_31b.json, D1_heldout_probe_e4b.json, D1_heldout_probe_12b.json (one per size), and the analysis files written by the last cell: D1_analysis.json, D1_tables.md, D1_fig1_criteria.png, D1_fig2_depth2.png → commit to `results/` on branch `pilot/c-bottleneck`. Leave `/kaggle/working/private/` on Kaggle (competition-derived).
- The **GPU L4 x4** option only appears after the competition `gemma-4-developer-agent` is attached as an input.
- **Freeze protection:** every request to the inference server has a timeout, and a watchdog polls its `/health`. If the server stops responding (~90 s), the watchdog kills it, the next unit of work gets a fresh server, and the session stops after a second freeze. Freezes are listed under `freezes` in the output file.

**How to run it in the right environment.** These notebooks must run in the **official starter's pinned Kaggle environment (Python 3.12)**. New Kaggle notebooks default to Python 3.13, and pip then rejects the wheelhouse's cp312 wheels.

1. Open the official starter: https://www.kaggle.com/code/ryanholbrook/getting-started-gemma-4-developer-agent and click **Copy & Edit**. The copy keeps the starter's pinned environment (Python 3.12) and its inputs: the competition, the wheelhouse and the four Gemma 4 models.
2. In the copy: **File → Import Notebook** → upload `notebooks/D1_heldout_probe__L4x4.ipynb` from branch `pilot/c-bottleneck`. This replaces the cells; the environment and inputs stay.
3. Rename the notebook (title, top left) to `D1_heldout_probe__L4x4`.
4. Right panel → **Settings**: Environment = **Pin to original environment** (do not switch to "Always use latest environment"); Accelerator = **GPU L4 x4**; Internet = **Off**.
5. Inputs: Add input → Your Work → Notebooks → `D0b_heldout_export__CPU` (its output; pilot D's notebook on branch `pilot/d-heldout`). Keep the four Gemma 4 models that came with the copy.
6. **Save Version → Save & Run All**. The first code cell prints the Python version and stops within seconds if it is not 3.12. In that case the environment is not the starter's: redo steps 1–4.

In [ ]:
# Environment check (must stay the first code cell)
import sys
print('Python', sys.version)
EXPECTED = (3, 12)   # the official starter's pinned Kaggle environment; the wheelhouse is built for cp312
if sys.version_info[:2] != EXPECTED:
    raise RuntimeError(
        f'WRONG ENVIRONMENT: this session runs Python {sys.version_info[0]}.{sys.version_info[1]}, these notebooks need '
        f'Python {EXPECTED[0]}.{EXPECTED[1]}. Open the official starter (ryanholbrook/getting-started-gemma-4-developer-agent), '
        'click Copy & Edit, then File -> Import Notebook with this .ipynb, and keep Settings -> Environment = '
        '"Pin to original environment". Stopped before installing anything.')
print('environment OK')

In [ ]:
import os
os.makedirs('/kaggle/working/pilot/common', exist_ok=True)
os.makedirs('/kaggle/working/pilot/pilot_c', exist_ok=True)
os.makedirs('/kaggle/working/pilot/results', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/pilot/common/__init__.py
# (empty module)

In [ ]:
%%writefile /kaggle/working/pilot/common/diffs.py
"""Unified-diff parsing helpers (no task data, no pilot-specific logic).

Parses the `--- a/X / +++ b/X / @@ ... @@` diffs used in the competition's
`patch` and `test_patch` fields into per-file change records with old-side
line numbers, so callers can map changes onto the base snapshot.
"""
from __future__ import annotations

import re
from dataclasses import dataclass, field

_HUNK_RE = re.compile(r"^@@ -(\d+)(?:,(\d+))? \+(\d+)(?:,(\d+))? @@")

# Mirrors the scorer's notion of protected test files (HARNESS_README §8.1).
_TEST_DIR_PARTS = {"tests", "test", "testing"}


def is_test_path(path: str) -> bool:
    parts = path.split("/")
    name = parts[-1]
    if name.startswith("test_") and name.endswith(".py"):
        return True
    if name.endswith("_test.py"):
        return True
    return name.endswith(".py") and any(p in _TEST_DIR_PARTS for p in parts[:-1])


@dataclass
class Insertion:
    """Lines added after old line `after_old` (0 = at file start)."""

    after_old: int
    lines: list[str]


@dataclass
class FileDiff:
    old_path: str | None
    new_path: str | None
    removed_old_lines: list[int] = field(default_factory=list)  # old-side line numbers deleted/replaced
    insertions: list[Insertion] = field(default_factory=list)
    hunks: int = 0
    added: int = 0
    removed: int = 0
    all_hunks_from_zero: bool = True  # every hunk is "@@ -0,0 ..." (file absent on old side)

    @property
    def path(self) -> str:
        return self.new_path or self.old_path or ""

    @property
    def maybe_new_file(self) -> bool:
        return self.old_path is None or (self.hunks > 0 and self.all_hunks_from_zero)

    @property
    def deleted_file(self) -> bool:
        return self.new_path is None


def _strip_prefix(p: str) -> str | None:
    p = p.split("\t", 1)[0].strip()
    if p == "/dev/null":
        return None
    if p.startswith('"') and p.endswith('"'):
        p = p[1:-1]
    if p.startswith(("a/", "b/")):
        p = p[2:]
    return p


def parse_unified_diff(text: str) -> list[FileDiff]:
    """Parse a unified diff into FileDiff records (robust to missing `diff --git` headers)."""
    files: list[FileDiff] = []
    cur: FileDiff | None = None
    lines = text.splitlines()
    i = 0
    old_ln = new_ln = 0
    rem_old = rem_new = 0
    pending: Insertion | None = None

    def flush_pending() -> None:
        nonlocal pending
        if pending is not None and cur is not None:
            cur.insertions.append(pending)
        pending = None

    while i < len(lines):
        line = lines[i]
        in_hunk = rem_old > 0 or rem_new > 0
        if not in_hunk and line.startswith("--- ") and i + 1 < len(lines) and lines[i + 1].startswith("+++ "):
            flush_pending()
            cur = FileDiff(old_path=_strip_prefix(line[4:]), new_path=_strip_prefix(lines[i + 1][4:]))
            files.append(cur)
            i += 2
            continue
        if not in_hunk and line.startswith("@@ ") and cur is not None:
            flush_pending()
            m = _HUNK_RE.match(line)
            if m:
                o_start = int(m.group(1))
                o_cnt = int(m.group(2)) if m.group(2) is not None else 1
                n_cnt = int(m.group(4)) if m.group(4) is not None else 1
                cur.hunks += 1
                if not (o_start == 0 and o_cnt == 0):
                    cur.all_hunks_from_zero = False
                # old_ln = number of the last old line consumed so far
                old_ln = o_start - 1 if o_cnt > 0 else o_start
                rem_old, rem_new = o_cnt, n_cnt
            i += 1
            continue
        if in_hunk and cur is not None:
            if line.startswith("\\"):  # "\ No newline at end of file"
                i += 1
                continue
            tag = line[:1]
            body = line[1:]
            if tag == "+":
                if pending is None or pending.after_old != old_ln:
                    flush_pending()
                    pending = Insertion(after_old=old_ln, lines=[])
                pending.lines.append(body)
                cur.added += 1
                rem_new -= 1
            elif tag == "-":
                flush_pending()
                old_ln += 1
                cur.removed_old_lines.append(old_ln)
                cur.removed += 1
                rem_old -= 1
            else:  # context (' ' or empty line inside hunk)
                flush_pending()
                old_ln += 1
                rem_old -= 1
                rem_new -= 1
            i += 1
            continue
        i += 1
    flush_pending()
    return files


_ADDED_TEST_DEF = re.compile(r"^\+\s*(?:async\s+)?def\s+(test_[A-Za-z0-9_]+)\s*\(", re.M)
_REMOVED_TEST_DEF = re.compile(r"^-\s*(?:async\s+)?def\s+(test_[A-Za-z0-9_]+)\s*\(", re.M)
_HUNK_HEADER_TEST = re.compile(r"^@@[^@]+@@\s*(?:async\s+)?def\s+(test_[A-Za-z0-9_]+)\b", re.M)
_CONTEXT_TEST_DEF = re.compile(r"^ \s*(?:async\s+)?def\s+(test_[A-Za-z0-9_]+)\s*\(", re.M)


def required_test_names(test_patch: str) -> list[str]:
    """Test function names the scorer requires to pass, derived from test_patch.

    Re-implements the documented rule of the scorer (swegemma 0.2.11,
    `_extract_test_functions_from_patch`): added `def test_*` lines if any;
    otherwise test functions named in hunk headers / context, minus removed ones.
    Verified equal to the scorer's function on all 129 dev tasks (see
    pilot_c/check_test_names.py).
    """
    if not test_patch:
        return []
    added = _ADDED_TEST_DEF.findall(test_patch)
    if added:
        return list(dict.fromkeys(added))
    removed = set(_REMOVED_TEST_DEF.findall(test_patch))
    headers = _HUNK_HEADER_TEST.findall(test_patch)
    context = _CONTEXT_TEST_DEF.findall(test_patch)
    ctx = set(context)
    out = []
    for fn in dict.fromkeys(headers + context):
        if fn in removed:
            continue
        if fn not in ctx and (any(r.startswith(fn) for r in removed) or any(c != fn and c.startswith(fn) for c in ctx)):
            continue
        out.append(fn)
    return out

In [ ]:
%%writefile /kaggle/working/pilot/common/locate.py
"""Map diff changes onto enclosing Python scopes (def/class) of the base file.

Given the base-commit source of a file and a parsed FileDiff, returns the
innermost enclosing function/class for every changed region, using `ast` on
the base file. Deleted/replaced lines map by line number; inserted blocks map
by their anchor line plus indentation (an insertion indented less than a
scope's body belongs to the parent scope).
"""
from __future__ import annotations

import ast
from dataclasses import dataclass

from .diffs import FileDiff


@dataclass(frozen=True)
class Scope:
    qualname: str  # e.g. "APIRoute.__init__" (relative to module)
    kind: str  # "function" | "class"
    start: int  # first line incl. decorators
    def_line: int
    end: int
    body_indent: int
    depth: int
    header_indent: int = 0


def module_path(file_path: str) -> str:
    """'src/requests/models.py' -> 'requests.models'; 'pkg/__init__.py' -> 'pkg'."""
    p = file_path[:-3] if file_path.endswith(".py") else file_path
    parts = [x for x in p.split("/") if x]
    if parts and parts[0] in ("src", "lib"):
        parts = parts[1:]
    if parts and parts[-1] == "__init__":
        parts = parts[:-1]
    return ".".join(parts)


def collect_scopes(source: str) -> list[Scope]:
    tree = ast.parse(source)
    scopes: list[Scope] = []

    def visit(node: ast.AST, prefix: str, depth: int) -> None:
        for child in ast.iter_child_nodes(node):
            if isinstance(child, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
                q = f"{prefix}.{child.name}" if prefix else child.name
                start = min([child.lineno] + [d.lineno for d in child.decorator_list])
                body_indent = child.body[0].col_offset if child.body else child.col_offset + 4
                kind = "class" if isinstance(child, ast.ClassDef) else "function"
                scopes.append(Scope(q, kind, start, child.lineno, child.end_lineno or child.lineno, body_indent, depth, child.col_offset))
                visit(child, q, depth + 1)
            else:
                visit(child, prefix, depth)

    visit(tree, "", 0)
    return scopes


def _innermost(scopes: list[Scope], line: int) -> list[Scope]:
    """Scopes containing `line`, outermost first."""
    return sorted((s for s in scopes if s.start <= line <= s.end), key=lambda s: s.depth)


def _indent(s: str) -> int:
    return len(s) - len(s.lstrip(" \t"))


def report_scope(chain: list[Scope]) -> Scope | None:
    """Collapse a containment chain to what a developer would name:
    the outermost function (methods count as functions), or the innermost class
    if the change is in a class body outside any method."""
    if not chain:
        return None
    for s in chain:
        if s.kind == "function":
            return s
    return chain[-1]


def changed_scopes(source: str, fd: FileDiff) -> tuple[list[Scope], bool]:
    """Return (scopes touched by the diff, touches_module_level_code)."""
    scopes = collect_scopes(source)
    src_lines = source.splitlines()
    hits: dict[str, Scope] = {}
    module_level = False

    def last_code_line_at_or_before(n: int) -> int:
        while n >= 1 and (n > len(src_lines) or not src_lines[n - 1].strip()):
            n -= 1
        return n

    for ln in fd.removed_old_lines:
        s = report_scope(_innermost(scopes, ln))
        if s is None:
            module_level = True
        else:
            hits.setdefault(s.qualname, s)

    removed = set(fd.removed_old_lines)
    for ins in fd.insertions:
        code = [x for x in ins.lines if x.strip()]
        if not code:
            continue
        ind = _indent(code[0])
        if ins.after_old in removed:
            # replacement: the new lines take the place of removed ones, so they may
            # sit at a scope's header level (e.g. an edited decorator or signature)
            chain = [s for s in _innermost(scopes, ins.after_old) if s.header_indent <= ind]
        else:
            anchor = last_code_line_at_or_before(ins.after_old)
            chain = [s for s in _innermost(scopes, anchor) if s.body_indent <= ind] if anchor >= 1 else []
        s = report_scope(chain)
        if s is None:
            module_level = True
        else:
            hits.setdefault(s.qualname, s)

    ordered = sorted(hits.values(), key=lambda s: s.def_line)
    return ordered, module_level

In [ ]:
%%writefile /kaggle/working/pilot/common/tasks.py
"""Task loading and gold-location extraction (reusable; no pilot-specific logic).

Nothing here writes task data anywhere: callers decide where outputs go.
Competition data (tasks.jsonl, snapshots) must stay out of git.
"""
from __future__ import annotations

import json
import tarfile
from dataclasses import dataclass, field, asdict
from pathlib import Path

from .diffs import FileDiff, is_test_path, parse_unified_diff, required_test_names
from .locate import changed_scopes, module_path


def load_tasks(path: str | Path) -> list[dict]:
    out, seen = [], set()
    with open(path, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                t = json.loads(line)
                if t["instance_id"] not in seen:
                    seen.add(t["instance_id"])
                    out.append(t)
    return out


def read_snapshot_members(tgz: str | Path, wanted: set[str]) -> dict[str, str]:
    """Return {path: text} for the wanted repo-relative paths, reading the archive in one pass.
    Never extracts to disk and never executes anything from the archive."""
    found: dict[str, str] = {}
    want = set(wanted)
    with tarfile.open(tgz, mode="r|gz") as tf:
        for m in tf:
            name = m.name[2:] if m.name.startswith("./") else m.name
            if name in want and m.isfile():
                fh = tf.extractfile(m)
                if fh is not None:
                    found[name] = fh.read().decode("utf-8", errors="replace")
                if len(found) == len(want):
                    break
    return found


@dataclass
class GoldFunction:
    file: str
    qualname: str  # relative to module, e.g. "APIRoute.__init__"
    dotted: str  # e.g. "fastapi.routing.APIRoute.__init__" (graph-node style)
    kind: str  # "function" | "class"
    def_line: int


@dataclass
class GoldLocations:
    instance_id: str
    repo: str
    src_files: list[str] = field(default_factory=list)  # non-test .py files changed by the gold patch
    new_src_files: list[str] = field(default_factory=list)  # subset absent from the base snapshot
    other_files: list[str] = field(default_factory=list)  # non-.py or test files in the gold patch
    functions: list[GoldFunction] = field(default_factory=list)
    module_level_files: list[str] = field(default_factory=list)  # files with changes outside any def/class
    test_files: list[str] = field(default_factory=list)
    new_test_files: list[str] = field(default_factory=list)  # test files absent from the base snapshot
    test_names: list[str] = field(default_factory=list)  # scorer-required test functions
    gold_added: int = 0
    gold_removed: int = 0
    parse_errors: list[str] = field(default_factory=list)

    def to_dict(self) -> dict:
        return asdict(self)


def diff_scopes(diff_text: str, snapshot_tgz: str | Path, texts: dict[str, str] | None = None) -> list[str]:
    """Dotted names of the def/class scopes (in the base snapshot) changed by any diff,
    e.g. an agent's patch. Module-level changes are reported as '<module>:<path>'."""
    fds = [fd for fd in parse_unified_diff(diff_text) if fd.path.endswith(".py") and fd.old_path]
    if texts is None:
        texts = read_snapshot_members(snapshot_tgz, {fd.path for fd in fds})
    out: list[str] = []
    for fd in fds:
        if fd.path not in texts:
            continue
        try:
            scopes, mod_level = changed_scopes(texts[fd.path], fd)
        except SyntaxError:
            continue
        mod = module_path(fd.path)
        out += [f"{mod}.{s.qualname}" if mod else s.qualname for s in scopes]
        if mod_level:
            out.append(f"<module>:{fd.path}")
    return out


def gold_locations(task: dict, snapshot_tgz: str | Path | None) -> GoldLocations:
    """Derive gold file(s), enclosing function(s) and required test names for one task.

    Uses only *locations* from the gold patch (paths, line numbers) and test
    names from test_patch; the base-file AST comes from the snapshot.
    If snapshot_tgz is None, function-level fields stay empty.
    """
    g = GoldLocations(task["instance_id"], task["repo"])
    fdiffs: list[FileDiff] = parse_unified_diff(task["patch"])
    tdiffs: list[FileDiff] = parse_unified_diff(task["test_patch"])
    py_src = [fd for fd in fdiffs if fd.path.endswith(".py") and not is_test_path(fd.path)]
    g.src_files = list(dict.fromkeys(fd.path for fd in py_src))
    g.other_files = [fd.path for fd in fdiffs if fd not in py_src]
    g.gold_added = sum(fd.added for fd in fdiffs)
    g.gold_removed = sum(fd.removed for fd in fdiffs)
    g.test_files = list(dict.fromkeys(fd.path for fd in tdiffs if fd.new_path))
    g.test_names = required_test_names(task["test_patch"])

    if snapshot_tgz is None:
        return g
    wanted = {fd.path for fd in py_src if fd.old_path} | set(g.test_files)
    texts = read_snapshot_members(snapshot_tgz, wanted)
    g.new_test_files = [p for p in g.test_files if p not in texts]
    for fd in py_src:
        if fd.path not in texts:
            g.new_src_files.append(fd.path)
            continue
        try:
            scopes, mod_level = changed_scopes(texts[fd.path], fd)
        except SyntaxError as e:  # pragma: no cover - base files should parse
            g.parse_errors.append(f"{fd.path}: {e}")
            continue
        mod = module_path(fd.path)
        for s in scopes:
            g.functions.append(GoldFunction(fd.path, s.qualname, f"{mod}.{s.qualname}" if mod else s.qualname, s.kind, s.def_line))
        if mod_level:
            g.module_level_files.append(fd.path)
    return g

In [ ]:
%%writefile /kaggle/working/pilot/common/trace.py
"""Parse swegemma/adk-eval-core session traces into a flat event list.

Supports the ATIF v1.7 JSON written by `SessionTrace.save()` (the harness
default, `results_dir/traces/trace_<id>.json`) and the legacy
`{"entries": [...]}` format. Mirrors `SessionTrace.from_dict` in
adk-eval-core 0.1.0 without importing it, so this works on any machine.
"""
from __future__ import annotations

import json
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any


@dataclass
class Event:
    type: str  # tool_call | tool_response | thinking | text | final | continuation_nudge | compaction | ...
    author: str = ""
    elapsed: float = 0.0
    tool: str = ""
    args: dict[str, Any] = field(default_factory=dict)
    result: str = ""
    content: str = ""
    timestamp: float = 0.0  # absolute wall clock (s since epoch) when the harness recorded it
    meta: dict[str, Any] | None = None  # e.g. compaction start/end timestamps


@dataclass
class ToolCall:
    index: int  # order among tool calls in the task (0-based)
    tool: str
    args: dict[str, Any]
    author: str
    elapsed: float
    result: str | None = None  # raw response text (JSON string) if matched
    timestamp: float = 0.0

    def result_json(self) -> dict[str, Any] | None:
        if self.result is None:
            return None
        try:
            v = json.loads(self.result)
        except (ValueError, TypeError):
            return None
        # ADK sometimes wraps tool output as {"result": "<json string>"}
        if isinstance(v, dict) and set(v) == {"result"} and isinstance(v["result"], str):
            try:
                inner = json.loads(v["result"])
                if isinstance(inner, dict):
                    return inner
            except ValueError:
                pass
        return v if isinstance(v, dict) else None


def _atif_events(data: dict[str, Any]) -> list[Event]:
    evs: list[Event] = []
    for step in data.get("steps", []):
        source = step.get("source", "agent")
        msg = step.get("message")
        extra = step.get("extra") or {}
        author = extra.get("author", source)
        etype = extra.get("event_type")
        el = float(extra.get("elapsed_s", 0.0) or 0.0)
        ts = float(extra.get("timestamp", 0.0) or 0.0)
        if msg is not None or (etype is not None and not step.get("tool_calls") and not step.get("observation") and etype != "usage"):
            if not etype:
                etype = {"system": "system_instruction", "user": "task_prompt"}.get(source, "text")
            evs.append(Event(etype, author, el, content=str(msg) if msg is not None else "", timestamp=ts,
                             meta=extra.get("metadata")))
        for tc in step.get("tool_calls", []) or []:
            tx = tc.get("extra") or {}
            evs.append(Event("tool_call", tx.get("author", author), float(tx.get("elapsed_s", el) or 0.0),
                             tool=tc.get("function_name", ""), args=tc.get("arguments") or {},
                             timestamp=float(tx.get("timestamp", ts) or 0.0)))
        obs = step.get("observation")
        if obs:
            ox = obs.get("extra") or {}
            content = obs.get("content", "")
            if not isinstance(content, str):
                content = json.dumps(content, default=str)
            evs.append(Event("tool_response", ox.get("author") or "system", float(ox.get("elapsed_s", el) or 0.0),
                             tool=ox.get("tool_name", ""), result=content, timestamp=float(ox.get("timestamp", ts) or 0.0)))
    return evs


def _legacy_events(data: dict[str, Any]) -> list[Event]:
    return [
        Event(e.get("type") or "custom", e.get("author") or "", float(e.get("elapsed_s") or 0.0),
              tool=e.get("tool") or "", args=e.get("args") or {}, result=e.get("result") or "", content=e.get("content") or "",
              timestamp=float(e.get("timestamp") or 0.0), meta=e.get("metadata"))
        for e in data.get("entries", [])
    ]


def parse_trace(data: dict[str, Any]) -> list[Event]:
    if str(data.get("schema_version", "")).startswith("ATIF"):
        return _atif_events(data)
    return _legacy_events(data)


def load_trace(path: str | Path) -> list[Event]:
    with open(path, encoding="utf-8") as f:
        return parse_trace(json.load(f))


def pair_tool_calls(events: list[Event]) -> list[ToolCall]:
    """Match each tool_response to the earliest unanswered tool_call with the same tool name."""
    calls: list[ToolCall] = []
    pending: dict[str, list[ToolCall]] = {}
    for ev in events:
        if ev.type == "tool_call":
            c = ToolCall(len(calls), ev.tool, ev.args or {}, ev.author, ev.elapsed, timestamp=ev.timestamp)
            calls.append(c)
            pending.setdefault(ev.tool, []).append(c)
        elif ev.type == "tool_response":
            q = pending.get(ev.tool)
            if q:
                q.pop(0).result = ev.result
    return calls

In [ ]:
%%writefile /kaggle/working/pilot/common/metrics.py
"""Per-task and aggregate metrics shared by all pilots.

Common metric set (identical definitions in every pilot):
  resolved          scorer verdict (Phase 2 pytest + JUnit validation) AND non-empty patch, as in the
                    scorer, which never verifies an empty prediction (resolved_local = raw harness flag)
  wall_s            TaskResult.duration_seconds (setup + agent + verification)
  agent_s           last event timestamp in the trace (time inside the agent loop)
  tool_calls        tool_call events in the ROOT trace. Calls made inside an AgentTool sub-agent
                    (e.g. code_analyzer_agent) run in a separate ADK Runner and are NOT in the
                    trace; only the AgentTool call itself is (verified in google-adk 1.36.1
                    tools/agent_tool.py; pilot A found this first). Localization metrics are
                    therefore lower bounds when the agent delegates reading to the sub-agent.
  budget_tool_calls harness-counted calls (TaskResult.tool_calls; budget-gated tools only)
  repeat_share      share of tool calls identical (same tool + canonical JSON args) to an
                    earlier call in the same task
  malformed_share   malformed / (tool_calls + unparsed_tool_calls); malformed = unknown tool,
                    missing/unexpected arguments, or argument validation errors; unparsed =
                    turns cut off inside an unclosed <|tool_call> (harness nudge type 1)
  failed_edits      edit_file / write_file calls that returned status=error
  skill_calls       calls to ADK skill tools (list/search/load_skill, load_skill_resource,
                    run_skill_script)
Optional localization metrics (need gold locations, see common.tasks.gold_locations):
  viewed_gold_file, first_gold_view_call, edited_gold_file, edited_gold_function
"""
from __future__ import annotations

import json
import re
import statistics
from typing import Any, Iterable

from .diffs import parse_unified_diff
from .trace import Event, ToolCall, pair_tool_calls

SKILL_TOOLS = {"list_skills", "search_skills", "load_skill", "load_skill_resource", "run_skill_script"}
EDIT_TOOLS = {"edit_file", "write_file"}
_MALFORMED_TYPES = {"TypeError", "ValidationError", "KeyError", "JSONDecodeError", "ArgumentError"}
_MALFORMED_PATTERNS = (
    re.compile(r"Tool '.*' not found", re.S),
    re.compile(r"mandatory input parameters are not present"),
    re.compile(r"unexpected keyword argument"),
    re.compile(r"missing \d+ required (positional|keyword)"),
    re.compile(r"cannot be greater than end_line"),
)
_UNPARSED_NUDGE = "tool call finished closing"


def canonical_call(c: ToolCall) -> str:
    return c.tool + "\x00" + json.dumps(c.args, sort_keys=True, default=str)


def _error_info(c: ToolCall) -> tuple[bool, str, str]:
    r = c.result_json()
    if r is None:
        return False, "", ""
    if r.get("status") == "error":
        return True, str(r.get("error_type", "")), str(r.get("error_message") or r.get("error") or "")
    return False, "", ""


def is_malformed(c: ToolCall) -> bool:
    is_err, etype, msg = _error_info(c)
    if not is_err:
        return False
    if etype in _MALFORMED_TYPES:
        return True
    return any(p.search(msg) for p in _MALFORMED_PATTERNS)


def call_touches_path(c: ToolCall, path: str) -> bool:
    """True if a tool call reads/views `path` (read_file, or a shell command naming it)."""
    if c.tool == "read_file":
        fp = str(c.args.get("filepath", "")).lstrip("/")
        fp = fp[len("workspace/"):] if fp.startswith("workspace/") else fp
        return fp == path or fp.endswith("/" + path)
    if c.tool == "run_command":
        return path in str(c.args.get("command", ""))
    return False


def task_metrics(
    result: dict[str, Any],
    events: list[Event],
    gold: dict[str, Any] | None = None,
    agent_patch_scopes: Iterable[str] | None = None,
) -> dict[str, Any]:
    """Compute the common metric row for one task (root-agent calls only, see module doc).

    result: a TaskResult row (task_results.jsonl line or TaskResult.model_dump()).
    events: parsed trace (common.trace.load_trace / parse_trace).
    gold:   optional GoldLocations.to_dict() for localization metrics.
    agent_patch_scopes: optional dotted names of scopes the agent's patch changed.
    The returned row holds numbers/flags only (no patch, no trace text).
    """
    calls = pair_tool_calls(events)
    n = len(calls)
    seen: set[str] = set()
    repeats = 0
    for c in calls:
        k = canonical_call(c)
        if k in seen:
            repeats += 1
        seen.add(k)
    unparsed = sum(1 for e in events if e.type == "continuation_nudge" and _UNPARSED_NUDGE in e.content)
    nudges = sum(1 for e in events if e.type == "continuation_nudge")
    malformed = sum(1 for c in calls if is_malformed(c))
    edit_calls = [c for c in calls if c.tool in EDIT_TOOLS]
    failed_edits = sum(1 for c in edit_calls if _error_info(c)[0])
    by_tool: dict[str, int] = {}
    for c in calls:
        by_tool[c.tool] = by_tool.get(c.tool, 0) + 1
    agent_s = max((e.elapsed for e in events), default=0.0)
    patch = result.get("agent_patch") or ""
    patch_files = [fd.path for fd in parse_unified_diff(patch)] if patch else []

    row: dict[str, Any] = {
        "instance_id": result.get("task_id") or result.get("instance_id"),
        # The scorer only verifies NON-EMPTY predictions (empty -> "NO_PATCH", unresolved), while the local
        # swegemma Evaluator also verifies empty patches; a task whose tests already pass at base_commit then
        # counts as resolved locally. `resolved` follows the scorer; `resolved_local` is the raw harness flag.
        "resolved": bool(result.get("resolved")) and bool((result.get("agent_patch") or "").strip()),
        "resolved_local": bool(result.get("resolved")),
        "error": (result.get("error_message") or "")[:120] or None,
        "wall_s": round(float(result.get("duration_seconds") or 0.0), 1),
        "agent_s": round(agent_s, 1),
        "tool_calls": n,
        "budget_tool_calls": int(result.get("tool_calls") or 0),
        "llm_calls": int(result.get("total_llm_calls") or 0),
        "total_tokens": int(result.get("total_tokens") or 0),
        "repeat_calls": repeats,
        "repeat_share": round(repeats / n, 4) if n else 0.0,
        "malformed_calls": malformed,
        "unparsed_tool_calls": unparsed,
        "malformed_share": round((malformed + unparsed) / (n + unparsed), 4) if (n + unparsed) else 0.0,
        "edit_calls": len(edit_calls),
        "failed_edits": failed_edits,
        "skill_calls": sum(1 for c in calls if c.tool in SKILL_TOOLS),
        "nudges": nudges,
        "submitted": any(c.tool == "submit_patch" for c in calls),
        "patch_nonempty": bool(patch.strip()),
        "patch_files": len(patch_files),
        "calls_by_tool": by_tool,
    }
    if gold is not None:
        gfiles = list(gold.get("src_files") or [])
        first = next((c.index for c in calls if any(call_touches_path(c, g) for g in gfiles)), None)
        row["viewed_gold_file"] = first is not None
        row["first_gold_view_call"] = first
        row["edited_gold_file"] = bool(set(patch_files) & set(gfiles))
        row["edited_only_gold_files"] = bool(patch_files) and set(patch_files) <= set(gfiles)
        if agent_patch_scopes is not None:
            gfun = {f["dotted"] for f in gold.get("functions") or []}
            row["edited_gold_function"] = bool(gfun & set(agent_patch_scopes)) if gfun else None
    return row


def _mean(xs: list[float]) -> float | None:
    return round(statistics.fmean(xs), 3) if xs else None


def _median(xs: list[float]) -> float | None:
    return round(statistics.median(xs), 3) if xs else None


def aggregate(rows: list[dict[str, Any]]) -> dict[str, Any]:
    """Aggregate common metrics over a list of task rows (one condition/run)."""
    n = len(rows)
    tot_calls = sum(r["tool_calls"] for r in rows)
    tot_unparsed = sum(r["unparsed_tool_calls"] for r in rows)
    out: dict[str, Any] = {
        "n_tasks": n,
        "resolved": sum(r["resolved"] for r in rows),
        "resolve_rate": round(sum(r["resolved"] for r in rows) / n, 4) if n else None,
        "wall_s_mean": _mean([r["wall_s"] for r in rows]),
        "wall_s_median": _median([r["wall_s"] for r in rows]),
        "agent_s_mean": _mean([r["agent_s"] for r in rows]),
        "tool_calls_mean": _mean([r["tool_calls"] for r in rows]),
        # pooled shares (calls across all tasks) and per-task means
        "repeat_share_pooled": round(sum(r["repeat_calls"] for r in rows) / tot_calls, 4) if tot_calls else 0.0,
        "malformed_share_pooled": round((sum(r["malformed_calls"] for r in rows) + tot_unparsed) / (tot_calls + tot_unparsed), 4)
        if (tot_calls + tot_unparsed) else 0.0,
        "repeat_share_task_mean": _mean([r["repeat_share"] for r in rows]),
        "malformed_share_task_mean": _mean([r["malformed_share"] for r in rows]),
        "failed_edits_total": sum(r["failed_edits"] for r in rows),
        "edit_calls_total": sum(r["edit_calls"] for r in rows),
        "skill_calls_total": sum(r["skill_calls"] for r in rows),
        "submitted_share": round(sum(r["submitted"] for r in rows) / n, 4) if n else None,
        "patch_nonempty_share": round(sum(r["patch_nonempty"] for r in rows) / n, 4) if n else None,
        "errors": sum(1 for r in rows if r.get("error")),
    }
    for k in ("viewed_gold_file", "edited_gold_file", "edited_only_gold_files", "edited_gold_function"):
        vals = [r[k] for r in rows if r.get(k) is not None]
        if vals:
            out[f"{k}_share"] = round(sum(vals) / len(vals), 4)
    fg = [r["first_gold_view_call"] for r in rows if r.get("first_gold_view_call") is not None]
    if fg:
        out["first_gold_view_call_median"] = _median(fg)
    return out


def paired_flips(a: list[dict[str, Any]], b: list[dict[str, Any]]) -> dict[str, Any]:
    """Paired comparison of two runs over the same tasks (b vs a).

    Returns discordant counts and an exact two-sided McNemar (binomial) p-value.
    """
    ra = {r["instance_id"]: r["resolved"] for r in a}
    rb = {r["instance_id"]: r["resolved"] for r in b}
    common = sorted(set(ra) & set(rb))
    gain = sum(1 for t in common if rb[t] and not ra[t])
    loss = sum(1 for t in common if ra[t] and not rb[t])
    m = gain + loss
    if m == 0:
        p = 1.0
    else:
        from math import comb
        k = min(gain, loss)
        p = min(1.0, 2 * sum(comb(m, i) for i in range(k + 1)) / 2 ** m)
    return {"n_common": len(common), "b_only": gain, "a_only": loss, "both": sum(1 for t in common if ra[t] and rb[t]),
            "delta": (sum(rb[t] for t in common) - sum(ra[t] for t in common)), "mcnemar_exact_p": round(p, 4)}

In [ ]:
%%writefile /kaggle/working/pilot/common/runtime_probes.py
"""Runtime instrumentation for swegemma / google-adk runs (generic; counts only, no content saved).

Why: in google-adk 1.36.1 token-threshold compaction runs as a request processor before a model
call (flows/llm_flows/compaction.py) and appends its event to the session WITHOUT yielding it, so
the harness SessionTrace (fed by yielded events) is expected to miss it. The summarizer
(apps/llm_event_summarizer.py) formats only `part.text` of the compacted events, so function
calls/responses (incl. load_skill outputs) are dropped, and the initial task prompt is compacted
like any other event (all but the last `event_retention_size` events).

This module measures that directly:
  REQUESTS     one record per LLM request: messages by role, tool messages, whether the task
               prompt ("Problem Statement:") and each tracked marker string are still present
  COMPACTIONS  one record per summarizer call: events compacted, function calls / responses
               dropped, text vs non-text chars, whether the task prompt was compacted, which
               tracked markers were present before and survive in the summary
  SUMMARIES    per task, the text of every compaction summary (visible parts only) with how much of the
               task statement it keeps. TEXT: competition-derived, write it only under private/
               (dump_summaries); everything else here is counts.
Per request it also records where the task-prompt marker sits (a compaction summary = a message containing a
recorded summary, which ADK shows as a user message "For context: [model] said: ..."; the original prompt = another user
message holding the task statement; other),
the requested output tokens, the prompt tokens the server reports, and the exception type if it failed.
Task attribution: a ContextVar set around Evaluator.evaluate_task.

Usage: install(); set MARKERS[task_id] = [strings to track]; run; read REQUESTS / COMPACTIONS.
"""
from __future__ import annotations

import contextvars
import json
import os
import re
import time
from collections import Counter
from typing import Any

CURRENT_TASK: contextvars.ContextVar[str | None] = contextvars.ContextVar("pilot_current_task", default=None)
REQUESTS: list[dict[str, Any]] = []
COMPACTIONS: list[dict[str, Any]] = []
MARKERS: dict[str, list[str]] = {}
SUMMARIES: dict[str, list[dict[str, Any]]] = {}
PROBLEM: dict[str, str] = {}          # task statement as given to the agent (set from the task in evaluate_task)
TASK_PROMPT_MARKER = "Problem Statement:"
_FP_CHARS = 160
AGENT_REQUEST_TIMEOUT_S = 900
SKILL_RESPONSE_TOOLS = {"load_skill", "load_skill_resource"}
_installed = False


def _msg_text(m: Any) -> str:
    c = m.get("content") if isinstance(m, dict) else getattr(m, "content", None)
    if c is None:
        return ""
    if isinstance(c, str):
        return c
    if isinstance(c, list):
        return " ".join(str(x.get("text", "")) if isinstance(x, dict) else str(x) for x in c)
    return str(c)


def _get(m: Any, k: str) -> Any:
    return m.get(k) if isinstance(m, dict) else getattr(m, k, None)


def _norm(text: str) -> str:
    return " ".join(str(text).split())


_WORD = re.compile(r"[a-z0-9_]+")


def statement_retention(statement: str, summary: str) -> dict[str, Any]:
    """How much of `statement` a summary keeps: share of the statement's word 5-grams that occur in the summary,
    and share of its distinct words of >= 4 characters that occur in it. Numbers only."""
    sw, uw = _WORD.findall(statement.lower()), _WORD.findall(summary.lower())

    def grams(w: list[str], n: int) -> set[tuple[str, ...]]:
        return {tuple(w[i:i + n]) for i in range(len(w) - n + 1)}
    s5, u5 = grams(sw, 5), grams(uw, 5)
    sv, uv = {w for w in sw if len(w) >= 4}, set(uw)
    return {"ngram5_recall": round(len(s5 & u5) / len(s5), 4) if s5 else None,
            "word_recall": round(len(sv & uv) / len(sv), 4) if sv else None,
            "statement_words": len(sw), "summary_words": len(uw)}


def marker_sources(tid: str | None, messages: list[Any]) -> tuple[list[str], bool, bool]:
    """(where the task-prompt marker occurs: 'prompt' / 'summary' / 'other'; original statement present in a user
    message that is not a summary; a recorded summary present in any message)."""
    stmt = _norm(PROBLEM.get(tid or "", ""))[:_FP_CHARS]
    fps = [s["fp"] for s in SUMMARIES.get(tid or "", []) if s.get("fp")]
    src: set[str] = set()
    prompt_present = summary_present = False
    for m in messages:
        role, text = str(_get(m, "role")), _msg_text(m)
        nt = _norm(text)
        # ADK (contents.py, _present_other_agent_message) shows a compaction summary to the model as a USER message
        # "For context: [model] said: <summary>", so a summary is recognised by its text, whatever the role.
        is_summary = any(fp in nt for fp in fps)
        is_prompt = role == "user" and bool(stmt) and stmt in nt and not is_summary
        prompt_present |= is_prompt
        summary_present |= is_summary
        if TASK_PROMPT_MARKER in text:
            src.add("prompt" if is_prompt else "summary" if is_summary else "other")
    return sorted(src), prompt_present, summary_present


def request_shape(messages: list[Any], tools: Any, kwargs: dict[str, Any] | None = None) -> dict[str, Any]:
    tid = CURRENT_TASK.get()
    texts = [_msg_text(m) for m in messages]
    roles = Counter(str(_get(m, "role")) for m in messages)
    marks = MARKERS.get(tid or "", [])
    src, prompt_present, summary_present = marker_sources(tid, messages)
    kw = kwargs or {}
    return {
        "task": tid, "t": time.time(), "n_messages": len(messages), "roles": dict(roles),
        "n_tool_messages": roles.get("tool", 0),
        "n_assistant_tool_calls": sum(len(_get(m, "tool_calls") or []) for m in messages),
        "has_tools": bool(tools), "chars": sum(len(x) for x in texts),
        "has_task_prompt": any(TASK_PROMPT_MARKER in x for x in texts),
        "marker_sources": src, "original_prompt_present": prompt_present, "summary_present": summary_present,
        "requested_output_tokens": kw.get("max_completion_tokens") or kw.get("max_tokens"),
        "markers_present": [i for i, s in enumerate(marks) if any(s in x for x in texts)],
    }


def dump_summaries(path: Any) -> int:
    """Write every recorded summary (TEXT, competition-derived) as JSONL. Call it with a path under private/."""
    from pathlib import Path
    n = 0
    p = Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    with open(p, "w", encoding="utf-8") as f:
        for tid, items in SUMMARIES.items():
            for i, s in enumerate(items):
                f.write(json.dumps({"task": tid, "index": i, **{k: v for k, v in s.items() if k != "fp"}}) + "\n")
                n += 1
    return n


def compaction_record(events: list[Any], result: Any) -> dict[str, Any]:
    tid = CURRENT_TASK.get()
    marks = MARKERS.get(tid or "", [])
    n_fc = n_fr = n_skill = 0
    text_chars = nontext_chars = 0
    task_prompt = False
    raw_blobs: list[str] = []
    for e in events:
        parts = getattr(getattr(e, "content", None), "parts", None) or []
        for p in parts:
            if getattr(p, "text", None):
                text_chars += len(p.text)
                raw_blobs.append(p.text)
                if getattr(e, "author", "") == "user" and TASK_PROMPT_MARKER in p.text:
                    task_prompt = True
            fc = getattr(p, "function_call", None)
            if fc is not None:
                n_fc += 1
                blob = json.dumps(dict(fc.args or {}), default=str)
                nontext_chars += len(blob)
                raw_blobs.append(blob)
            fr = getattr(p, "function_response", None)
            if fr is not None:
                n_fr += 1
                if getattr(fr, "name", "") in SKILL_RESPONSE_TOOLS:
                    n_skill += 1
                blob = json.dumps(getattr(fr, "response", None), default=str)
                nontext_chars += len(blob)
                raw_blobs.append(blob)
    summary = ""
    comp = getattr(getattr(result, "actions", None), "compaction", None)
    cc = getattr(comp, "compacted_content", None)
    for p in getattr(cc, "parts", None) or []:
        if getattr(p, "text", None) and not getattr(p, "thought", False):   # what later requests see
            summary += p.text
    raw = "\n".join(raw_blobs)
    before = [i for i, s in enumerate(marks) if s in raw]
    ret = statement_retention(PROBLEM.get(tid or "", ""), summary) if summary else {}
    if result is not None and summary:
        SUMMARIES.setdefault(tid or "", []).append({"t": time.time(), "text": summary, "fp": _norm(summary)[:_FP_CHARS],
                                                    "has_marker": TASK_PROMPT_MARKER in summary, **ret})
    return {
        "summary_ngram5_recall": ret.get("ngram5_recall"), "summary_word_recall": ret.get("word_recall"),
        "task": tid, "t": time.time(), "n_events": len(events), "function_calls_dropped": n_fc,
        "function_responses_dropped": n_fr, "skill_responses_dropped": n_skill,
        "text_chars": text_chars, "nontext_chars": nontext_chars, "task_prompt_compacted": task_prompt,
        "summary_chars": len(summary), "summary_has_task_prompt_marker": TASK_PROMPT_MARKER in summary,
        "markers_before": before, "markers_in_summary": [i for i in before if marks[i] in summary],
        "produced": result is not None,
    }


def install() -> None:
    """Patch (idempotently) the ADK LLM client, the ADK summarizer and swegemma's evaluate_task."""
    global _installed
    if _installed:
        return
    from google.adk.apps import llm_event_summarizer as les
    from google.adk.models import lite_llm
    from swegemma import evaluate as sg_eval

    import litellm
    from common.server_watchdog import tracked

    # per-request timeout for agent requests (LiteLLM's default is 6000 s). 900 s = the per-task time budget:
    # a single request longer than the whole task budget is useless anyway.
    timeout_s = float(os.environ.get("PILOT_AGENT_REQUEST_TIMEOUT", AGENT_REQUEST_TIMEOUT_S))
    litellm.request_timeout = timeout_s
    orig_ac = lite_llm.LiteLLMClient.acompletion

    async def acompletion(self, model, messages, tools, **kwargs):
        rec = None
        try:
            rec = request_shape(messages, tools, kwargs)
            REQUESTS.append(rec)
        except Exception:  # noqa: BLE001 - instrumentation must never break a run
            pass
        kwargs.setdefault("timeout", timeout_s)
        with tracked():  # counts in-flight requests for the watchdog; fails fast after a freeze
            try:
                resp = await orig_ac(self, model, messages, tools, **kwargs)
            except Exception as e:
                if rec is not None:
                    rec["error"] = type(e).__name__
                raise
        if rec is not None:
            try:
                u = resp.get("usage") if isinstance(resp, dict) else getattr(resp, "usage", None)
                rec["prompt_tokens"] = u.get("prompt_tokens") if isinstance(u, dict) else getattr(u, "prompt_tokens", None)
            except Exception:  # noqa: BLE001
                pass
        return resp

    orig_sum = les.LlmEventSummarizer.maybe_summarize_events

    async def maybe_summarize_events(self, *, events):
        res = await orig_sum(self, events=events)
        try:
            COMPACTIONS.append(compaction_record(events, res))
        except Exception:  # noqa: BLE001
            pass
        return res

    orig_eval = sg_eval.Evaluator.evaluate_task

    async def evaluate_task(self, *args, **kwargs):
        from common.server_watchdog import ServerFrozen, STATE, TRIPPED
        if TRIPPED.is_set():  # server frozen: do not even set up the task's sandbox
            raise ServerFrozen(f"inference server frozen ({STATE['reason']}); task skipped")
        task = kwargs.get("task", args[0] if args else None)
        tid = getattr(task, "instance_id", None)
        if tid is not None:
            PROBLEM[tid] = str(getattr(task, "problem_statement", "") or "")
        tok = CURRENT_TASK.set(tid)
        try:
            return await orig_eval(self, *args, **kwargs)
        finally:
            CURRENT_TASK.reset(tok)

    lite_llm.LiteLLMClient.acompletion = acompletion
    les.LlmEventSummarizer.maybe_summarize_events = maybe_summarize_events
    sg_eval.Evaluator.evaluate_task = evaluate_task
    _installed = True


def reset() -> None:
    REQUESTS.clear()
    COMPACTIONS.clear()
    SUMMARIES.clear()


def task_compaction_stats(task_id: str, tool_call_timestamps: list[float]) -> dict[str, Any]:
    """Per-task summary: how often and when compaction fired, what it dropped, what survived."""
    comps = [c for c in COMPACTIONS if c["task"] == task_id and c["produced"]]
    reqs = [r for r in REQUESTS if r["task"] == task_id and r["has_tools"]]
    first_t = comps[0]["t"] if comps else None
    after = [r for r in reqs if first_t is not None and r["t"] > first_t]
    marks_before = sum(len(c["markers_before"]) for c in comps)
    return {
        "compactions": len(comps),
        "first_compaction_at_call": sum(1 for ts in tool_call_timestamps if first_t is not None and ts < first_t) if comps else None,
        "function_calls_dropped": sum(c["function_calls_dropped"] for c in comps),
        "function_responses_dropped": sum(c["function_responses_dropped"] for c in comps),
        "skill_responses_dropped": sum(c["skill_responses_dropped"] for c in comps),
        "nontext_share_dropped": round(sum(c["nontext_chars"] for c in comps) / max(1, sum(c["nontext_chars"] + c["text_chars"] for c in comps)), 4) if comps else None,
        "task_prompt_compacted": any(c["task_prompt_compacted"] for c in comps),
        "marker_survival": round(sum(len(c["markers_in_summary"]) for c in comps) / marks_before, 4) if marks_before else None,
        "requests": len(reqs),
        "requests_after_first_compaction": len(after),
        "after_with_task_prompt": sum(r["has_task_prompt"] for r in after),
        # where the marker sits after compaction (requests may have several sources)
        "after_marker_in_prompt": sum("prompt" in r.get("marker_sources", []) for r in after),
        "after_marker_in_summary_only": sum("summary" in r.get("marker_sources", []) and "prompt" not in r.get("marker_sources", [])
                                            for r in after),
        "after_marker_other_only": sum(bool(r.get("marker_sources")) and not {"prompt", "summary"} & set(r["marker_sources"])
                                       for r in after),
        "after_with_original_prompt": sum(bool(r.get("original_prompt_present")) for r in after),
        "summaries": len(SUMMARIES.get(task_id, [])),
        "summaries_with_marker": sum(bool(x.get("has_marker")) for x in SUMMARIES.get(task_id, [])),
        "summary_ngram5_recall_mean": _mean([x.get("ngram5_recall") for x in SUMMARIES.get(task_id, [])]),
        "summary_ngram5_recall_max": max([x["ngram5_recall"] for x in SUMMARIES.get(task_id, []) if x.get("ngram5_recall") is not None], default=None),
        "summary_word_recall_mean": _mean([x.get("word_recall") for x in SUMMARIES.get(task_id, [])]),
        "requested_output_tokens": sorted({r["requested_output_tokens"] for r in reqs if r.get("requested_output_tokens")}),
        "max_prompt_tokens": max([r["prompt_tokens"] for r in reqs if r.get("prompt_tokens")], default=None),
        "context_errors": sum("ContextWindow" in str(r.get("error") or "") for r in reqs),
        "prompt_tokens_before_first_context_error": _before_error(reqs),
        "max_tool_messages_before": max((r["n_tool_messages"] for r in reqs if first_t is None or r["t"] <= first_t), default=0),
        "max_tool_messages_after": max((r["n_tool_messages"] for r in after), default=None),
    }


def _mean(xs: list[Any]) -> float | None:
    v = [x for x in xs if x is not None]
    return round(sum(v) / len(v), 4) if v else None


def _before_error(reqs: list[dict[str, Any]]) -> int | None:
    """Prompt tokens of the last successful request before the first context-window error (None if no error)."""
    last = None
    for r in sorted(reqs, key=lambda r: r["t"]):
        if "ContextWindow" in str(r.get("error") or ""):
            return last
        if r.get("prompt_tokens"):
            last = r["prompt_tokens"]
    return None

In [ ]:
%%writefile /kaggle/working/pilot/common/server_watchdog.py
"""Freeze protection for a local OpenAI-compatible inference server (vLLM) during GPU notebooks.

Why: a main-track forum report says the gemma4 tool parser can loop forever on an array tool argument
(in the harness tool set the only one is `get_code_subgraph(nodes: list[str])`) and freeze the server
for the rest of the run. Without protection a freeze burns L4x4 time until the session limit: LiteLLM's
default request timeout is 6000 s, and adk-eval-core's ModelRetryPlugin retries timeouts up to 5 times.

Three layers, used by every GPU notebook:
  1. per-request timeouts (set by the callers: runtime_probes for agent requests, probe_runner for probes);
  2. `Watchdog`: a thread polling GET /health; after `max_fail` consecutive failures, or when requests
     are in flight but none has finished for `stall_s`, it trips: it sets TRIPPED and kills the server;
  3. `tracked()`: wraps every request; once TRIPPED is set, new and retried requests fail immediately with
     ServerFrozen (an adk-eval-core EvalCoreError when available, which its retry plugin does not retry).
The notebook loop then restarts the server for the next unit of work, and stops the session after a
second freeze.
"""
from __future__ import annotations

import contextlib
import os
import threading
import time
import urllib.request
from typing import Any, Callable

try:  # non-retryable for adk-eval-core's ModelRetryPlugin
    from adk_eval_core.errors import EvalCoreError as _Base
except Exception:  # noqa: BLE001
    _Base = RuntimeError


class ServerFrozen(_Base):
    """The inference server stopped responding and was killed by the watchdog."""


TRIPPED = threading.Event()
STATE: dict[str, Any] = {"reason": None, "t": None}
_LOCK = threading.Lock()
TRACK: dict[str, Any] = {"inflight": 0, "started": 0, "done": 0, "last_done": time.time()}
_OPENER = urllib.request.build_opener(urllib.request.ProxyHandler({}))


def reset() -> None:
    TRIPPED.clear()
    STATE.update({"reason": None, "t": None})
    with _LOCK:
        TRACK.update({"inflight": 0, "started": 0, "done": 0, "last_done": time.time()})


@contextlib.contextmanager
def tracked():
    """Wrap one request to the inference server: counts it, and fails fast after a freeze."""
    if TRIPPED.is_set():
        raise ServerFrozen(f"inference server frozen ({STATE['reason']}); request not sent")
    with _LOCK:
        TRACK["inflight"] += 1
        TRACK["started"] += 1
    try:
        yield
    except Exception as e:  # noqa: BLE001
        if TRIPPED.is_set() and not isinstance(e, ServerFrozen):
            raise ServerFrozen(f"inference server frozen ({STATE['reason']}): {type(e).__name__}") from e
        raise
    finally:
        with _LOCK:
            TRACK["inflight"] -= 1
            TRACK["done"] += 1
            TRACK["last_done"] = time.time()


def _env(name: str, default: float) -> float:
    return float(os.environ.get(name, default))


class Watchdog(threading.Thread):
    def __init__(self, root_url: str, kill: Callable[[], None], interval: float | None = None,
                 health_timeout: float | None = None, max_fail: int | None = None, stall_s: float | None = None,
                 log: Callable[[str], None] = print):
        super().__init__(daemon=True)
        self.root = root_url.rstrip("/")
        self.root = self.root[:-3] if self.root.endswith("/v1") else self.root
        self.kill = kill
        # defaults: a freeze is detected within ~90 s; overridable for dry runs
        self.interval = interval if interval is not None else _env("PILOT_WD_INTERVAL", 30)
        self.health_timeout = health_timeout if health_timeout is not None else _env("PILOT_WD_HEALTH_TIMEOUT", 10)
        self.max_fail = int(max_fail if max_fail is not None else _env("PILOT_WD_MAX_FAIL", 3))
        self.stall_s = stall_s if stall_s is not None else _env("PILOT_WD_STALL_S", 1200)
        self.log = log
        self._halt = threading.Event()  # not `_stop`: that name is a threading.Thread internal
        self.checks = 0

    def healthy(self) -> bool:
        try:
            with _OPENER.open(f"{self.root}/health", timeout=self.health_timeout) as r:
                return 200 <= r.status < 300
        except Exception:  # noqa: BLE001
            return False

    def run(self) -> None:
        fails = 0
        while not self._halt.wait(self.interval):
            self.checks += 1
            fails = 0 if self.healthy() else fails + 1
            with _LOCK:
                stalled = TRACK["inflight"] > 0 and time.time() - TRACK["last_done"] > self.stall_s
            if fails >= self.max_fail or stalled:
                self.trip("health check failed %d times" % fails if fails >= self.max_fail
                          else "no request finished for %.0f s" % self.stall_s)
                return

    def trip(self, reason: str) -> None:
        STATE.update({"reason": reason, "t": time.time()})
        TRIPPED.set()
        self.log(f"WATCHDOG: inference server considered frozen ({reason}); killing it")
        killer = threading.Thread(target=self._kill, daemon=True)
        killer.start()
        killer.join(60)

    def _kill(self) -> None:
        try:
            self.kill()
        except Exception as e:  # noqa: BLE001
            self.log(f"WATCHDOG: kill raised {type(e).__name__}: {e}")

    def stop(self) -> None:
        self._halt.set()


def state() -> dict[str, Any]:
    with _LOCK:
        return {"tripped": TRIPPED.is_set(), **STATE, **{k: v for k, v in TRACK.items() if k != "last_done"}}

In [ ]:
%%writefile /kaggle/working/pilot/pilot_c/__init__.py
# (empty module)

In [ ]:
%%writefile /kaggle/working/pilot/pilot_c/hints.py
"""Oracle-hint rendering for Pilot C (locations and test names only, never patch content).

Conditions (nested for localization, orthogonal for tests):
  O0      no hint
  O-file  gold source file(s)
  O-func  gold file(s) + enclosing function/class names (with def line)
  O-test  scorer-required test names + their test file(s)
  O-all   O-func + O-test

Hints are appended to problem_statement in a LOCAL copy of the tasks used only
by the local evaluation runner. They must never be placed in a submission.
"""
from __future__ import annotations

import copy

CONDITIONS = ("O0", "O-file", "O-func", "O-test", "O-all")
_HEADER = "## Location hints (verified ground truth for this task)"


def _file_lines(g: dict) -> list[str]:
    out = []
    new = set(g.get("new_src_files") or [])
    for f in g.get("src_files") or []:
        out.append(f"- The fix changes `{f}`" + (" (a new file that does not exist yet)." if f in new else "."))
    if not out:
        out.append("- The fix does not change any existing non-test Python source file.")
    return out


def _func_lines(g: dict) -> list[str]:
    out = []
    by_file: dict[str, list[dict]] = {}
    for fn in g.get("functions") or []:
        by_file.setdefault(fn["file"], []).append(fn)
    for f in g.get("src_files") or []:
        for fn in by_file.get(f, []):
            what = "class body of" if fn["kind"] == "class" else "function"
            out.append(f"- In `{f}`: {what} `{fn['dotted']}` (defined at line {fn['def_line']}).")
        if f in (g.get("module_level_files") or []):
            out.append(f"- In `{f}`: module-level code (imports, constants, or new top-level definitions).")
    return out


def _test_lines(g: dict) -> list[str]:
    names = g.get("test_names") or []
    files = g.get("test_files") or []
    new = set(g.get("new_test_files") or [])
    out = []
    for f in files:
        status = "this test file does not exist yet; it will be added when your fix is verified" if f in new \
            else "this test file exists; the listed tests may be new or changed"
        out.append(f"- Verification runs `{f}` ({status}).")
    if names:
        out.append("- Tests that must pass: " + ", ".join(f"`{n}`" for n in names) + ".")
    return out


def render_hint(g: dict, condition: str) -> str:
    if condition not in CONDITIONS:
        raise ValueError(condition)
    if condition == "O0":
        return ""
    lines: list[str] = []
    if condition in ("O-file", "O-func", "O-all"):
        lines += _file_lines(g)
    if condition in ("O-func", "O-all"):
        lines += _func_lines(g)
    if condition in ("O-test", "O-all"):
        lines += _test_lines(g)
    return "\n".join([_HEADER, *lines])


def rewrite_task(task: dict, g: dict, condition: str) -> dict:
    """Return a copy of `task` with the hint appended to problem_statement."""
    t = copy.deepcopy(task)
    hint = render_hint(g, condition)
    if hint:
        t["problem_statement"] = t["problem_statement"].rstrip() + "\n\n" + hint + "\n"
    return t

In [ ]:
%%writefile /kaggle/working/pilot/pilot_c/select_tasks.py
"""Select the pilot tasks (instance IDs only are written; safe to commit).

Rule (fixed before any agent run):
  eligible = exactly 1 changed source file that already exists at base_commit,
             1-2 enclosing functions/classes (function-level oracle well defined),
             gold patch <= 40 changed lines, >= 1 scorer-required test name,
             repo in {fastapi, rich, requests},
             and the GOLD patch resolves the task in the sandbox (gold check JSON, if given)
  pilot15  = stratified random sample: fastapi 7, rich 6, requests 2 (seeded)
  reserve  = next 4 per repo from the same shuffled order (replace tasks whose gold
             patch fails the CPU gold check)
  smoke3   = first pilot task of each repo (used for the smoke test and size timing)

usage: python -I pilot_c/select_tasks.py <gold_all.json> <out task_ids.json> [gold_check.json ...]
"""
import json
import random
import sys

SEED = 20261031
QUOTA = {"fastapi/fastapi": 7, "Textualize/rich": 6, "psf/requests": 2}
RESERVE = 4


def eligible(g):
    return (len(g["src_files"]) == 1 and not g["new_src_files"] and 1 <= len(g["functions"]) <= 2
            and g["gold_added"] + g["gold_removed"] <= 40 and len(g["test_names"]) >= 1 and g["repo"] in QUOTA)


def main():
    G = json.load(open(sys.argv[1]))
    checks = {}
    for path in sys.argv[3:]:
        for r in json.load(open(path))["rows"]:
            checks[r["instance_id"]] = bool(r["gold_resolved"])
    if checks:
        G = {i: g for i, g in G.items() if checks.get(i) is True}
    rng = random.Random(SEED)
    pilot, reserve, smoke, pool_sizes = [], [], [], {}
    for repo, k in QUOTA.items():
        pool = sorted(i for i, g in G.items() if g["repo"] == repo and eligible(g))
        pool_sizes[repo] = len(pool)
        rng.shuffle(pool)
        pilot += pool[:k]
        reserve += pool[k:k + RESERVE]
        smoke.append(pool[0])
    out = {"seed": SEED, "gold_check_sources": sys.argv[3:], "n_gold_checked": len(checks), "rule": __doc__.split("Rule (fixed before any agent run):")[1].split("usage:")[0].strip(),
           "eligible_pool_sizes": pool_sizes, "pilot15": pilot, "reserve": reserve, "smoke3": smoke}
    json.dump(out, open(sys.argv[2], "w"), indent=1)
    print(json.dumps({k: out[k] for k in ("eligible_pool_sizes", "smoke3")}), len(pilot), len(reserve))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/pilot/pilot_c/kaggle_runner.py
"""Kaggle-side runner for Pilot C (imported by the notebooks; runs only on Kaggle).

Reproduces the official starter's evaluation path (swegemma Evaluator, subprocess
sandbox, vLLM with max_model_len=32768, tool/reasoning parser gemma4, the same
compaction and context-cache settings as the scorer per HARNESS_README §7.2) and
changes only: model alias, per-task budget (eval_config.yaml), task copies with
oracle hints, results dir, concurrency.

Outputs:
  /kaggle/working/private/...         raw runs (patches, traces) - NEVER commit
  /kaggle/working/C0x_<what>.json      numbers only - commit to results/ (names: notebooks/README.md)
"""
from __future__ import annotations

import asyncio
import concurrent.futures
import datetime as _dt
import glob
import json
import os
import platform
import re
import shutil
import subprocess
import sys
import time
from pathlib import Path
from typing import Any

REPO_ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(REPO_ROOT))

from common.metrics import aggregate, paired_flips, task_metrics  # noqa: E402
from common.tasks import diff_scopes, gold_locations, load_tasks  # noqa: E402
from common import runtime_probes as rp  # noqa: E402
from common.trace import load_trace  # noqa: E402
from pilot_c.hints import _HEADER as HINT_HEADER, rewrite_task  # noqa: E402

WORK = Path(os.environ.get("PILOT_WORK_DIR", "/kaggle/working"))
PRIVATE = WORK / "private"
ALLOWED = ("gemma-4-31b-it-qat-w4a16-ct", "gemma-4-12b-it-qat-w4a16-ct",
           "gemma-4-e4b-it-qat-w4a16-ct", "gemma-4-e2b-it-qat-w4a16-ct")
SHORT = {m: m.split("-")[2] for m in ALLOWED}  # 31b / 12b / e4b / e2b
MODEL_VLLM_EXTRA_ARGS = {
    "gemma-4-12b-it-qat-w4a16-ct": ("--hf-overrides", '{"architectures": ["Gemma4ForCausalLM"], "model_type": "gemma4"}'),
}
GPU_ONLY_WHEELS = ("vllm", "flashinfer", "bitsandbytes", "xgrammar", "nvidia_", "quack_kernels", "apache_tvm_ffi")


def _first_dir(*cands: str) -> Path:
    for c in cands:
        for p in sorted(glob.glob(c)):
            if Path(p).is_dir():
                return Path(p)
    raise FileNotFoundError(f"none of {cands} found")


def data_dir() -> Path:
    if os.environ.get("PILOT_DATA_DIR"):  # local testing only
        return Path(os.environ["PILOT_DATA_DIR"])
    return _first_dir("/kaggle/input/competitions/gemma-4-developer-agent", "/kaggle/input/gemma-4-developer-agent")


def wheelhouse_dir() -> Path:
    return _first_dir("/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse",
                      "/kaggle/input/gemma-4-developer-agent-wheelhouse")


def set_env() -> None:
    """Same environment variables as the official starter notebook."""
    env = {
        "TRANSFORMERS_NO_TF": "1", "TRANSFORMERS_NO_FLAX": "1", "TRANSFORMERS_OFFLINE": "1",
        "VLLM_NO_USAGE_STATS": "1", "VLLM_WORKER_MULTIPROC_METHOD": "spawn", "VLLM_ENGINE_READY_TIMEOUT_S": "1200",
        "VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS": "1", "NCCL_DEBUG": "WARN", "LITELLM_LOCAL_MODEL_COST_MAP": "True",
        "LITELLM_TELEMETRY": "False", "LITELLM_LOG": "WARNING", "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
        "OTEL_SDK_DISABLED": "true", "OTEL_PYTHON_DISABLED": "true",
    }
    os.environ.update(env)
    os.environ.setdefault("KAGGLE_SANDBOX_DIR", str(data_dir() / "sandbox"))
    os.environ.setdefault("OPENAI_API_KEY", "EMPTY")
    os.environ.setdefault("LOCAL_API_KEY", "EMPTY")


EXPECTED_PYTHON = (3, 12)  # the official starter's pinned Kaggle environment; the wheelhouse is built for cp312
# Wheelhouse wheels the CPU notebooks need (gold check, test verification). Found by running that code path and
# recording every imported distribution: of 90, only these 5 come from the wheelhouse; all are pure Python
# (py3-none-any). Everything else is provided by the Kaggle image. The other 36 wheels serve the model (vLLM stack).
CPU_WHEELS = ("swegemma", "adk_eval_core", "adk_submission", "google_adk", "google_genai")


def _dist_of(wheel_name: str) -> str:
    return re.sub(r"[-_.]+", "_", wheel_name.split("-")[0]).lower()


def _wheel_supported(wheel_name: str) -> bool:
    """True if the wheel's python tag fits this interpreter (pure-python, abi3 or the exact cpXY)."""
    parts = wheel_name[:-4].split("-")
    py, abi = parts[-3], parts[-2]
    here = f"cp{sys.version_info.major}{sys.version_info.minor}"
    if py.startswith("py") or "none" in abi and not py.startswith("cp"):
        return True
    if abi == "abi3":
        return any(int(t[3:] or 0) <= sys.version_info.minor for t in py.split(".") if t.startswith("cp3"))
    return here in py.split(".")


def install_wheelhouse(gpu: bool = True, only: tuple[str, ...] | None = None) -> None:
    """Starter-identical wheel install. gpu=False skips GPU-only wheels; `only` restricts to those distributions
    (the CPU notebooks pass CPU_WHEELS). Refuses, before calling pip, any wheel built for another Python."""
    import importlib
    for pat in ("/usr/local/lib/python*/dist-packages/*cutlass*.pth", "/usr/local/lib/python*/site-packages/*cutlass*.pth"):
        for pth in glob.glob(pat):
            try:
                os.unlink(pth)
            except OSError:
                pass
    tmp = Path("/tmp/wheelhouse")
    if tmp.exists():
        shutil.rmtree(tmp)
    tmp.mkdir(parents=True, exist_ok=True)
    for w in wheelhouse_dir().glob("*.whl"):
        if "cutlass" in w.name.lower() or (not gpu and w.name.startswith(GPU_ONLY_WHEELS)):
            continue
        if only is not None and _dist_of(w.name) not in only:
            continue
        name = w.name.replace("cu128", "+cu128") if ("cu128" in w.name and "+" not in w.name) else w.name
        t = tmp / name
        if not t.exists():
            try:
                os.symlink(w, t)
            except OSError:
                shutil.copyfile(w, t)
    wheels = sorted(str(w) for w in tmp.glob("*.whl"))
    if only is not None:
        missing = sorted(set(only) - {_dist_of(Path(w).name) for w in wheels})
        if missing:
            raise RuntimeError(f"wheelhouse is missing {missing}; is the dataset metric/gemma-4-developer-agent-wheelhouse attached?")
    bad = [Path(w).name for w in wheels if not _wheel_supported(Path(w).name)]
    if bad:
        raise RuntimeError(f"Python {sys.version.split()[0]} cannot install {bad[:3]}... (wheelhouse is built for "
                           f"cp{EXPECTED_PYTHON[0]}{EXPECTED_PYTHON[1]}). Run this notebook in the official starter's "
                           "pinned environment: see the first cell of the notebook.")
    print(f"installing {len(wheels)} wheels: {', '.join(_dist_of(Path(w).name) for w in wheels)}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "--force-reinstall", *wheels], check=True)
    importlib.invalidate_caches()


def quiet_logs() -> None:
    import logging
    for name in ("LiteLLM", "LiteLLM Router", "LiteLLM Proxy", "litellm", "httpx", "httpcore", "openai", "urllib3",
                 "opentelemetry", "opentelemetry.context"):
        lg = logging.getLogger(name)
        lg.setLevel(logging.WARNING)
        if "litellm" in name.lower():
            lg.propagate = False
            lg.handlers.clear()


def run_sync(fn, *args, **kwargs):
    """Run an async callable from a notebook (same helper as the starter)."""
    coro_fn = lambda: fn(*args, **kwargs)  # noqa: E731
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None
    if loop is not None and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: asyncio.run(coro_fn())).result()
    return asyncio.run(coro_fn())


def env_info() -> dict[str, Any]:
    info: dict[str, Any] = {"python": platform.python_version(), "nproc": os.cpu_count()}
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                             capture_output=True, text=True, timeout=30).stdout.strip().splitlines()
        info["gpus"] = out
    except Exception:  # noqa: BLE001
        info["gpus"] = []
    try:
        import importlib.metadata as md
        info["versions"] = {p: md.version(p) for p in ("swegemma", "adk-eval-core", "adk-submission", "google-adk", "vllm")
                            if _has_dist(md, p)}
    except Exception:  # noqa: BLE001
        pass
    return info


def _has_dist(md, p: str) -> bool:
    try:
        md.version(p)
        return True
    except Exception:  # noqa: BLE001
        return False


# --------------------------------------------------------------------------- tasks & hints

def build_gold(task_ids: list[str]) -> dict[str, dict]:
    d = data_dir()
    tasks = {t["instance_id"]: t for t in load_tasks(d / "tasks.jsonl")}
    out = {}
    for iid in task_ids:
        out[iid] = gold_locations(tasks[iid], d / "snapshots" / f"{iid}.tgz").to_dict()
    return out


def write_task_copy(task_ids: list[str], gold: dict[str, dict], condition: str) -> Path:
    """Local hinted copy of tasks.jsonl for one condition (kept under PRIVATE)."""
    d = data_dir()
    tasks = {t["instance_id"]: t for t in load_tasks(d / "tasks.jsonl")}
    PRIVATE.mkdir(parents=True, exist_ok=True)
    path = PRIVATE / f"tasks_{condition}.jsonl"
    with open(path, "w", encoding="utf-8") as f:
        for iid in task_ids:
            f.write(json.dumps(rewrite_task(tasks[iid], gold[iid], condition)) + "\n")
    return path


# --------------------------------------------------------------------------- agent & server

def prepare_agent_dir(model: str, budget: dict[str, Any], sampling_overrides: dict[str, Any] | None = None) -> Path:
    """Official sample_submission with the model alias swapped and the sample's dummy LoRA
    adapters removed for EVERY size (starter logic for non-31B; for 31B too, so all sizes run the
    plain base model), and eval_config.yaml replaced by the pilot budget. `sampling_overrides` changes keys of
    configs/sampling.yaml (shared by the agent and its code_analyzer sub-agent), e.g. {'max_output_tokens': 8192}."""
    assert model in ALLOWED, model
    src = data_dir() / "sample_submission"
    dst = WORK / f"agent_{SHORT[model]}"
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    shutil.rmtree(dst / "adapters", ignore_errors=True)
    for y in [dst / "agent.yaml", *sorted((dst / "sub_agents").glob("*.yaml"))]:
        if y.is_file():
            txt = y.read_text(encoding="utf-8")
            txt = re.sub(r"^(\s*model:\s*).*$", rf"\g<1>{model}", txt, flags=re.M)
            txt = re.sub(r"^\s*adapter:\s*.*\n?", "", txt, flags=re.M)
            y.write_text(txt, encoding="utf-8")
    lines = ["evaluation:"] + [f"  {k}: {v}" for k, v in budget.items()]
    (dst / "eval_config.yaml").write_text("\n".join(lines) + "\n", encoding="utf-8")
    if sampling_overrides:
        import yaml
        sp = dst / "configs" / "sampling.yaml"
        samp = yaml.safe_load(sp.read_text(encoding="utf-8")) or {}
        samp.update(sampling_overrides)
        sp.write_text(yaml.safe_dump(samp, sort_keys=False), encoding="utf-8")
    return dst


def describe_agent(agent_dir: Path) -> dict[str, Any]:
    import yaml
    a = yaml.safe_load((agent_dir / "agent.yaml").read_text().replace("!include", "#include")) or {}
    samp = yaml.safe_load((agent_dir / "configs" / "sampling.yaml").read_text())
    tools = [t if isinstance(t, str) else "agent_tool:" + str(t.get("agent_tool", {}).get("config_path")) for t in a.get("tools", [])]
    return {"model": a.get("model"), "tools": tools, "sampling": samp,
            "eval_config": yaml.safe_load((agent_dir / "eval_config.yaml").read_text())}


class GpuMemoryNotReleased(RuntimeError):
    """GPU memory did not come back after a server was stopped; starting another server would fail."""


SERVER_LOG_DIR = PRIVATE / "server_logs"
LAST_SERVER_ERRORS: list[str] = []


def _own_process_group() -> None:
    """Make adk_submission start every inference server in its own session / process group.

    VllmServer.start() uses a plain Popen, so the server shares the notebook kernel's process group and
    VllmServer.stop() kills only the API-server PID: vLLM's EngineCore and its tensor-parallel worker
    processes survive and keep ~20 GiB on every GPU (Kaggle C03 run, 2026-10-31)."""
    import types
    import adk_submission.server as ads
    if getattr(ads.subprocess, "_pilot_own_group", False):
        return
    real = subprocess

    def popen(*args, **kwargs):
        kwargs.setdefault("start_new_session", True)
        return real.Popen(*args, **kwargs)

    shim = types.SimpleNamespace(**{n: getattr(real, n) for n in dir(real) if not n.startswith("__")})
    shim.Popen = popen
    shim._pilot_own_group = True
    ads.subprocess = shim


def gpu_memory() -> list[tuple[int, int]] | None:
    """(free MiB, total MiB) per GPU from nvidia-smi, or None when there is no GPU (dry runs)."""
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=memory.free,memory.total", "--format=csv,noheader,nounits"],
                             capture_output=True, text=True, timeout=30)
    except (FileNotFoundError, subprocess.TimeoutExpired):
        return None
    if out.returncode != 0 or not out.stdout.strip():
        return None
    return [tuple(int(float(x)) for x in line.split(",")[:2]) for line in out.stdout.strip().splitlines()]


def wait_gpu_free(need_frac: float, timeout_s: float = 300, poll_s: float = 5) -> dict[str, Any]:
    """Block until every GPU has at least need_frac of its memory free; raise GpuMemoryNotReleased on timeout."""
    t0 = time.time()
    while True:
        mem = gpu_memory()
        if mem is None:
            return {"checked": False}
        if all(free >= need_frac * total for free, total in mem):
            return {"checked": True, "waited_s": round(time.time() - t0, 1), "free_gib": [round(f / 1024, 2) for f, _ in mem]}
        if time.time() - t0 > timeout_s:
            raise GpuMemoryNotReleased(
                f"GPU memory not released {timeout_s:.0f}s after stopping the previous server: free "
                f"{[round(f / 1024, 2) for f, _ in mem]} GiB of {[round(t / 1024, 2) for _, t in mem]} GiB; the next "
                f"server needs {need_frac:.0%} free on every GPU. Stopping instead of starting another server.")
        time.sleep(poll_s)


def _save_server_log(server, tag: str) -> list[str]:
    """Keep the server log (VllmServer.stop() deletes it) under private/, return its error lines."""
    path = getattr(server, "log_path", None)
    if not path or not os.path.exists(path):
        return []
    try:
        lines = Path(path).read_text(encoding="utf-8", errors="replace").splitlines()
    except OSError:
        return []
    SERVER_LOG_DIR.mkdir(parents=True, exist_ok=True)
    (SERVER_LOG_DIR / f"{tag}_{int(time.time())}.log").write_text("\n".join(lines[-2000:]), encoding="utf-8")
    errs = [l[:300] for l in lines if re.search(r"ERROR|Traceback|Error:|out of memory|OutOfMemory", l)]
    return errs[-20:]


def error_kinds(lines: list[str]) -> list[str]:
    """Exception class names only (safe to commit; the lines themselves stay private)."""
    return sorted({m for l in lines for m in re.findall(r"\b([A-Z][A-Za-z]+(?:Error|Exception))\b", l)})


def stop_server(server, tag: str = "server", timeout: float = 15.0) -> dict[str, Any]:
    """Stop an inference server and everything it spawned: SIGTERM, then SIGKILL, to its whole process group,
    plus any descendants recorded before the kill. Saves the server log first."""
    import signal
    global LAST_SERVER_ERRORS
    LAST_SERVER_ERRORS = _save_server_log(server, tag)
    info: dict[str, Any] = {"group_killed": False, "survivors": 0, "error_kinds": error_kinds(LAST_SERVER_ERRORS)}
    proc = getattr(server, "process", None)
    if proc is None:
        safe_stop(server)
        return info
    pid = proc.pid
    try:
        import psutil
        desc = psutil.Process(pid).children(recursive=True)
    except Exception:  # noqa: BLE001
        psutil, desc = None, []
    try:
        pgid = os.getpgid(pid)
    except ProcessLookupError:
        pgid = None
    own = pgid is not None and pgid != os.getpgrp()
    try:
        if own:
            os.killpg(pgid, signal.SIGTERM)
        else:
            proc.terminate()
        try:
            proc.wait(timeout=timeout)
        except subprocess.TimeoutExpired:
            pass
        if own:
            try:
                os.killpg(pgid, signal.SIGKILL)
                info["group_killed"] = True
            except ProcessLookupError:
                pass
        elif proc.poll() is None:
            proc.kill()
        for d in desc:
            try:
                d.kill()
            except Exception:  # noqa: BLE001
                pass
        if psutil is not None and desc:
            _, alive = psutil.wait_procs(desc, timeout=15)
            info["survivors"] = len(alive)
        try:
            proc.wait(timeout=10)
        except Exception:  # noqa: BLE001
            pass
    finally:
        safe_stop(server)
    info["descendants"] = len(desc)
    return info


def start_server(agent_dir: Path, gpu_memory_utilization: float = 0.90, extra_args: tuple[str, ...] = ()):
    """Starter-identical vLLM launch (except optional memory settings for the probe), in its own process group.
    Waits until every GPU is free first. Returns (server, models, adapters, startup_seconds)."""
    if os.environ.get("PILOT_MOCK_LLM"):  # local papermill dry runs only (no GPU, scripted mock LLM)
        return _start_mock_server(agent_dir)
    import litellm
    import torch
    from adk_submission import VllmConfig, VllmServer, discover_adapters
    from adk_submission.discovery import discover_declared_models
    from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS

    _own_process_group()
    litellm.drop_params = True
    declared = discover_declared_models(agent_dir)
    assert len(declared) == 1, declared
    model = next(iter(declared))
    assert model in ALLOWED, model
    model_path = Path(f"/kaggle/input/models/google/gemma-4/other/{model}/2")
    if not model_path.exists():
        model_path = _first_dir(f"/kaggle/input/models/google/gemma-4/*/{model}/*", f"/kaggle/input/gemma-4/*/{model}/*")
    adapters = discover_adapters(str(agent_dir), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
    amap = getattr(adapters, "adapters", None) or {}
    ranks = [r for info in amap.values() if (r := getattr(info, "rank", None)) is not None and r > 0]
    n = torch.cuda.device_count() if torch.cuda.is_available() else 1
    tp = 4 if n >= 4 else (2 if n >= 2 else 1)
    waited = wait_gpu_free(gpu_memory_utilization + 0.02)
    cfg = VllmConfig(
        model=str(model_path), port=8000, host="127.0.0.1", tool_call_parser="gemma4", reasoning_parser="gemma4",
        max_model_len=32768, dtype="bfloat16" if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else "auto",
        gpu_memory_utilization=gpu_memory_utilization, enable_auto_tool_choice=True, enable_lora=bool(amap),
        max_loras=max(1, len(amap)), max_lora_rank=max(ranks) if ranks else 128, scheduler_reserve_full_isl=False,
        tensor_parallel_size=tp, startup_timeout=60 * 20,
        extra_args=tuple(MODEL_VLLM_EXTRA_ARGS.get(model, ())) + tuple(extra_args),
    )
    t0 = time.perf_counter()
    server = VllmServer(cfg, adapter_manifest=adapters)
    try:
        server.start()
    except Exception:
        stop_server(server, tag=f"{SHORT[model]}_failed_start")  # also kills workers of a half-started server
        print("server log errors:", *LAST_SERVER_ERRORS[-8:], sep="\n  ")
        raise
    startup = time.perf_counter() - t0
    models = server.create_model_registry(aliases=[model], model_prefix="openai/", api_key=os.environ.get("LOCAL_API_KEY", "EMPTY"))
    print(f"vLLM up for {model} in {startup:.0f}s (tp={tp}, gpu_memory_utilization={gpu_memory_utilization}, "
          f"extra={list(extra_args)}, waited for GPU memory: {waited})")
    return server, models, adapters, startup


def _start_mock_server(agent_dir: Path):
    """Dry-run stand-in for vLLM: the scripted OpenAI-compatible mock from tests/ (PILOT_REPO)."""
    import litellm
    from adk_submission import discover_adapters
    from adk_submission.registry import ModelRegistry
    from adk_submission.resolvers.generation import install_litellm_reasoning_patch
    from google.adk.models.lite_llm import LiteLlm
    from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS
    sys.path.insert(0, os.environ["PILOT_REPO"])
    from tests.mock_openai_server import kill, make_script, serve

    litellm.drop_params = True
    install_litellm_reasoning_patch()
    model = describe_agent(agent_dir)["model"]
    if os.environ.get("PILOT_MOCK_FAIL_START") == SHORT.get(model):  # dry-run test of a size whose server fails
        raise RuntimeError("mock: inference server exited prematurely")
    srv = serve(make_script("README.md"), port=int(os.environ.get("PILOT_MOCK_PORT", "8001")))
    reg = ModelRegistry()
    reg.register(model, LiteLlm(model="openai/mock", api_base=f"http://127.0.0.1:{srv.server_address[1]}/v1", api_key="EMPTY"))
    ad = discover_adapters(str(agent_dir), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)

    class _S:
        base_url = f"http://127.0.0.1:{srv.server_address[1]}/v1"

        def stop(self):
            kill(srv)
    return _S(), reg, ad, 0.0


def server_root(server) -> str:
    """http://host:port of a started server (VllmServer.base_url or the dry-run mock's), without /v1."""
    base = str(getattr(server, "base_url", None) or "http://127.0.0.1:8000").rstrip("/")
    return base[:-3] if base.endswith("/v1") else base


def safe_stop(server) -> None:
    """Stop a server that may already have been killed by the watchdog."""
    try:
        server.stop()
    except Exception as e:  # noqa: BLE001
        print(f"server.stop(): {type(e).__name__}: {e}")


# --------------------------------------------------------------------------- evaluation

def make_eval_config(agent_dir: Path, models, adapters, tasks_path: Path, task_ids: list[str], results_dir: Path,
                     concurrency: int, sandbox: str = "subprocess"):
    import yaml
    from google.adk.agents.context_cache_config import ContextCacheConfig
    from google.adk.apps._configs import EventsCompactionConfig
    from swegemma.config import EvalConfig, build_submission_limits

    raw = yaml.safe_load((agent_dir / "eval_config.yaml").read_text()) or {}
    ev = raw.get("evaluation", raw)
    turns = ev.get("max_turns", ev.get("max_llm_calls"))
    limits, gen = build_submission_limits()
    d = data_dir()
    return EvalConfig(
        tasks_path=tasks_path, snapshots_dir=d / "snapshots", results_dir=results_dir, submission_dir=agent_dir,
        models=models, sandbox=sandbox,
        timeout_seconds=int(ev.get("timeout_seconds", 300)),
        verification_timeout_seconds=int(ev.get("verification_timeout_seconds", 300)),
        max_time_minutes=float(ev.get("max_time_minutes", 60.0)), max_tool_calls=int(ev.get("max_tool_calls", 100)),
        max_turns=int(turns) if turns is not None else None,
        limits=limits, generation_constraints=gen, adapter_manifest=adapters,
        context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
        events_compaction_config=EventsCompactionConfig(compaction_interval=15, overlap_size=2, token_threshold=14336,
                                                        event_retention_size=5),
        graph_dir=str(d / "graphs"), embeddings_dir=str(d / "embeddings"), wheels_dir=d / "wheels",
        verbose=False, task_ids=list(task_ids), concurrency=concurrency, display_mode="quiet",
    )


def _result_dict(r) -> dict[str, Any]:
    try:
        d = r.model_dump(exclude={"trace"})
    except Exception:  # noqa: BLE001
        d = dict(r.__dict__)
        d.pop("trace", None)
    d.setdefault("instance_id", getattr(r, "instance_id", None) or d.get("task_id"))
    return d


def collect_rows(results, results_dir: Path, gold: dict[str, dict]) -> list[dict[str, Any]]:
    d = data_dir()
    rows = []
    for r in results:
        rd = _result_dict(r)
        iid = rd.get("instance_id") or rd.get("task_id")
        tpath = rd.get("trace_path") or str(results_dir / "traces" / f"trace_{iid}.json")
        events = load_trace(tpath) if tpath and Path(tpath).exists() else []
        patch = rd.get("agent_patch") or ""
        scopes = diff_scopes(patch, d / "snapshots" / f"{iid}.tgz") if patch.strip() else []
        row = task_metrics(rd, events, gold=gold.get(iid), agent_patch_scopes=scopes)
        row["trace_found"] = bool(events)
        row["trace_compaction_entries"] = sum(1 for e in events if e.type == "compaction")
        row.update({f"cmp_{k}": v for k, v in rp.task_compaction_stats(
            iid, [e.timestamp for e in events if e.type == "tool_call"]).items()})
        rows.append(row)
    return rows


def compaction_summary(rows: list[dict[str, Any]]) -> dict[str, Any]:
    """Run-level answer to: how often/when does compaction fire, what does it drop, is it in the trace?"""
    fired = [r for r in rows if r.get("cmp_compactions")]
    firsts = sorted(r["cmp_first_compaction_at_call"] for r in fired if r.get("cmp_first_compaction_at_call") is not None)
    surv = [r["cmp_marker_survival"] for r in fired if r.get("cmp_marker_survival") is not None]
    return {
        "tasks": len(rows), "tasks_with_compaction": len(fired),
        "compactions_total": sum(r.get("cmp_compactions") or 0 for r in rows),
        "first_compaction_at_call_median": firsts[len(firsts) // 2] if firsts else None,
        "first_compaction_at_call_values": firsts,
        "function_calls_dropped": sum(r.get("cmp_function_calls_dropped") or 0 for r in rows),
        "function_responses_dropped": sum(r.get("cmp_function_responses_dropped") or 0 for r in rows),
        "skill_responses_dropped": sum(r.get("cmp_skill_responses_dropped") or 0 for r in rows),
        "task_prompt_compacted_tasks": sum(bool(r.get("cmp_task_prompt_compacted")) for r in fired),
        "requests_after_compaction_with_task_prompt": sum(r.get("cmp_after_with_task_prompt") or 0 for r in fired),
        "requests_after_compaction": sum(r.get("cmp_requests_after_first_compaction") or 0 for r in fired),
        "marker_survival_mean": round(sum(surv) / len(surv), 4) if surv else None,
        "after_marker_in_prompt": sum(r.get("cmp_after_marker_in_prompt") or 0 for r in fired),
        "after_marker_in_summary_only": sum(r.get("cmp_after_marker_in_summary_only") or 0 for r in fired),
        "after_marker_other_only": sum(r.get("cmp_after_marker_other_only") or 0 for r in fired),
        "after_with_original_prompt": sum(r.get("cmp_after_with_original_prompt") or 0 for r in fired),
        "summaries": sum(r.get("cmp_summaries") or 0 for r in rows),
        "summaries_with_marker": sum(r.get("cmp_summaries_with_marker") or 0 for r in rows),
        "context_errors": sum(r.get("cmp_context_errors") or 0 for r in rows),
        "trace_compaction_entries_total": sum(r.get("trace_compaction_entries") or 0 for r in rows),
        "markers": "per task: hint header, gold file path(s), gold function short names",
    }


def _markers_for(g: dict) -> list[str]:
    out = [HINT_HEADER] + list(g.get("src_files") or [])
    out += [f["qualname"].split(".")[-1] for f in g.get("functions") or []]
    return list(dict.fromkeys(out))


def run_condition(agent_dir: Path, models, adapters, task_ids: list[str], gold: dict[str, dict], condition: str,
                  replicate: int, concurrency: int, tag: str = "") -> dict[str, Any]:
    from swegemma.evaluate import Evaluator
    model_short = agent_dir.name.replace("agent_", "")
    tasks_path = write_task_copy(task_ids, gold, condition)
    rdir = PRIVATE / "runs" / model_short / (f"{condition}_r{replicate}" + (f"_{tag}" if tag else ""))
    if rdir.exists():
        shutil.rmtree(rdir)
    rdir.mkdir(parents=True)
    cfg = make_eval_config(agent_dir, models, adapters, tasks_path, task_ids, rdir, concurrency)
    rp.install()
    rp.reset()
    for iid in task_ids:
        rp.MARKERS[iid] = _markers_for(gold.get(iid) or {})
    t0 = time.perf_counter()
    res = run_sync(Evaluator(cfg).run)
    batch = time.perf_counter() - t0
    rows = collect_rows(res.task_results, rdir, gold)
    n_summaries = rp.dump_summaries(rdir / "summaries.jsonl")   # summary TEXT: private, never committed
    agg = aggregate(rows)
    agg["compaction"] = compaction_summary(rows)
    print(f"[{model_short} {condition} r{replicate}] resolved {agg['resolved']}/{agg['n_tasks']} "
          f"batch {batch/60:.1f} min, mean wall {agg['wall_s_mean']}s, calls {agg['tool_calls_mean']}, "
          f"compactions {agg['compaction']['compactions_total']} (in trace: {agg['compaction']['trace_compaction_entries_total']})")
    return {"model": model_short, "condition": condition, "replicate": replicate, "concurrency": concurrency,
            "batch_wall_s": round(batch, 1), "summaries_saved": n_summaries, "aggregate": agg, "rows": rows}


def save_summary(path: Path, summary: dict[str, Any]) -> None:
    runs = summary.get("runs", [])
    comps = []
    base = {(r["model"]): r for r in runs if r["condition"] == "O0" and r["replicate"] == 1}
    for r in runs:
        b = base.get(r["model"])
        if b is not None and r is not b:
            comps.append({"model": r["model"], "a": "O0_r1", "b": f"{r['condition']}_r{r['replicate']}",
                          **paired_flips(b["rows"], r["rows"])})
    summary["paired_vs_O0_r1"] = comps
    summary["saved_utc"] = _dt.datetime.utcnow().isoformat(timespec="seconds")
    path.write_text(json.dumps(summary, indent=1))
    print(f"saved {path}")


# --------------------------------------------------------------------------- CPU gold check

def gold_check(task_ids: list[str], out_path: Path, timeout_s: int = 300, baseline: bool = True) -> dict[str, Any]:
    """CPU-only: apply each task's GOLD patch in the scorer's Phase-2 sandbox and record resolved; also run the
    same verification with NO patch (baseline). valid = gold passes AND no-patch fails. Numbers only."""
    from adk_submission.registry import ModelRegistry
    from swegemma.config import EvalConfig
    from swegemma.deduplication import resolve_task_snapshot_paths
    from swegemma.evaluate import Evaluator
    from swegemma.harness.verification import verify_task
    from swegemma.models import load_tasks as sg_load

    d = data_dir()
    PRIVATE.mkdir(parents=True, exist_ok=True)
    cfg = EvalConfig(tasks_path=d / "tasks.jsonl", snapshots_dir=d / "snapshots", results_dir=PRIVATE / "gold_check",
                     submission_dir=d / "sample_submission", models=ModelRegistry(), sandbox="subprocess",
                     timeout_seconds=timeout_s, verification_timeout_seconds=timeout_s, wheels_dir=d / "wheels",
                     graph_dir=str(d / "graphs"), embeddings_dir=str(d / "embeddings"), display_mode="quiet")
    ev = Evaluator(cfg)
    tasks = {t.instance_id: t for t in sg_load(d / "tasks.jsonl")}
    rows = []
    for i, iid in enumerate(task_ids, 1):
        t = tasks[iid]
        snap, base, patch = resolve_task_snapshot_paths(d / "snapshots", iid, t.repo)
        t0 = time.perf_counter()
        out_text = ""
        try:
            r = run_sync(verify_task, ev.docker, cfg, t, snap, base_snapshot_path=base, patch_path=patch,
                         agent_patch=t.patch, agent_error=None, trace=None, start_time=t0)
            ok, err = bool(r.resolved), (r.error or "")[:160] or None
            out_text = r.test_output or ""
        except Exception as e:  # noqa: BLE001
            ok, err = False, f"{type(e).__name__}: {e}"[:160]
        dt = time.perf_counter() - t0
        base_ok = None
        if baseline:  # the same tests with NO patch must fail, else any harmless edit "solves" the task
            try:
                rb = run_sync(verify_task, ev.docker, cfg, t, snap, base_snapshot_path=base, patch_path=patch,
                              agent_patch="", agent_error=None, trace=None, start_time=time.perf_counter())
                base_ok = bool(rb.resolved)
            except Exception:  # noqa: BLE001
                base_ok = None
        cls, mods = classify_gold_failure(ok, err, out_text)
        rows.append({"instance_id": iid, "repo": t.repo, "gold_resolved": ok, "baseline_resolved": base_ok,
                     "valid": bool(ok and base_ok is False), "verify_s": round(dt, 1), "error": err,
                     "failure_class": cls, "missing_modules": mods})  # for missing_fixture: fixture names
        print(f"[{i}/{len(task_ids)}] {iid} gold={ok} no_patch={base_ok} {cls} {mods or ''} {dt:.0f}s {err or ''}")
        out_path.write_text(json.dumps({"rows": rows, "summary": gold_check_summary(rows), "env": env_info()}, indent=1))
    return {"rows": rows, "summary": gold_check_summary(rows)}


_MISSING_MOD = re.compile(r"No module named '([A-Za-z0-9_\.]+)'")
_FIXTURE = re.compile(r"fixture '([A-Za-z0-9_]+)' not found|recursive dependency involving fixture '([A-Za-z0-9_]+)'")


def classify_gold_failure(ok: bool, err: str | None, test_output: str) -> tuple[str, list[str]]:
    """Why a gold patch failed (module names / fixture names only; test output is not stored)."""
    if ok:
        return "resolved", []
    mods = sorted({m.split(".")[0] for m in _MISSING_MOD.findall(test_output or "")})
    if mods:
        return "missing_module", mods
    fx = sorted({a or b for a, b in _FIXTURE.findall(test_output or "")})
    if fx:
        return "missing_fixture", fx
    e = err or ""
    if "Failed to apply" in e:
        return "patch_apply_failed", []
    if "TimeoutExceeded" in e or "timed out" in e.lower():
        return "timeout", []
    if e:
        return "harness_error", []
    return "test_failure", []


def gold_check_summary(rows: list[dict[str, Any]]) -> dict[str, Any]:
    from collections import Counter
    by_repo: dict[str, dict[str, int]] = {}
    for r in rows:
        d = by_repo.setdefault(r.get("repo", "?"), {"checked": 0, "gold_resolved": 0})
        d["checked"] += 1
        d["gold_resolved"] += int(r["gold_resolved"])
        d["valid"] = d.get("valid", 0) + int(bool(r.get("valid")))
    mods = Counter(m for r in rows if r["failure_class"] == "missing_module" for m in r.get("missing_modules") or [])
    fixtures = Counter(m for r in rows if r["failure_class"] == "missing_fixture" for m in r.get("missing_modules") or [])
    fastapi_inline_or_dirty = sum(1 for r in rows if r.get("repo") == "fastapi/fastapi" and r["failure_class"] == "missing_module"
                                  and {"inline_snapshot", "dirty_equals"} & set(r.get("missing_modules") or []))
    return {"checked": len(rows), "gold_resolved": sum(r["gold_resolved"] for r in rows),
            "baseline_resolved_without_patch": sum(1 for r in rows if r.get("baseline_resolved")),
            "valid": sum(1 for r in rows if r.get("valid")), "by_repo": by_repo,
            "failure_classes": dict(Counter(r["failure_class"] for r in rows)), "missing_modules": dict(mods),
            "missing_fixtures": dict(fixtures),
            "fastapi_failing_on_inline_snapshot_or_dirty_equals": fastapi_inline_or_dirty}


def select_from_gold_check(gold_all: dict[str, dict], check_rows: list[dict[str, Any]], seed: int = 20261031) -> dict[str, Any]:
    """Pre-registered order: per repo, eligible VALID tasks (gold passes and no-patch fails HERE), seeded shuffle.
    smoke3 = first of each repo. IDs only."""
    import random
    from pilot_c.select_tasks import QUOTA, eligible
    ok = {r["instance_id"] for r in check_rows if r.get("valid", r["gold_resolved"])}
    rng = random.Random(seed)
    order: dict[str, list[str]] = {}
    for repo in QUOTA:
        pool = sorted(i for i, g in gold_all.items() if g["repo"] == repo and eligible(g) and i in ok)
        rng.shuffle(pool)
        order[repo] = pool
    smoke = [v[0] for v in order.values() if v]
    # fill rule (pre-registered): if a repo has no valid task, take the next valid tasks round-robin
    # from the other repos' orders, so the timing set always has 3 tasks when 3 exist
    k = 1
    while len(smoke) < 3 and any(len(v) > k for v in order.values()):
        for v in order.values():
            if len(v) > k and len(smoke) < 3:
                smoke.append(v[k])
        k += 1
    return {"seed": seed, "eligible_valid_in_order": order, "smoke3": smoke, "valid_all": sorted(ok)}


def find_input_json(name: str) -> Path:
    """Locate an attached notebook output (e.g. C00_gold_check.json from notebooks/C00_gold_check__CPU.ipynb)
    anywhere under /kaggle/input, independent of the slug Kaggle gives the attached notebook."""
    root = os.environ.get("PILOT_INPUT_ROOT", "/kaggle/input")
    hits = sorted(glob.glob(f"{root}/**/{name}", recursive=True))
    if not hits:
        producer = {"C00_gold_check.json": "notebooks/C00_gold_check__CPU.ipynb",
                    "probe_records.jsonl": "notebooks/C03_probe__L4x4.ipynb",
                    "probe_contexts.json": "notebooks/C03_probe__L4x4.ipynb",
                    "C03_probe.json": "notebooks/C03_probe__L4x4.ipynb"}.get(name, "the notebook that writes it")
        raise FileNotFoundError(f"{name} not found under {root}: attach the output of {producer} as an input first "
                                "(see notebooks/README.md)")
    return Path(hits[0])

In [ ]:
%%writefile /kaggle/working/pilot/pilot_c/task_ids.json
{
 "status": "Candidates drawn with the pre-registered rule, pre-filtered by a LOCAL gold check. The pilot notebook re-checks gold on Kaggle and fills each repo quota in this order.",
 "seed": 20261031,
 "rule": "eligible = exactly 1 changed source file that already exists at base_commit,\n             1-2 enclosing functions/classes (function-level oracle well defined),\n             gold patch <= 40 changed lines, >= 1 scorer-required test name,\n             repo in {fastapi, rich, requests},\n             and the GOLD patch resolves the task in the sandbox (gold check JSON, if given)\n  pilot15  = stratified random sample: fastapi 7, rich 6, requests 2 (seeded)\n  reserve  = next 4 per repo from the same shuffled order (replace tasks whose gold\n             patch fails the CPU gold check)\n  smoke3   = first pilot task of each repo (used for the smoke test and size timing)",
 "quota": {
  "fastapi/fastapi": 7,
  "Textualize/rich": 6,
  "psf/requests": 2
 },
 "candidates_in_order": {
  "fastapi/fastapi": [
   "fastapi_9753",
   "fastapi_15589",
   "fastapi_5077",
   "fastapi_14479",
   "fastapi_14463",
   "fastapi_14301",
   "fastapi_5624",
   "fastapi_13920",
   "fastapi_14794",
   "fastapi_9425",
   "fastapi_14873"
  ],
  "Textualize/rich": [
   "rich_3518",
   "rich_3944",
   "rich_4075",
   "rich_4076",
   "rich_3480",
   "rich_3067",
   "rich_3894",
   "rich_3882",
   "rich_4077",
   "rich_3470"
  ],
  "psf/requests": [
   "requests_7309",
   "requests_7427",
   "requests_7315"
  ]
 },
 "pilot15": [
  "fastapi_9753",
  "fastapi_15589",
  "fastapi_5077",
  "fastapi_14479",
  "fastapi_14463",
  "fastapi_14301",
  "fastapi_5624",
  "rich_3518",
  "rich_3944",
  "rich_4075",
  "rich_4076",
  "rich_3480",
  "rich_3067",
  "requests_7309",
  "requests_7427"
 ],
 "reserve": [
  "fastapi_13920",
  "fastapi_14794",
  "fastapi_9425",
  "fastapi_14873",
  "rich_3894",
  "rich_3882",
  "rich_4077",
  "rich_3470",
  "requests_7315"
 ],
 "eligible_pool_sizes_after_local_gold_check": {
  "fastapi/fastapi": 18,
  "Textualize/rich": 22,
  "psf/requests": 3
 },
 "smoke3_gpu": [
  "fastapi_15588",
  "rich_3454",
  "requests_7309"
 ]
}

In [ ]:
%%writefile /kaggle/working/pilot/pilot_c/probe.py
"""Information-ladder probe: per-task records, canonical actions, rung texts, controls, classification.

Pure CPU code (no harness imports). Implements docs/PLAN.md §3-§5 and the Level B record of
docs/PROBE_INPUT.md (paper/main). Outputs derived from task data stay private (on Kaggle).
"""
from __future__ import annotations

import json
import re
import tarfile
from pathlib import Path
from typing import Any

from common.diffs import FileDiff, is_test_path, parse_unified_diff, required_test_names
from common.locate import collect_scopes, module_path, report_scope
from pilot_c.hints import _HEADER as LOC_HEADER

MAX_READ_LINES = 150
MAX_READ_CHARS = 10000
TEST_SOURCE_MAX_LINES = 60

LADDER = ("L0", "L1", "L2", "L3", "L4", "L5")
FIND_CONDS = ("L0", "L1", "L2", "T", "W", "G-json", "G-edges", "G-adj", "X-file", "X-func", "X-graph")
USE_CONDS = ("L0", "L1", "L2", "L3", "L4", "L5", "T", "W", "G-json", "G-edges", "G-adj",
             "X-file", "X-func", "X-graph", "X-test", "X-words")
D2_FIND = ("L0", "L1", "L2", "G-json", "G-edges", "X-func")
D2_USE = ("L0", "L2", "L3", "L4", "L5", "T", "W", "G-json", "X-func", "X-test")
GRAPH_CONDS = {"G-json": "json", "G-edges": "edges", "G-adj": "adj", "X-graph": "json", "G-null": "json"}
# Follow-up (notebooks/C04_probe_followup__L4x4.ipynb, PLAN §11 items 10-13; exploratory, added after C03's results):
# G-null = a graph turn in the same format as G-json whose nodes are all in another file (another task's gold subgraph,
# same repo, length-matched), so it carries the tool-response ending without this task's location.
FOLLOWUP_D1 = (("find", "G-null"), ("use", "G-null"))
FOLLOWUP_D2 = (("find", "X-graph"), ("find", "G-null"))
RESCORE_USE = ("L0", "L2", "L5", "W")
RESCORE_FIND = ("L0", "L1", "L2", "G-json", "X-graph")   # PLAN §11 item 14 (2026-11-02)
WORDS_CONDS = {"L4", "L5", "W", "X-words"}


# --------------------------------------------------------------------------- snapshot access

def snapshot_scan(tgz: str | Path, wanted: set[str]) -> tuple[dict[str, str], list[str]]:
    """One pass over the archive: text of `wanted` files + list of all non-test .py paths."""
    texts: dict[str, str] = {}
    py: list[str] = []
    with tarfile.open(tgz, mode="r|gz") as tf:
        for m in tf:
            name = m.name[2:] if m.name.startswith("./") else m.name
            if not m.isfile() or name.startswith(".git/"):
                continue
            if name.endswith(".py") and not is_test_path(name):
                py.append(name)
            if name in wanted:
                fh = tf.extractfile(m)
                if fh is not None:
                    texts[name] = fh.read().decode("utf-8", errors="replace")
    return texts, sorted(py)


def harness_read_output(filepath: str, text: str, start: int, end: int) -> str:
    """Exact swegemma 0.2.11 read_file response (tools/workspace.py) for a line range."""
    lines = text.splitlines()
    total = len(lines)
    s, e = max(start, 1), min(end, total)
    trunc = False
    if e - s + 1 > MAX_READ_LINES:
        e, trunc = s + MAX_READ_LINES - 1, True
    snippet = "\n".join(lines[s - 1:e])
    if len(snippet) > MAX_READ_CHARS:
        snippet, trunc = snippet[:MAX_READ_CHARS], True
    return json.dumps({"status": "ok", "filepath": filepath, "content": snippet, "start_line": s, "end_line": e,
                       "total_lines": total, "is_truncated": trunc}, ensure_ascii=False)


# --------------------------------------------------------------------------- canonical edit

def _new_region(lines: list[str], fd: FileDiff, a: int, b: int) -> list[str]:
    removed = set(fd.removed_old_lines)
    ins: dict[int, list[str]] = {}
    for x in fd.insertions:
        ins.setdefault(x.after_old, []).extend(x.lines)
    out: list[str] = []
    for ln in range(a, b + 1):
        if ln not in removed:
            out.append(lines[ln - 1])
        out.extend(ins.get(ln, []))
    return out


def canonical_edit(base: str, fd: FileDiff, fs: int, fe: int) -> dict[str, Any] | None:
    """old/new strings for the gold change inside lines [fs, fe] of the base file (PLAN §4.3)."""
    lines = base.splitlines()
    rem = [ln for ln in fd.removed_old_lines if fs <= ln <= fe]
    ins = [x for x in fd.insertions if fs <= x.after_old <= fe and any(l.strip() for l in x.lines)]
    if not rem and not ins:
        return None
    pts = rem + [x.after_old for x in ins]
    a, b = max(1, min(pts)), min(len(lines), max(pts))
    for _ in range(12):
        old = "\n".join(lines[a - 1:b])
        if old.strip() and base.count(old) == 1:
            break
        if b < min(len(lines), fe + 3):
            b += 1
        elif a > max(1, fs - 3):
            a -= 1
        else:
            return None
    else:
        return None
    new = "\n".join(_new_region(lines, fd, a, b))
    while not new.strip() and (a > 1 or b < len(lines)):  # pure deletion: keep a neighbouring anchor line
        if a > 1:
            a -= 1
        else:
            b += 1
        old = "\n".join(lines[a - 1:b])
        new = "\n".join(_new_region(lines, fd, a, b))
    if new == old or base.count(old) != 1:
        return None
    changed = sorted(set(rem) | {x.after_old for x in ins})
    outside = sum(1 for ln in fd.removed_old_lines if not fs <= ln <= fe) + \
        sum(1 for x in fd.insertions if not fs <= x.after_old <= fe and any(l.strip() for l in x.lines))
    return {"old_string": old, "new_string": new, "old_start": a, "old_end": b,
            "changed_lines": [min(changed), max(changed)], "outside_edits": outside}


def apply_edit(base: str, old: str, new: str) -> tuple[str | None, str]:
    """Apply an edit_file call. Uses the harness's own 4-tier apply_replacement (adk-eval-core) when it is
    installed (Kaggle, dry runs); otherwise a local exact + whitespace-flexible approximation."""
    try:
        from adk_eval_core.editing.edit import apply_replacement
    except ImportError:
        apply_replacement = None
    if apply_replacement is not None:
        r = apply_replacement(base, old, new)
        ok = not getattr(r, "error_message", None) and getattr(r, "occurrences", 0) == 1
        return (r.new_content if ok else None), (getattr(r, "strategy", "harness") if ok else "none")
    if old and base.count(old) == 1:
        return base.replace(old, new, 1), "exact"
    bl = base.splitlines()
    ol = [x.strip() for x in old.splitlines()]
    if not ol:
        return None, "none"
    hits = [i for i in range(len(bl) - len(ol) + 1) if [x.strip() for x in bl[i:i + len(ol)]] == ol]
    if len(hits) != 1:
        return None, "none"
    i = hits[0]
    ind = len(bl[i]) - len(bl[i].lstrip())
    nl = new.splitlines()
    nind = (len(nl[0]) - len(nl[0].lstrip())) if nl else 0
    shifted = [(" " * max(0, ind + len(x) - len(x.lstrip()) - nind) + x.lstrip()) if x.strip() else x for x in nl]
    return "\n".join(bl[:i] + shifted + bl[i + len(ol):]) + ("\n" if base.endswith("\n") else ""), "flexible"


def _norm(text: str) -> str:
    return "\n".join(l.rstrip() for l in text.strip().splitlines())


# --------------------------------------------------------------------------- test source / diff

def test_source(test_patch: str, names: list[str] | None = None, max_lines: int = TEST_SOURCE_MAX_LINES) -> str:
    """The reproduction-test rung: test_patch as it reads AFTER applying it (context + added lines,
    no removed lines), per file and hunk, capped at `max_lines`. Uniform for new test functions and
    for changes to test data/parametrization (where the scorer's test name may not enclose the change)."""
    out: list[str] = []
    for fd_block in re.split(r"(?m)^(?=--- )", test_patch):
        if not fd_block.strip():
            continue
        m = re.search(r"(?m)^\+\+\+ b/(.+)$", fd_block)
        out.append(f"# {m.group(1) if m else '?'}")
        for hunk in re.split(r"(?m)^(?=@@ )", fd_block)[1:]:
            body = [l[1:] for l in hunk.splitlines()[1:] if l[:1] in ("+", " ")]
            out += body + ["# ..."]
    return "\n".join(out[:max_lines]).rstrip()


def file_diff_text(patch: str, path: str) -> str:
    for block in re.split(r"(?m)^(?=--- )", patch):
        if block.startswith(f"--- a/{path}\n") or f"\n+++ b/{path}\n" in block[:400]:
            return block.rstrip("\n")
    return ""


# --------------------------------------------------------------------------- Level B record

def build_probe_record(task: dict, gold: dict, snapshot_tgz: str | Path) -> tuple[dict | None, str]:
    """Return (record, reason). record is None when the task is excluded (PLAN §9, rule 3)."""
    if len(gold.get("src_files") or []) != 1 or gold.get("new_src_files") or not gold.get("functions"):
        return None, "not eligible"
    path = gold["src_files"][0]
    texts, py = snapshot_scan(snapshot_tgz, {path})
    base = texts.get(path)
    if base is None:
        return None, "gold file missing in snapshot"
    fd = next(f for f in parse_unified_diff(task["patch"]) if f.path == path)
    scopes = {s.qualname: s for s in collect_scopes(base)}
    edit = fn = None
    for f in sorted(gold["functions"], key=lambda f: f["def_line"]):
        sc = scopes.get(f["qualname"])
        if sc is None:
            continue
        e = canonical_edit(base, fd, sc.start, sc.end)
        if e is not None:
            edit, fn = e, (f, sc)
            break
    if edit is None:
        return None, "no canonical edit"
    f, sc = fn
    total = len(base.splitlines())
    rs, re_ = sc.start, min(sc.end, sc.start + MAX_READ_LINES - 1)
    if edit["changed_lines"][1] > re_:
        return None, "change outside read span"
    names = required_test_names(task["test_patch"])
    tfiles = [fd2.path for fd2 in parse_unified_diff(task["test_patch"]) if fd2.new_path]
    rec = {
        "instance_id": task["instance_id"], "repo": task["repo"], "base_commit": task["base_commit"],
        "problem_statement": task["problem_statement"], "hints_text": task.get("hints_text") or "",
        "gold": {"file": path, "qualname": f["qualname"], "dotted": f["dotted"], "kind": f["kind"],
                 "start_line": sc.start, "def_line": sc.def_line, "end_line": sc.end, "file_total_lines": total},
        "find_target": {"tool": "read_file", "args": {"filepath": path, "start_line": rs, "end_line": re_}},
        "read_output": harness_read_output(path, base, rs, re_),
        "edit_target": {"tool": "edit_file", "args": {"filepath": path, "old_string": edit["old_string"],
                                                      "new_string": edit["new_string"]}},
        "edit_region": [edit["old_start"], edit["old_end"]], "changed_lines": edit["changed_lines"],
        "outside_edits": edit["outside_edits"],
        "diff": file_diff_text(task["patch"], path),
        "test": {"files": tfiles, "names": names, "files_exist": not (set(gold.get("new_test_files") or []) >= set(tfiles)),
                 "source": test_source(task["test_patch"])},
        "graph": None, "words": None, "wrong": {},
        "_base_text": base, "_py_files": py,  # private helpers, dropped before saving
    }
    rec["wrong"].update(_wrong_location(rec, base, py))
    return rec, "ok"


# --------------------------------------------------------------------------- rung texts

def loc_file_text(path: str) -> str:
    return f"{LOC_HEADER}\n- The fix changes `{path}`."


def loc_func_text(path: str, dotted: str, kind: str, def_line: int) -> str:
    what = "class body of" if kind == "class" else "function"
    return f"{loc_file_text(path)}\n- In `{path}`: {what} `{dotted}` (defined at line {def_line})."


def test_text(rec: dict, source: str | None = None, files: list[str] | None = None, exists: bool | None = None) -> str:
    files = rec["test"]["files"] if files is None else files
    exists = rec["test"]["files_exist"] if exists is None else exists
    status = "exists" if exists else "does not exist yet; it is added when your fix is verified"
    src = rec["test"]["source"] if source is None else source
    return (f"## Reproduction test (verified)\nThe fix is verified by this test from `{', '.join(files)}` "
            f"(the file {status}):\n```python\n{src}\n```")


def words_text(words: str) -> str:
    return f"## The change, in words\n{words}"


def diff_text(diff: str) -> str:
    return f"## The exact change\n```diff\n{diff}\n```"


def rung_text(rec: dict, cond: str) -> str | None:
    """Text appended to the problem statement for `cond` (None = graph-only or none). None if unavailable."""
    g, w = rec["gold"], rec["wrong"]
    words = rec.get("words")
    parts: list[str] = []
    if cond in ("L1", "L2", "L3", "L4", "L5"):
        parts.append(loc_file_text(g["file"]) if cond == "L1" else loc_func_text(g["file"], g["dotted"], g["kind"], g["def_line"]))
    if cond in ("L3", "L4", "L5", "T"):
        parts.append(test_text(rec))
    if cond in ("L4", "L5", "W"):
        if not words:
            return None
        parts.append(words_text(words))
    if cond == "L5":
        parts.append(diff_text(rec["diff"]))
    if cond == "X-file":
        parts.append(loc_file_text(w["file"]))
    if cond == "X-func":
        parts.append(loc_func_text(g["file"], w["dotted"], w["kind"], w["def_line"]))
    if cond == "X-test":
        if not w.get("test_source"):
            return None
        parts += [loc_func_text(g["file"], g["dotted"], g["kind"], g["def_line"]),
                  test_text(rec, w["test_source"], w["test_files"], w["test_exists"])]
    if cond == "X-words":
        if not w.get("words") or not words:
            return None
        parts += [loc_func_text(g["file"], g["dotted"], g["kind"], g["def_line"]), test_text(rec), words_text(w["words"])]
    return "\n\n".join(parts) if parts else ""


def insert_hint(user_text: str, problem_statement: str, hint: str) -> tuple[str, str]:
    """Append `hint` to the problem statement inside the harness task prompt, exactly as
    pilot_c.hints.rewrite_task would have produced it. Returns (text, method)."""
    if not hint:
        return user_text, "none"
    new_ps = problem_statement.rstrip() + "\n\n" + hint + "\n"
    if problem_statement in user_text:
        return user_text.replace(problem_statement, new_ps, 1), "exact"
    m = re.search(r"\n\n## ", user_text[user_text.find("Problem Statement:"):])
    if m:
        k = user_text.find("Problem Statement:") + m.start()
        return user_text[:k] + "\n\n" + hint + user_text[k:], "before-next-section"
    return user_text + "\n\n" + hint, "append"


# --------------------------------------------------------------------------- graph encodings

def encode_graph(focal: str, nodes: list[str], edges: list[dict], fmt: str) -> str:
    """Same content, three layouts (PLAN §4.2). nodes/edges as from get_code_subgraph."""
    nodes = [focal] + [n for n in nodes if n != focal]
    if fmt == "json":
        return json.dumps({"status": "ok", "nodes": nodes, "edges": edges, "node_count": len(nodes),
                           "edge_count": len(edges), "is_truncated": False}, ensure_ascii=False)
    if fmt == "edges":
        lines = [f"{e['from']} -[{e.get('type', 'calls')}]-> {e['to']}" for e in edges]
        return "\n".join(lines) if lines else f"{focal} (no edges)"
    if fmt == "adj":
        out = []
        for n in nodes:
            calls = [e["to"] for e in edges if e["from"] == n]
            by = [e["from"] for e in edges if e["to"] == n]
            out.append(f"{n}: calls {', '.join(calls) or '-'}; called by {', '.join(by) or '-'}")
        return "\n".join(out)
    raise ValueError(fmt)


# --------------------------------------------------------------------------- wrong controls

def _closest(target_len: int, cands: list[tuple[str, Any]], render) -> tuple[Any, float] | None:
    best = None
    for key, val in sorted(cands, key=lambda kv: kv[0]):
        r = len(render(val)) / max(1, target_len)
        d = abs(r - 1)
        if best is None or d < best[2]:
            best = (val, r, d)
    return (best[0], round(best[1], 3)) if best else None


def _wrong_location(rec: dict, base: str, py: list[str]) -> dict:
    g = rec["gold"]
    out: dict[str, Any] = {}
    files = [p for p in py if p != g["file"] and p.split("/")[0] == g["file"].split("/")[0]] or [p for p in py if p != g["file"]]
    pick = _closest(len(loc_file_text(g["file"])), [(p, p) for p in files], loc_file_text)
    if pick:
        out["file"], out["file_len_ratio"] = pick
    mod = module_path(g["file"])
    gold_q = g["qualname"]
    cands = []
    for s in collect_scopes(base):
        rs = report_scope([s])
        if s.qualname == gold_q or gold_q.startswith(s.qualname + ".") or s.qualname.startswith(gold_q + "."):
            continue
        if rs is not s or s.qualname.count(".") > 1:
            continue
        cands.append((s.qualname, {"dotted": f"{mod}.{s.qualname}" if mod else s.qualname, "kind": s.kind,
                                   "def_line": s.def_line, "qualname": s.qualname}))
    pick = _closest(len(loc_func_text(g["file"], g["dotted"], g["kind"], g["def_line"])), cands,
                    lambda v: loc_func_text(g["file"], v["dotted"], v["kind"], v["def_line"]))
    if pick:
        v, r = pick
        out.update({"dotted": v["dotted"], "kind": v["kind"], "def_line": v["def_line"], "qualname": v["qualname"],
                    "func_len_ratio": r})
    return out


def assign_cross_task_controls(records: list[dict]) -> None:
    """X-test and X-words donors: another record of the same repo with the closest text length."""
    for rec in records:
        same = [r for r in records if r["repo"] == rec["repo"] and r["instance_id"] != rec["instance_id"]]
        pick = _closest(len(rec["test"]["source"]), [(r["instance_id"], r) for r in same], lambda r: r["test"]["source"])
        if pick:
            d, ratio = pick
            rec["wrong"].update({"test_from": d["instance_id"], "test_source": d["test"]["source"],
                                 "test_files": d["test"]["files"], "test_exists": d["test"]["files_exist"],
                                 "test_len_ratio": ratio})
        if rec.get("words"):
            wsame = [(r["instance_id"], r) for r in same if r.get("words")]
            pick = _closest(len(rec["words"]), wsame, lambda r: r["words"])
            if pick:
                d, ratio = pick
                rec["wrong"].update({"words_from": d["instance_id"], "words": d["words"], "words_len_ratio": ratio})


def assign_null_graphs(records: list[dict], tol: float = 0.15) -> None:
    """G-null donors: another record's gold subgraph (same repo if possible) with no node in this task's gold module
    and a different gold file; the closest JSON length to this task's G-json wins (ties by task id). Sets
    rec["null_graph"] = {focal, nodes, edges, from, file, qualname, len_ratio, within_tol}."""
    for rec in records:
        g = rec.get("graph")
        if not g:
            continue
        mod = module_path(rec["gold"]["file"])
        target_len = len(encode_graph(g["focal"], g["nodes"], g["edges"], "json"))

        def unrelated(d: dict) -> bool:
            dg = d.get("graph") or {}
            nodes = [dg.get("focal", "")] + list(dg.get("nodes") or [])
            return (d["instance_id"] != rec["instance_id"] and d["gold"]["file"] != rec["gold"]["file"] and bool(dg)
                    and not any(n == mod or n.startswith(mod + ".") for n in nodes if mod))
        pool = [d for d in records if d["repo"] == rec["repo"] and unrelated(d)] or [d for d in records if unrelated(d)]
        pick = _closest(target_len, [(d["instance_id"], d) for d in pool],
                        lambda d: encode_graph(d["graph"]["focal"], d["graph"]["nodes"], d["graph"]["edges"], "json"))
        if pick:
            d, ratio = pick
            rec["null_graph"] = {"focal": d["graph"]["focal"], "nodes": d["graph"]["nodes"], "edges": d["graph"]["edges"],
                                 "from": d["instance_id"], "file": d["gold"]["file"], "qualname": d["gold"]["qualname"],
                                 "len_ratio": ratio, "within_tol": abs(ratio - 1) <= tol}


# --------------------------------------------------------------------------- words rung

def words_prompt(diff: str, strict: bool = False) -> str:
    extra = " Do not copy any code, expression or string literal from the diff." if strict else ""
    return ("Describe the following code change in 1 to 3 plain English sentences for a developer who has to "
            "re-implement it. Say what behaviour changes and where. You may name functions or parameters, but do not "
            f"write code.{extra}\n\n```diff\n{diff}\n```")


def words_leak(words: str, diff: str, min_len: int = 20) -> bool:
    if len(words) > 600:
        return True
    plus = [l[1:].strip() for l in diff.splitlines() if l.startswith("+") and not l.startswith("+++")]
    for l in plus:
        for i in range(0, max(0, len(l) - min_len) + 1):
            if len(l) >= min_len and l[i:i + min_len] in words:
                return True
    return False


# --------------------------------------------------------------------------- depth-2 classification

def _norm_path(p: Any) -> str:
    p = str(p or "").strip().lstrip("/")
    return p[len("workspace/"):] if p.startswith("workspace/") else p


def _parse_args(a: Any) -> dict:
    if isinstance(a, dict):
        return a
    try:
        v = json.loads(a or "{}")
        return v if isinstance(v, dict) else {}
    except (ValueError, TypeError):
        return {}


def classify_find(rec: dict, tool_calls: list[dict]) -> dict:
    g, w = rec["gold"], rec["wrong"]
    out = {"tool_call": bool(tool_calls), "hit_file": False, "hit_func": False, "followed_wrong": False, "tool": None}
    if not tool_calls:
        return out
    fn = tool_calls[0].get("function", {})
    name, args = fn.get("name"), _parse_args(fn.get("arguments"))
    out["tool"] = name

    def touches(path: str, lo: int | None, hi: int | None) -> tuple[bool, bool]:
        if name == "read_file" and _norm_path(args.get("filepath")) == path:
            s = int(args.get("start_line") or 1)
            e = int(args.get("end_line") or s + MAX_READ_LINES - 1)
            return True, (lo is not None and not (e < lo or s > hi))
        if name == "run_command" and path in str(args.get("command", "")):
            return True, False
        return False, False

    out["hit_file"], out["hit_func"] = touches(g["file"], g["start_line"], g["end_line"])
    if name in ("get_code_neighbors", "get_code_subgraph", "search_similar_code"):
        blob = json.dumps(args)
        out["hit_func"] = g["qualname"].split(".")[-1] in blob
    if w.get("file"):
        out["followed_wrong"] = touches(w["file"], None, None)[0]
    if w.get("qualname"):
        out["followed_wrong"] = out["followed_wrong"] or (w["qualname"].split(".")[-1] in json.dumps(args))
    return out


def classify_use(rec: dict, tool_calls: list[dict]) -> tuple[dict, str | None]:
    """Returns (flags, edited_file_text or None). Edited text is private (for CPU test verification)."""
    base = rec["_base_text"]
    g = rec["gold"]
    out = {"tool_call": bool(tool_calls), "tool": None, "edit_gold_file": False, "applies": False,
           "on_target": False, "exact": False, "followed_wrong": False}
    if not tool_calls:
        return out, None
    fn = tool_calls[0].get("function", {})
    name, args = fn.get("name"), _parse_args(fn.get("arguments"))
    out["tool"] = name
    path = _norm_path(args.get("filepath"))
    if name not in ("edit_file", "write_file") or path != g["file"]:
        if rec["wrong"].get("file") and path == rec["wrong"]["file"]:
            out["followed_wrong"] = True
        return out, None
    out["edit_gold_file"] = True
    if name == "edit_file":
        edited, how = apply_edit(base, str(args.get("old_string", "")), str(args.get("new_string", "")))
    else:
        edited, how = str(args.get("content", "")), "write"
    if edited is None:
        return out, None
    out["applies"] = True
    gold_edited, _ = apply_edit(base, rec["edit_target"]["args"]["old_string"], rec["edit_target"]["args"]["new_string"])
    bl, el = base.splitlines(), edited.splitlines()
    i = 0
    while i < min(len(bl), len(el)) and bl[i] == el[i]:
        i += 1
    j = 0
    while j < min(len(bl), len(el)) - i and bl[-1 - j] == el[-1 - j]:
        j += 1
    lo, hi = i + 1, len(bl) - j
    ga, gb = rec["changed_lines"]
    out["on_target"] = not (hi < ga - 1 or lo > gb + 1)
    out["exact"] = gold_edited is not None and _norm(edited) == _norm(gold_edited)
    if rec["wrong"].get("def_line") and rec["wrong"].get("qualname"):
        wl = rec["wrong"]["def_line"]
        out["followed_wrong"] = lo <= wl + 5 and hi >= wl and not out["on_target"]
    return out, edited


def public_record(rec: dict) -> dict:
    """Record without private helper fields (still competition-derived: keep on Kaggle)."""
    return {k: v for k, v in rec.items() if not k.startswith("_")}

In [ ]:
%%writefile /kaggle/working/pilot/pilot_c/capture_server.py
"""Capture server: an OpenAI-compatible stand-in LLM that records the agent's exact requests.

Used by the probe (docs/PLAN.md §3) to obtain byte-identical agent contexts without a GPU:
the real swegemma agent loop runs against this server; for each task the server answers with a
scripted action sequence (read the gold span, then submit_patch, then stop) and stores the full
request bodies. Request 0 is the S-find context, request 1 the S-use context (it contains the real
harness read_file output). Summarizer (no-tools) requests get a short text reply.
"""
from __future__ import annotations

import hashlib
import json
import threading
import time
import uuid
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from typing import Any


def _text(m: dict) -> str:
    c = m.get("content")
    if isinstance(c, list):
        return " ".join(str(x.get("text", "")) for x in c if isinstance(x, dict))
    return str(c or "")


class CaptureServer:
    def __init__(self, routes: dict[str, dict[str, Any]], port: int = 0):
        """routes: task_id -> {"needle": str found in the task prompt, "script": [ {name, arguments} | None ]}"""
        self.routes = routes
        self.captured: dict[str, list[dict]] = {k: [] for k in routes}
        self.unrouted = 0
        self.lock = threading.Lock()
        outer = self

        class H(BaseHTTPRequestHandler):
            def log_message(self, *a):
                pass

            def _send(self, obj: dict) -> None:
                b = json.dumps(obj).encode()
                self.send_response(200)
                self.send_header("Content-Type", "application/json")
                self.send_header("Content-Length", str(len(b)))
                self.end_headers()
                self.wfile.write(b)

            def do_GET(self):
                self._send({"object": "list", "data": [{"id": "capture", "object": "model"}]})

            def do_POST(self):
                req = json.loads(self.rfile.read(int(self.headers.get("Content-Length", 0))) or b"{}")
                msgs = req.get("messages", [])
                now = int(time.time())
                if not req.get("tools"):
                    return self._send(_completion("Summary.", None, now))
                first_user = next((_text(m) for m in msgs if m.get("role") == "user"), "")
                tid = next((k for k, r in outer.routes.items() if r["needle"] and r["needle"] in first_user), None)
                if tid is None:
                    with outer.lock:
                        outer.unrouted += 1
                    return self._send(_completion("Done.", None, now))
                with outer.lock:
                    step = len(outer.captured[tid])
                    outer.captured[tid].append(req)
                script = outer.routes[tid]["script"]
                item = script[step] if step < len(script) else None
                if item is None:
                    return self._send(_completion("Done.", None, now))
                return self._send(_completion(None, item, now))

        self.httpd = ThreadingHTTPServer(("127.0.0.1", port), H)
        threading.Thread(target=self.httpd.serve_forever, daemon=True).start()

    @property
    def url(self) -> str:
        return f"http://127.0.0.1:{self.httpd.server_address[1]}/v1"

    def stop(self) -> None:
        self.httpd.shutdown()
        self.httpd.server_close()


def _completion(text: str | None, call: dict | None, now: int) -> dict:
    if call is None:
        msg, fin = {"role": "assistant", "content": text}, "stop"
    else:
        msg = {"role": "assistant", "content": None, "tool_calls": [{
            "id": "call_" + uuid.uuid4().hex[:10], "type": "function",
            "function": {"name": call["name"], "arguments": json.dumps(call["arguments"])}}]}
        fin = "tool_calls"
    return {"id": "chatcmpl-capture", "object": "chat.completion", "created": now, "model": "capture",
            "choices": [{"index": 0, "message": msg, "finish_reason": fin}],
            "usage": {"prompt_tokens": 100, "completion_tokens": 10, "total_tokens": 110}}


def needle_for(problem_statement: str) -> str:
    """A short, distinctive substring of the problem statement as the harness inserts it."""
    ps = problem_statement.strip()
    return ps[:120] if len(ps) >= 20 else hashlib.sha1(ps.encode()).hexdigest()

In [ ]:
%%writefile /kaggle/working/pilot/pilot_c/probe_runner.py
"""Kaggle-side runner for the information-ladder probe (docs/PLAN.md). Runs only on Kaggle / dry runs.

Flow per session:
  1. records  : Level-B probe records for the valid, eligible tasks (CPU)
  2. graph    : harness get_code_neighbors / get_code_subgraph for gold and wrong symbols (CPU)
  3. capture  : real agent loop against the capture server -> exact S-find / S-use contexts (CPU)
  4. per size : start vLLM (starter-identical) -> self-check -> [31b: words] -> depth 1 -> depth 2
Used by notebooks/C03_probe__L4x4.ipynb and notebooks/C03v_test_verify__CPU.ipynb.
Outputs: /kaggle/working/C03_probe.json (numbers + task ids only; commit it),
         /kaggle/working/private/probe_*.jsonl (contexts, records, sampled edits; never commit).
"""
from __future__ import annotations

import concurrent.futures as cf
import copy
import difflib
import json
import os
import time
import urllib.error
import urllib.request
from pathlib import Path
from typing import Any

from pilot_c import kaggle_runner as kr
from pilot_c import probe as pb
from common.server_watchdog import ServerFrozen, TRIPPED, tracked
from pilot_c.capture_server import CaptureServer, needle_for

AGENT_SAMPLING = {"temperature": 0.2, "top_p": 0.95, "max_tokens": 6144,
                  "chat_template_kwargs": {"enable_thinking": True}, "thinking_token_budget": 4096}
SCORE_KW = {"chat_template_kwargs": {"enable_thinking": False}}
MAX_CELL_TOKENS = 28000


# --------------------------------------------------------------------------- http

_OPENER = urllib.request.build_opener(urllib.request.ProxyHandler({}))  # local server: never via a proxy


# per-request timeouts (s): a 28k-token prefill takes ~10 s on 31b; sampling is <= 6,144 new tokens
TIMEOUT_TOKENIZE, TIMEOUT_SCORE, TIMEOUT_SAMPLE, TIMEOUT_WORDS = 60, 180, 900, 300


class ServerHTTPError(RuntimeError):
    """An HTTP error from the inference server, with the response body (vLLM puts the cause there)."""


def _post(url: str, body: dict, timeout: int = TIMEOUT_SCORE) -> dict:
    req = urllib.request.Request(url, data=json.dumps(body).encode(), headers={"Content-Type": "application/json"})
    with tracked():  # counted by the watchdog; fails fast once the server is declared frozen
        try:
            with _OPENER.open(req, timeout=timeout) as r:
                return json.loads(r.read())
        except urllib.error.HTTPError as e:
            detail = e.read().decode("utf-8", errors="replace")[:600]
            raise ServerHTTPError(f"HTTP {e.code} from /{url.split('/', 3)[-1]}: {detail}") from None


def _get(url: str) -> dict:
    with _OPENER.open(url, timeout=60) as r:
        return json.loads(r.read())


def server_root(server) -> str:
    base = getattr(server, "base_url", None) or "http://127.0.0.1:8000"
    base = str(base).rstrip("/")
    return base[:-3] if base.endswith("/v1") else base


# --------------------------------------------------------------------------- 1-2: records and graphs

def build_records(task_ids: list[str]) -> tuple[list[dict], dict[str, str]]:
    d = kr.data_dir()
    tasks = {t["instance_id"]: t for t in kr.load_tasks(d / "tasks.jsonl")}
    gold = kr.build_gold(task_ids)
    recs, excluded = [], {}
    for iid in task_ids:
        rec, why = pb.build_probe_record(tasks[iid], gold[iid], d / "snapshots" / f"{iid}.tgz")
        if rec is None:
            excluded[iid] = why
        else:
            recs.append(rec)
    return recs, excluded


def attach_graphs(recs: list[dict]) -> None:
    """Gold and wrong-symbol subgraphs from the harness's own graph tools (no sandbox needed)."""
    from adk_submission.registry import ModelRegistry
    from swegemma.config import EvalConfig
    from swegemma.context import SwegemmaContext
    from swegemma.models import load_tasks as sg_load
    d = kr.data_dir()
    tasks = {t.instance_id: t for t in sg_load(d / "tasks.jsonl")}
    cfg = EvalConfig(tasks_path=d / "tasks.jsonl", snapshots_dir=d / "snapshots", results_dir=kr.PRIVATE / "graph_ctx",
                     submission_dir=d / "sample_submission", models=ModelRegistry(), sandbox="subprocess",
                     graph_dir=str(d / "graphs"), embeddings_dir=str(d / "embeddings"), display_mode="quiet")
    for rec in recs:
        t = tasks[rec["instance_id"]]
        ctx = SwegemmaContext(docker_manager=None, container_id="", problem_statement="", task=t, repo=t.repo,
                              graph_dir=cfg.graph_dir, embeddings_dir=cfg.embeddings_dir, budget=cfg.budget,
                              harness=cfg.harness)
        if hasattr(ctx, "start_agent_session"):
            ctx.start_agent_session()
        tools = ctx.create_tools()

        def sub(sym: str) -> dict | None:
            try:
                nb = tools["get_code_neighbors"](node=sym, max_neighbors=50)
                nb = json.loads(nb) if isinstance(nb, str) else nb
                neigh = [n.split(";", 1)[-1] for n in nb.get("neighbors", [])]
                neigh = [n for n in neigh if not n.startswith(("tests.", "test."))][:10]
                sg = tools["get_code_subgraph"](nodes=[sym] + neigh)
                sg = json.loads(sg) if isinstance(sg, str) else sg
                return {"focal": sym, "nodes": sg.get("nodes", [sym]), "edges": sg.get("edges", [])}
            except Exception as e:  # noqa: BLE001
                return {"focal": sym, "nodes": [sym], "edges": [], "error": f"{type(e).__name__}"}
        rec["graph"] = sub(rec["gold"]["dotted"])
        if rec["wrong"].get("dotted"):
            rec["wrong"]["graph"] = sub(rec["wrong"]["dotted"])


# --------------------------------------------------------------------------- 3: capture

def capture_contexts(recs: list[dict], budget: dict, concurrency: int = 6) -> dict[str, dict]:
    """Run the real agent loop (sample agent) against the capture server; return per-task contexts."""
    import litellm
    from adk_submission import discover_adapters
    from adk_submission.registry import ModelRegistry
    from adk_submission.resolvers.generation import install_litellm_reasoning_patch
    from google.adk.models.lite_llm import LiteLlm
    from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS

    routes = {r["instance_id"]: {"needle": needle_for(r["problem_statement"]),
                                 "script": [{"name": "read_file", "arguments": r["find_target"]["args"]},
                                            {"name": "submit_patch", "arguments": {}}]} for r in recs}
    srv = CaptureServer(routes)
    try:
        litellm.drop_params = True
        install_litellm_reasoning_patch()
        model = "gemma-4-e4b-it-qat-w4a16-ct"  # alias only; system prompt and tools are size-independent
        agent_dir = kr.prepare_agent_dir(model, budget)
        reg = ModelRegistry()
        reg.register(model, LiteLlm(model="openai/capture", api_base=srv.url, api_key="EMPTY"))
        ad = discover_adapters(str(agent_dir), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
        ids = [r["instance_id"] for r in recs]
        gold = {r["instance_id"]: {"src_files": [r["gold"]["file"]], "functions": []} for r in recs}
        kr.run_condition(agent_dir, reg, ad, ids, gold, "O0", 0, concurrency)
    finally:
        srv.stop()
    out = {}
    for r in recs:
        reqs = srv.captured.get(r["instance_id"], [])
        if len(reqs) >= 2:
            out[r["instance_id"]] = {"find": reqs[0]["messages"], "use": reqs[1]["messages"], "tools": reqs[0].get("tools")}
    return out


# --------------------------------------------------------------------------- message construction

def _content_text(m: dict) -> str:
    c = m.get("content")
    if isinstance(c, list):
        return "".join(str(x.get("text", "")) for x in c if isinstance(x, dict))
    return str(c or "")


def _set_content_text(m: dict, text: str) -> None:
    c = m.get("content")
    if isinstance(c, list):
        done = False
        for x in c:
            if isinstance(x, dict) and "text" in x and not done:
                x["text"], done = text, True
            elif isinstance(x, dict) and "text" in x:
                x["text"] = ""
    else:
        m["content"] = text


def _replace_leaf(obj: Any, old: str, new: str) -> tuple[Any, bool]:
    if isinstance(obj, str):
        return (new, True) if obj == old else (obj, False)
    if isinstance(obj, dict):
        hit = False
        out = {}
        for k, v in obj.items():
            out[k], h = _replace_leaf(v, old, new)
            hit = hit or h
        return out, hit
    if isinstance(obj, list):
        res = [_replace_leaf(v, old, new) for v in obj]
        return [x for x, _ in res], any(h for _, h in res)
    return obj, False


def rewrap_tool_content(captured: str, old_payload: str, new_payload: str) -> str | None:
    """Put `new_payload` into a tool message exactly where the harness put `old_payload`."""
    if captured == old_payload:
        return new_payload
    try:
        dec = json.loads(captured)
    except ValueError:
        return captured.replace(old_payload, new_payload) if old_payload in captured else None
    new, hit = _replace_leaf(dec, old_payload, new_payload)
    if not hit:
        try:
            new, hit = _replace_leaf(dec, json.dumps(json.loads(old_payload)), new_payload)
        except ValueError:
            pass
    if not hit:
        return None
    for ea in (False, True):
        if json.dumps(dec, ensure_ascii=ea) == captured:
            return json.dumps(new, ensure_ascii=ea)
    return json.dumps(new, ensure_ascii=False)


def context_check(rec: dict, ctx: dict) -> dict:
    """Is the captured S-use tool message the harness read of the gold span, as the record predicts?"""
    use = ctx["use"]
    tool_msg = next((m for m in reversed(use) if m.get("role") == "tool"), None)
    ok = tool_msg is not None and rewrap_tool_content(_content_text(tool_msg), rec["read_output"], "X") is not None
    return {"read_output_matches": ok, "n_find_msgs": len(ctx["find"]), "n_use_msgs": len(use)}


def build_messages(rec: dict, ctx: dict, state: str, cond: str) -> tuple[list[dict], dict, str] | None:
    """(messages without target, target assistant message, hint insertion method) or None if the cell is unavailable."""
    base = copy.deepcopy(ctx["find"] if state == "find" else ctx["use"])
    ui = next(i for i, m in enumerate(base) if m.get("role") == "user")
    method = "none"
    if cond in pb.GRAPH_CONDS:
        g = {"X-graph": rec["wrong"].get("graph"), "G-null": rec.get("null_graph")}.get(cond, rec["graph"])
        if not g:
            return None
        payload = pb.encode_graph(g["focal"], g["nodes"], g["edges"], pb.GRAPH_CONDS[cond])
        a_tpl = next(m for m in ctx["use"] if m.get("role") == "assistant" and m.get("tool_calls"))
        t_tpl = next(m for m in ctx["use"] if m.get("role") == "tool")
        a = copy.deepcopy(a_tpl)
        a["tool_calls"] = [copy.deepcopy(a_tpl["tool_calls"][0])]
        a["tool_calls"][0]["id"] = "call_probe_graph"
        a["tool_calls"][0]["function"] = {"name": "get_code_subgraph", "arguments": json.dumps({"nodes": [g["focal"]] + [n for n in g["nodes"] if n != g["focal"]]})}
        t = copy.deepcopy(t_tpl)
        if "tool_call_id" in t:
            t["tool_call_id"] = "call_probe_graph"
        if "name" in t:
            t["name"] = "get_code_subgraph"
        wrapped = rewrap_tool_content(_content_text(t_tpl), rec["read_output"], payload)
        if wrapped is None:
            return None
        _set_content_text(t, wrapped)
        base[ui + 1:ui + 1] = [a, t]
        method = "graph-turn"
    else:
        hint = pb.rung_text(rec, cond)
        if hint is None:
            return None
        text, method = pb.insert_hint(_content_text(base[ui]), rec["problem_statement"], hint)
        _set_content_text(base[ui], text)
    tgt_spec = rec["find_target"] if state == "find" else rec["edit_target"]
    a_tpl = next(m for m in ctx["use"] if m.get("role") == "assistant" and m.get("tool_calls"))
    target = copy.deepcopy(a_tpl)
    target["tool_calls"] = [copy.deepcopy(a_tpl["tool_calls"][0])]
    target["tool_calls"][0]["id"] = "call_probe_target"
    target["tool_calls"][0]["function"] = {"name": tgt_spec["tool"], "arguments": json.dumps(tgt_spec["args"], ensure_ascii=False)}
    return base, target, method


# --------------------------------------------------------------------------- scoring and sampling

def _lcp(a: list[int], b: list[int]) -> int:
    n = 0
    for x, y in zip(a, b):
        if x != y:
            break
        n += 1
    return n


def score(root: str, model: str, msgs: list[dict], tools: Any, target: dict) -> dict:
    """Sum of log-probabilities of the target assistant turn, as the model sees it when generating.

    P = context + generation prompt (what the model conditions on), N = context alone, F = context + target turn.
    "direct": F starts with P (always after a tool response, where the generation prompt adds nothing; and for the
    e2b/e4b template); S = F, target T = F[len(P):].
    "spliced": the 31b and 12b templates add an empty thought block to the generation prompt after a user turn, which
    the rendered assistant turn does not contain. Then A = F[len(N):] (turn header + call), G = P[len(N):]
    (header + empty thought), T = A minus the shared header, and S = P + T: what the model conditions on, plus the call."""
    body = {"model": model, "messages": msgs, "tools": tools, **SCORE_KW}
    P = _post(f"{root}/tokenize", {**body, "add_generation_prompt": True}, TIMEOUT_TOKENIZE)["tokens"]
    N = _post(f"{root}/tokenize", {**body, "add_generation_prompt": False}, TIMEOUT_TOKENIZE)["tokens"]
    F = _post(f"{root}/tokenize", {**body, "messages": msgs + [target], "add_generation_prompt": False},
              TIMEOUT_TOKENIZE)["tokens"]
    # The same call with empty arguments: its tokens agree with the target up to the first token that depends on the
    # arguments. That prefix is the "action" part (take a tool call now, and which tool); the rest is the arguments.
    F0 = _post(f"{root}/tokenize", {**body, "messages": msgs + [_empty_args(target)], "add_generation_prompt": False},
               TIMEOUT_TOKENIZE)["tokens"]
    if len(F) > len(P) and F[:len(P)] == P:  # e.g. after a tool response (P == N), or templates without an empty thought
        T, S, mode = F[len(P):], F, "direct"
        T0 = F0[len(P):]
    elif F[:len(N)] == N and P[:len(N)] == N:
        A, G = F[len(N):], P[len(N):]
        h = _lcp(A, G)
        if h == 0 or h >= len(A):
            return {"ok": False, "why": "prefix-mismatch", "n_ctx": len(P)}
        T = A[h:]
        S, mode = P + T, "spliced"
        T0 = F0[len(N):][h:]
    else:
        return {"ok": False, "why": "prefix-mismatch", "n_ctx": len(P)}
    if len(S) > MAX_CELL_TOKENS:  # PLAN §9 rule 4
        return {"ok": False, "why": "too-long", "n_ctx": len(P)}
    r = _post(f"{root}/v1/completions", {"model": model, "prompt": S, "max_tokens": 1, "temperature": 0,
                                         "prompt_logprobs": 0}, TIMEOUT_SCORE)
    pl = r["choices"][0].get("prompt_logprobs")
    if not pl or len(pl) != len(S):
        return {"ok": False, "why": "no-prompt-logprobs", "n_ctx": len(P)}
    lps = []
    for i in range(len(P), len(S)):
        e = pl[i] or {}
        v = e.get(str(S[i])) or e.get(S[i])
        if v is None:
            return {"ok": False, "why": "token-missing", "n_ctx": len(P)}
        lps.append(float(v["logprob"] if isinstance(v, dict) else v))
    out = {"ok": True, "logp": round(sum(lps), 4), "n_tok": len(lps), "mean_logp": round(sum(lps) / len(lps), 5),
           "n_ctx": len(P), "mode": mode}
    b = _lcp(T, T0)
    if 0 < b < len(T):
        out.update({"n_tok_action": b, "logp_action": round(sum(lps[:b]), 4), "logp_args": round(sum(lps[b:]), 4)})
    return out


def _empty_args(target: dict) -> dict:
    t = copy.deepcopy(target)
    t["tool_calls"][0]["function"] = {**t["tool_calls"][0]["function"], "arguments": "{}"}
    return t


def sample(root: str, model: str, msgs: list[dict], tools: Any, n: int, seed: int) -> list[dict]:
    r = _post(f"{root}/v1/chat/completions", {"model": model, "messages": msgs, "tools": tools, "n": n, "seed": seed,
                                               **AGENT_SAMPLING}, TIMEOUT_SAMPLE)
    out = []
    for ch in r.get("choices", []):
        m = ch.get("message") or {}
        out.append({"tool_calls": m.get("tool_calls") or [], "finish": ch.get("finish_reason"),
                    "completion_tokens": (r.get("usage") or {}).get("completion_tokens")})
    return out


def write_words(root: str, model: str, recs: list[dict]) -> dict[str, str]:
    status = {}
    for rec in recs:
        rec["words"] = None
        for strict in (False, True):
            r = _post(f"{root}/v1/chat/completions", {"model": model, "temperature": 0, "max_tokens": 300,
                      "messages": [{"role": "user", "content": pb.words_prompt(rec["diff"], strict)}],
                      "chat_template_kwargs": {"enable_thinking": False}}, TIMEOUT_WORDS)
            w = ((r["choices"][0].get("message") or {}).get("content") or "").strip()
            if w and not pb.words_leak(w, rec["diff"]):
                rec["words"] = w
                break
        status[rec["instance_id"]] = "ok" if rec["words"] else "leak/empty"
    return status


# --------------------------------------------------------------------------- the per-size loop

def cells_for(rec: dict) -> tuple[list[tuple], list[tuple]]:
    d1 = [("find", c, 1) for c in pb.FIND_CONDS] + [("use", c, 1) for c in pb.USE_CONDS] + [("find", "L0", 2), ("use", "L0", 2)]
    d2 = [("find", c, 1) for c in pb.D2_FIND] + [("use", c, 2) for c in pb.D2_USE]
    return d1, d2


def self_check(root: str, mid: str, size: str, rec: dict, ctx: dict) -> dict:
    """Score one S-use and one S-find cell and draw one short 2-sample generation; raise if the API path is broken."""
    try:
        checks = {}
        for st in ("use", "find"):
            m = build_messages(rec, ctx, st, "L0")
            checks[st] = score(root, mid, m[0], ctx["tools"], m[1])
        m = build_messages(rec, ctx, "use", "L0")
        smp = _post(f"{root}/v1/chat/completions", {"model": mid, "messages": m[0], "tools": ctx["tools"],
                    "n": 2, "seed": 1, **{**AGENT_SAMPLING, "max_tokens": 16}}, TIMEOUT_SAMPLE)
        checks["sample_choices"] = len(smp.get("choices", []))
    except Exception as e:  # noqa: BLE001 - any failure here means the API path is broken for this size
        raise RuntimeError(f"SELFCHECK failed on {size}: {type(e).__name__}: {str(e)[:400]}") from e
    if not (checks["use"].get("ok") and checks["find"].get("ok") and checks["sample_choices"] == 2):
        raise RuntimeError(f"SELFCHECK failed on {size}: {checks}")
    return checks


def run_size(server, model_alias: str, recs: list[dict], ctxs: dict[str, dict], deadline: float, workers: int = 16,
             write_words_here: bool = False) -> dict[str, Any]:
    """C03: every pre-registered cell (PLAN §4.6) for one size."""
    usable = [r for r in recs if r["instance_id"] in ctxs]
    d1 = [(r, st, c, rep, {}) for r in usable for (st, c, rep) in cells_for(r)[0]]
    d2 = [(r, st, c, n) for r in usable for (st, c, n) in cells_for(r)[1]]
    return run_cells(server, model_alias, recs, ctxs, deadline, d1, d2, workers, write_words_here=write_words_here)


def run_cells(server, model_alias: str, recs: list[dict], ctxs: dict[str, dict], deadline: float,
              d1_cells: list[tuple], d2_cells: list[tuple], workers: int = 16, write_words_here: bool = False,
              samples_file: str | None = None) -> dict[str, Any]:
    """Self-check, then depth-1 cells (rec, state, cond, rep, extra-fields) and depth-2 cells (rec, state, cond, n)."""
    root = server_root(server)
    mid = _get(f"{root}/v1/models")["data"][0]["id"]
    size = kr.SHORT[model_alias]
    out: dict[str, Any] = {"size": size, "d1": [], "d2": [], "words_status": None, "skipped": [], "selfcheck": None}
    usable = [r for r in recs if r["instance_id"] in ctxs]
    if not usable:
        raise RuntimeError("SELFCHECK failed: no captured agent contexts (the capture phase produced nothing)")
    out["selfcheck"] = self_check(root, mid, size, usable[0], ctxs[usable[0]["instance_id"]])
    if write_words_here:
        out["words_status"] = write_words(root, mid, usable)
        pb.assign_cross_task_controls(usable)
    t0 = time.time()

    def d1_job(rec, st, cond, rep, extra):
        head = {"task": rec["instance_id"], "size": size, "state": st, "cond": cond, "rep": rep, **extra}
        if TRIPPED.is_set() or time.time() > deadline:
            return {**head, "ok": False, "why": "server_frozen" if TRIPPED.is_set() else "time_guard"}
        mm = build_messages(rec, ctxs[rec["instance_id"]], st, cond)
        if mm is None:
            return {**head, "ok": False, "why": "unavailable"}
        res = score(root, mid, mm[0], ctxs[rec["instance_id"]]["tools"], mm[1])
        return {**head, "hint_method": mm[2], **res}

    with cf.ThreadPoolExecutor(workers) as ex:
        futs = [ex.submit(d1_job, *j) for j in d1_cells if j[0]["instance_id"] in ctxs]
        for f in cf.as_completed(futs):
            try:
                out["d1"].append(f.result())
            except Exception as e:  # noqa: BLE001
                why = "server_frozen" if isinstance(e, ServerFrozen) or TRIPPED.is_set() else f"{type(e).__name__}: {str(e)[:120]}"
                out["d1"].append({"size": size, "ok": False, "why": why})
    out["d1_minutes"] = round((time.time() - t0) / 60, 2)
    samples_private: list[dict] = []
    if not d2_cells:
        return out
    if TRIPPED.is_set():
        out["skipped"].append("depth2 (server frozen)")
        return out
    if time.time() > deadline:
        out["skipped"].append("depth2 (time guard)")
        return out
    t1 = time.time()

    def d2_job(rec, st, cond, n):
        if TRIPPED.is_set() or time.time() > deadline:
            return [], [{"task": rec["instance_id"], "size": size, "state": st, "cond": cond, "ok": False,
                         "why": "server_frozen" if TRIPPED.is_set() else "time_guard"}]
        mm = build_messages(rec, ctxs[rec["instance_id"]], st, cond)
        if mm is None:
            return [], [{"task": rec["instance_id"], "size": size, "state": st, "cond": cond, "ok": False, "why": "unavailable"}]
        crec = rec
        if cond == "G-null" and rec.get("null_graph"):  # followed_wrong = went to the unrelated graph's file or symbol
            crec = {**rec, "wrong": {"file": rec["null_graph"]["file"], "qualname": rec["null_graph"]["qualname"]}}
        rows, priv = [], []
        for k, smp in enumerate(sample(root, mid, mm[0], ctxs[rec["instance_id"]]["tools"], n, seed=1)):
            if st == "find":
                flags, edited = pb.classify_find(crec, smp["tool_calls"]), None
            else:
                flags, edited = pb.classify_use(crec, smp["tool_calls"])
            rows.append({"task": rec["instance_id"], "size": size, "state": st, "cond": cond, "sample": k, "ok": True,
                         "finish": smp["finish"], **flags})
            if edited is not None:
                diff = "".join(difflib.unified_diff(rec["_base_text"].splitlines(True), edited.splitlines(True),
                                                    f"a/{rec['gold']['file']}", f"b/{rec['gold']['file']}"))
                priv.append({"task": rec["instance_id"], "size": size, "cond": cond, "sample": k, "patch": diff})
        return priv, rows

    with cf.ThreadPoolExecutor(workers) as ex:
        futs = [ex.submit(d2_job, *j) for j in d2_cells if j[0]["instance_id"] in ctxs]
        for f in cf.as_completed(futs):
            try:
                priv, rows = f.result()
                out["d2"] += rows
                samples_private += priv
            except Exception as e:  # noqa: BLE001
                why = "server_frozen" if isinstance(e, ServerFrozen) or TRIPPED.is_set() else f"{type(e).__name__}: {str(e)[:120]}"
                out["d2"].append({"size": size, "ok": False, "why": why})
    out["d2_minutes"] = round((time.time() - t1) / 60, 2)
    kr.PRIVATE.mkdir(parents=True, exist_ok=True)
    with open(kr.PRIVATE / (samples_file or f"probe_samples_{size}.jsonl"), "w") as f:
        for s_ in samples_private:
            f.write(json.dumps(s_) + "\n")
    return out


# --------------------------------------------------------------------------- follow-up (notebooks/C04_probe_followup__L4x4.ipynb)

def load_c03_inputs() -> tuple[list[dict], dict[str, dict], dict]:
    """Records (with the words written by 31b and the cross-task controls), captured contexts and the results of
    notebooks/C03_probe__L4x4.ipynb, from its attached output. Re-derives the private base text of each gold file."""
    recs_path = kr.find_input_json("probe_records.jsonl")
    recs = [json.loads(l) for l in open(recs_path) if l.strip()]
    ctxs = json.loads((recs_path.parent / "probe_contexts.json").read_text())
    c03 = json.loads(kr.find_input_json("C03_probe.json").read_text())
    d = kr.data_dir()
    for r in recs:
        texts, _ = pb.snapshot_scan(d / "snapshots" / f"{r['instance_id']}.tgz", {r["gold"]["file"]})
        r["_base_text"] = texts[r["gold"]["file"]]
    keep = set(c03["task_ids"])
    recs = [r for r in recs if r["instance_id"] in keep]
    return recs, ctxs, c03


def followup_cells(size: str, recs: list[dict], c03: dict, rescore_sizes: tuple[str, ...], graph_controls: tuple = pb.FOLLOWUP_D1,
                   d2_extra: tuple = pb.FOLLOWUP_D2, rescore_use: tuple = pb.RESCORE_USE,
                   rescore_find: tuple = ()) -> tuple[list[tuple], list[tuple], dict]:
    """Cells for one size in the follow-up:
      design  : every pre-registered C03 cell (PLAN §4.6) that C03 did not complete for this size (time guard, skipped
                size). 'unavailable' cells stay unavailable.
      control : G-null at S-find and S-use (depth 1), X-graph and G-null at S-find (depth 2, n=1)  [exploratory]
      rescore : S-use L0/L2/L5/W (and `rescore_find` at S-find) again, to split log p into action and arguments
                (rescore_sizes only) [exploratory]"""
    done1 = {(r["task"], r["state"], r["cond"], r.get("rep", 1)) for r in c03["d1"]
             if r.get("size") == size and (r.get("ok") or r.get("why") == "unavailable")}
    done2 = {(r["task"], r["state"], r["cond"]) for r in c03["d2"]
             if r.get("size") == size and (r.get("ok") or r.get("why") == "unavailable")}
    d1, d2, counts = [], [], {"design_d1": 0, "control_d1": 0, "rescore_d1": 0}
    for r in recs:
        c1, c2 = cells_for(r)
        design1 = [(st, c, rep) for (st, c, rep) in c1 if (r["instance_id"], st, c, rep) not in done1]
        d1 += [(r, st, c, rep, {"purpose": "design"}) for (st, c, rep) in design1]
        d1 += [(r, st, c, 1, {"purpose": "control"}) for (st, c) in graph_controls]
        if size in rescore_sizes:
            d1 += [(r, "use", c, 1, {"purpose": "rescore"}) for c in rescore_use if ("use", c, 1) not in design1]
            d1 += [(r, "find", c, 1, {"purpose": "rescore"}) for c in rescore_find if ("find", c, 1) not in design1]
        d2 += [(r, st, c, n) for (st, c, n) in c2 if (r["instance_id"], st, c) not in done2]
        d2 += [(r, st, c, 1) for (st, c) in d2_extra]
    for x in d1:
        counts[f"{x[4]['purpose']}_d1"] += 1
    counts["d2_cells"] = len(d2)   # design + extra depth-2 cells (each draws n samples)
    return d1, d2, counts


def summarize(d1: list[dict], d2: list[dict]) -> dict[str, Any]:
    """Quick in-notebook summary (medians of paired deltas); full statistics run offline from the rows."""
    import statistics
    out: dict[str, Any] = {}
    by = {}
    for r in d1:
        if r.get("ok"):
            by[(r["size"], r["state"], r["cond"], r["rep"], r["task"])] = r["logp"]
    sizes = sorted({k[0] for k in by})
    for s in sizes:
        for st in ("find", "use"):
            conds = sorted({k[2] for k in by if k[0] == s and k[1] == st})
            for c in conds:
                ref = {"X-file": "L1", "X-func": "L2", "X-graph": "G-json", "X-test": "L3", "X-words": "L4"}.get(c, "L0")
                ds = [by[(s, st, c, 1, t)] - by[(s, st, ref, 1, t)] for (s2, st2, c2, rep, t) in by
                      if s2 == s and st2 == st and c2 == c and rep == 1 and (s, st, ref, 1, t) in by]
                if ds and c != ref:
                    out[f"{s}/{st}/{c}-vs-{ref}"] = {"n": len(ds), "median_delta_nats": round(statistics.median(ds), 3)}
            noise = [abs(by[(s, st, "L0", 2, t)] - by[(s, st, "L0", 1, t)]) for (s2, st2, c2, rep, t) in by
                     if s2 == s and st2 == st and c2 == "L0" and rep == 2 and (s, st, "L0", 1, t) in by]
            if noise:
                out[f"{s}/{st}/L0-rerun-abs-diff"] = {"n": len(noise), "max": round(max(noise), 4)}
    for s in sorted({r["size"] for r in d2 if r.get("ok")}):
        for st, keys in (("find", ("hit_file", "hit_func", "followed_wrong")), ("use", ("on_target", "exact", "applies", "followed_wrong"))):
            for c in sorted({r["cond"] for r in d2 if r.get("ok") and r["size"] == s and r["state"] == st}):
                rows = [r for r in d2 if r.get("ok") and r["size"] == s and r["state"] == st and r["cond"] == c]
                out[f"{s}/{st}/{c}/depth2"] = {"n": len(rows), "tool_call": round(sum(r["tool_call"] for r in rows) / len(rows), 3),
                                              **{k: round(sum(bool(r.get(k)) for r in rows) / len(rows), 3) for k in keys}}
    return out


# --------------------------------------------------------------------------- CPU verification (notebooks/C03v_test_verify__CPU.ipynb)

def verify_samples(records_path: Path, sample_files: list[Path], out_path: Path, timeout_s: int = 300) -> dict[str, Any]:
    """Apply each sampled S-use edit alone to the base snapshot and run the scorer's Phase-2 verification.
    Writes numbers only: (task, size, cond, sample, solves)."""
    from adk_submission.registry import ModelRegistry
    from swegemma.config import EvalConfig
    from swegemma.deduplication import resolve_task_snapshot_paths
    from swegemma.evaluate import Evaluator
    from swegemma.harness.verification import verify_task
    from swegemma.models import load_tasks as sg_load

    d = kr.data_dir()
    cfg = EvalConfig(tasks_path=d / "tasks.jsonl", snapshots_dir=d / "snapshots", results_dir=kr.PRIVATE / "probe_verify",
                     submission_dir=d / "sample_submission", models=ModelRegistry(), sandbox="subprocess",
                     timeout_seconds=timeout_s, verification_timeout_seconds=timeout_s, wheels_dir=d / "wheels",
                     graph_dir=str(d / "graphs"), embeddings_dir=str(d / "embeddings"), display_mode="quiet")
    ev = Evaluator(cfg)
    tasks = {t.instance_id: t for t in sg_load(d / "tasks.jsonl")}
    rows: list[dict] = []
    cache: dict[tuple[str, str], bool] = {}
    for sf in sample_files:
        for line in open(sf):
            s = json.loads(line)
            t = tasks[s["task"]]
            key = (s["task"], s["patch"])
            if key not in cache:
                snap, base, patch = resolve_task_snapshot_paths(d / "snapshots", t.instance_id, t.repo)
                try:
                    r = kr.run_sync(verify_task, ev.docker, cfg, t, snap, base_snapshot_path=base, patch_path=patch,
                                    agent_patch=s["patch"], agent_error=None, trace=None, start_time=time.perf_counter())
                    cache[key] = bool(r.resolved) and bool(s["patch"].strip())
                except Exception:  # noqa: BLE001
                    cache[key] = False
            rows.append({"task": s["task"], "size": s["size"], "cond": s["cond"], "sample": s["sample"], "solves": cache[key],
                         "source": "C04" if "_C04_" in sf.name else "C03"})
            out_path.write_text(json.dumps({"rows": rows}, indent=1))
    summ: dict[str, Any] = {}
    for r in rows:
        k = f"{r['size']}/{r['cond']}"
        a = summ.setdefault(k, {"n": 0, "solves": 0})
        a["n"] += 1
        a["solves"] += int(r["solves"])
    res = {"rows": rows, "summary": summ, "unique_patches_verified": len(cache)}
    out_path.write_text(json.dumps(res, indent=1))
    return res

In [ ]:
%%writefile /kaggle/working/pilot/pilot_c/analyze_c06.py
"""Analysis of the C05 validation rerun with max_output_tokens = 8192, as pre-registered in
docs/PLAN_C06_validation_fix.md §6. Standalone: numpy + math only (matplotlib optional, for the figures), so it runs
in the notebook's last cell on Kaggle and offline.

Inputs : C06 output (notebooks/C06_validation_fix_12b__L4x4.ipynb), C05 output (results/C05_validation_12b.json),
         12b probe predictors (results/C06_predictors_12b.json, built from C03 by build_predictors),
         optionally the private summaries (private/runs/12b/*/summaries.jsonl, Kaggle only: numbers are extracted).
Outputs: C06_analysis.json, C06_tables.md, C06_fig1_how_runs_ended.png, C06_fig2_hint_effect.png,
         C06_fig3_summary_retention.png (numbers only; no task text).
Usage  : python -m pilot_c.analyze_c06 <c06.json> <c05.json> <predictors.json> <out_dir> [summaries_glob]
"""
from __future__ import annotations

import glob
import json
import math
import sys
from pathlib import Path

import numpy as np

CONDS = ("O0", "O-all")
ENDS = ("resolved", "context_crash", "time_budget", "tool_call_budget", "unresolved_other")
MIN_POSITIVES = 4
PREDICTORS = {"H6_d_use_L3-L0": ("d_use_L3_L0", "pre-registered (H6)"),
              "x_d_find_L2-L0": ("d_find_L2_L0", "exploratory")}


# --------------------------------------------------------------------------- small exact statistics (no scipy)

def mcnemar_exact(b: int, c: int) -> float | None:
    m = b + c
    if m == 0:
        return None
    k = min(b, c)
    return round(min(1.0, 2 * sum(math.comb(m, i) for i in range(k + 1)) / 2 ** m), 6)


def auc(scores: np.ndarray, labels: np.ndarray) -> float | None:
    pos, neg = scores[labels == 1], scores[labels == 0]
    if len(pos) == 0 or len(neg) == 0:
        return None
    gt = (pos[:, None] > neg[None, :]).sum() + 0.5 * (pos[:, None] == neg[None, :]).sum()
    return float(gt / (len(pos) * len(neg)))


def auc_block(scores: dict[str, float], labels: dict[str, int], seed: int = 0) -> dict:
    ts = sorted(set(scores) & set(labels))
    s, y = np.array([scores[t] for t in ts], float), np.array([labels[t] for t in ts], int)
    pos = int(y.sum())
    res = {"n": len(ts), "positives": pos, "negatives": len(ts) - pos, "auc": None, "ci": None,
           "dropped_resamples": None, "informative": pos >= MIN_POSITIVES and len(ts) - pos >= 1}
    a = auc(s, y)
    if a is None:
        return res
    res["auc"] = round(a, 4)
    rng = np.random.default_rng(seed)
    vals, dropped = [], 0
    for _ in range(10_000):
        i = rng.integers(0, len(ts), len(ts))
        v = auc(s[i], y[i])
        if v is None:
            dropped += 1
        else:
            vals.append(v)
    res["ci"] = [round(float(np.quantile(vals, 0.025)), 4), round(float(np.quantile(vals, 0.975)), 4)] if vals else None
    res["dropped_resamples"] = dropped
    return res


def _q(xs: list[float]) -> dict | None:
    v = [x for x in xs if x is not None]
    if not v:
        return None
    return {"n": len(v), "median": round(float(np.median(v)), 4), "q25": round(float(np.quantile(v, 0.25)), 4),
            "q75": round(float(np.quantile(v, 0.75)), 4), "max": round(float(max(v)), 4)}


# --------------------------------------------------------------------------- data access

def ended(x: dict) -> str:
    e = x.get("error") or ""
    if "ContextWindowExceeded" in e:
        return "context_crash"
    if x.get("resolved"):
        return "resolved"
    if "session timeout" in e:
        return "time_budget"
    if "tool call budget" in e:
        return "tool_call_budget"
    return "unresolved_other"


def rows_by(run: dict) -> dict[tuple[str, str], dict]:
    return {(r["condition"], x["instance_id"]): x for r in run["runs"] for x in r["rows"]}


def build_predictors(c03_path: str, c04_path: str | None = None, size: str = "12b") -> dict[str, dict[str, float]]:
    """Per task: Δ_use(L3−L0) (H6) and Δ_find(L2−L0) for `size`, from C03's depth-1 design cells; cells C03 did not
    complete are filled from C04 when given (PLAN_C05 §7). Numbers only."""
    lp: dict[tuple[str, str, str], float] = {}
    for p in [c03_path, c04_path]:
        if p and Path(p).exists():
            for r in json.loads(Path(p).read_text())["d1"]:
                if r.get("ok") and r.get("size") == size and r.get("rep", 1) == 1 and r.get("purpose", "design") == "design":
                    lp.setdefault((r["state"], r["cond"], r["task"]), r["logp"])
    tasks = sorted({k[2] for k in lp})
    out = {}
    for t in tasks:
        v = {}
        if ("use", "L3", t) in lp and ("use", "L0", t) in lp:
            v["d_use_L3_L0"] = round(lp[("use", "L3", t)] - lp[("use", "L0", t)], 4)
        if ("find", "L2", t) in lp and ("find", "L0", t) in lp:
            v["d_find_L2_L0"] = round(lp[("find", "L2", t)] - lp[("find", "L0", t)], 4)
        if v:
            out[t] = v
    return out


# --------------------------------------------------------------------------- analyses (PLAN_C06 §6)

def hint_effect(rows: dict, tasks: list[str], exclude_crash: bool) -> dict:
    keep = [t for t in tasks if all((c, t) in rows for c in CONDS)
            and not (exclude_crash and any(ended(rows[(c, t)]) == "context_crash" for c in CONDS))]
    res = {c: sum(bool(rows[(c, t)].get("resolved")) for t in keep) for c in CONDS}
    f2p = sum(1 for t in keep if not rows[("O0", t)].get("resolved") and rows[("O-all", t)].get("resolved"))
    p2f = sum(1 for t in keep if rows[("O0", t)].get("resolved") and not rows[("O-all", t)].get("resolved"))
    return {"tasks": len(keep), "resolved": res, "fail_to_pass": f2p, "pass_to_fail": p2f,
            "mcnemar_exact_p": mcnemar_exact(f2p, p2f)}


def summary_numbers(summaries_glob: str | None, rows06: dict) -> list[dict]:
    """Per-summary retention numbers. From the private summaries files when available (Kaggle), else per-run means."""
    out = []
    if summaries_glob:
        for f in sorted(glob.glob(summaries_glob)):
            cond = "O-all" if "/O-all_" in f else "O0"
            for line in open(f, encoding="utf-8"):
                s = json.loads(line)
                out.append({"condition": cond, "task": s.get("task"), "ngram5_recall": s.get("ngram5_recall"),
                            "word_recall": s.get("word_recall"), "has_marker": bool(s.get("has_marker")), "source": "summaries"})
    if not out:
        for (c, t), x in rows06.items():
            if x.get("cmp_summaries"):
                out.append({"condition": c, "task": t, "ngram5_recall": x.get("cmp_summary_ngram5_recall_mean"),
                            "word_recall": x.get("cmp_summary_word_recall_mean"),
                            "has_marker": bool(x.get("cmp_summaries_with_marker")), "source": "per-run mean"})
    return out


def compaction_block(rows: dict, c: str) -> dict:
    xs = [x for (cc, _), x in rows.items() if cc == c]
    comp = [x for x in xs if (x.get("cmp_compactions") or 0) > 0]
    g = lambda k: sum(x.get(k) or 0 for x in comp)  # noqa: E731
    return {"runs": len(xs), "runs_with_compaction": len(comp),
            "task_prompt_event_compacted": sum(bool(x.get("cmp_task_prompt_compacted")) for x in comp),
            "requests_after_first_compaction": g("cmp_requests_after_first_compaction"),
            "with_marker_any_source": g("cmp_after_with_task_prompt"),
            "marker_in_original_prompt": g("cmp_after_marker_in_prompt"),
            "marker_in_summary_only": g("cmp_after_marker_in_summary_only"),
            "marker_elsewhere_only": g("cmp_after_marker_other_only"),
            "with_original_prompt_present": g("cmp_after_with_original_prompt"),
            "summaries": sum(x.get("cmp_summaries") or 0 for x in xs),
            "summaries_with_marker": sum(x.get("cmp_summaries_with_marker") or 0 for x in xs)}


def context_block(rows: dict, c: str) -> dict:
    xs = [x for (cc, _), x in rows.items() if cc == c]
    req = sorted({v for x in xs for v in (x.get("cmp_requested_output_tokens") or [])})
    return {"requested_output_tokens_seen": req,
            "max_prompt_tokens": _q([x.get("cmp_max_prompt_tokens") for x in xs]),
            "context_errors": sum(x.get("cmp_context_errors") or 0 for x in xs),
            "prompt_tokens_before_first_context_error": sorted(x["cmp_prompt_tokens_before_first_context_error"] for x in xs
                                                               if x.get("cmp_prompt_tokens_before_first_context_error"))}


def common_metrics(rows: dict, c: str) -> dict:
    xs = [x for (cc, _), x in rows.items() if cc == c]
    m = lambda k: round(float(np.mean([x.get(k) or 0 for x in xs])), 3) if xs else None  # noqa: E731
    return {"runs": len(xs), "resolved": sum(bool(x.get("resolved")) for x in xs), "wall_s_mean": m("wall_s"),
            "tool_calls_mean": m("tool_calls"), "repeat_share_mean": m("repeat_share"),
            "malformed_share_mean": m("malformed_share"), "failed_edits_total": sum(x.get("failed_edits") or 0 for x in xs),
            "skill_calls_total": sum(x.get("skill_calls") or 0 for x in xs)}


def analyze(c06: dict, c05: dict, predictors: dict, summaries_glob: str | None = None) -> dict:
    r6, r5 = rows_by(c06), rows_by(c05)
    tasks = [t for t in c06["task_ids"] if all((c, t) in r6 for c in CONDS)]        # tasks C06 ran in both conditions
    A: dict = {"plan": "docs/PLAN_C06_validation_fix.md", "tasks_compared": len(tasks), "stopped": c06.get("stopped"),
               "stages": [{k: s.get(k) for k in ("stage", "discordant", "b_only", "a_only", "skipped")} for s in c06.get("stages", [])],
               "session_elapsed_h": c06.get("session_elapsed_h")}
    # setting check: every request must have asked for 8192 output tokens
    seen = sorted({v for x in r6.values() for v in (x.get("cmp_requested_output_tokens") or [])})
    A["setting_check"] = {"requested_output_tokens_seen": seen, "ok": seen == [8192]}
    # A1 crashes per condition, C06 vs C05 (same tasks)
    A["A1_crashes"] = {}
    for c in CONDS:
        t_ok = [t for t in tasks if (c, t) in r5]
        k5 = {t: int(ended(r5[(c, t)]) == "context_crash") for t in t_ok}
        k6 = {t: int(ended(r6[(c, t)]) == "context_crash") for t in t_ok}
        b = sum(1 for t in t_ok if k5[t] and not k6[t])
        cc = sum(1 for t in t_ok if not k5[t] and k6[t])
        A["A1_crashes"][c] = {"tasks": len(t_ok), "C05": sum(k5.values()), "C06": sum(k6.values()),
                              "crash_C05_only": b, "crash_C06_only": cc, "mcnemar_exact_p": mcnemar_exact(b, cc),
                              "ended_C05": {e: sum(1 for t in t_ok if ended(r5[(c, t)]) == e) for e in ENDS},
                              "ended_C06": {e: sum(1 for t in t_ok if ended(r6[(c, t)]) == e) for e in ENDS}}
    # A2 / A3 hint effect: all tasks (crash = failure) and with crashes removed; C05 on the same tasks for comparison
    A["A2_hint_effect_all"] = {"C06": hint_effect(r6, tasks, False), "C05": hint_effect(r5, tasks, False)}
    A["A3_hint_effect_crashes_removed"] = {"C06": hint_effect(r6, tasks, True), "C05": hint_effect(r5, tasks, True)}
    # A4 H6 replication on C06 outcomes
    A["A4_predictors"] = {}
    for name, (key, label) in PREDICTORS.items():
        sc = {t: predictors[t][key] for t in tasks if t in predictors and key in predictors[t]}
        primary = {t: int(bool(r6[("O-all", t)].get("resolved"))) for t in tasks if not r6[("O0", t)].get("resolved")}
        secondary = {t: int(not r6[("O0", t)].get("resolved") and bool(r6[("O-all", t)].get("resolved"))) for t in tasks}
        p = auc_block(sc, primary)
        p["criterion_auc_ge_0.65"] = None if not (p["informative"] and p["auc"] is not None) else p["auc"] >= 0.65
        A["A4_predictors"][name] = {"label": label, "n_with_value": len(sc), "primary_unresolved_at_O0": p,
                                    "secondary_all_tasks": auc_block(sc, secondary)}
    # A5 compaction and summaries; A6 context numbers; A7 common metrics
    A["A5_compaction"] = {c: compaction_block(r6, c) for c in CONDS}
    sn = summary_numbers(summaries_glob, r6)
    A["A5_summary_retention"] = {c: {"ngram5_recall": _q([s["ngram5_recall"] for s in sn if s["condition"] == c]),
                                     "word_recall": _q([s["word_recall"] for s in sn if s["condition"] == c]),
                                     "with_marker": sum(s["has_marker"] for s in sn if s["condition"] == c),
                                     "n": sum(1 for s in sn if s["condition"] == c),
                                     "source": sn[0]["source"] if sn else None} for c in CONDS}
    A["A5_summary_points"] = [{k: s[k] for k in ("condition", "ngram5_recall", "word_recall", "has_marker")} for s in sn]
    A["A6_context"] = {c: context_block(r6, c) for c in CONDS}
    A["A7_common_metrics"] = {"C06": {c: common_metrics(r6, c) for c in CONDS},
                              "C05_same_tasks": {c: common_metrics({k: v for k, v in r5.items() if k[1] in tasks}, c) for c in CONDS}}
    return A


# --------------------------------------------------------------------------- tables and figures

def _p(p):
    return "—" if p is None else ("<0.001" if p < 0.001 else f"{p:.3f}")


def tables_md(A: dict) -> str:
    L = ["# C06 validation rerun (max_output_tokens 8192) — tables (data only)", "",
         f"Plan: `{A['plan']}`. Tasks run in both conditions: {A['tasks_compared']}. Stopped: {A['stopped']}. "
         f"Session: {A['session_elapsed_h']} h. Setting check (requested output tokens seen): "
         f"{A['setting_check']['requested_output_tokens_seen']} → {'OK' if A['setting_check']['ok'] else 'NOT APPLIED'}.", "",
         "## A1. How runs ended, C05 vs C06 (same tasks)", "",
         "| Condition | Run | " + " | ".join(ENDS) + " | crash McNemar (C05→C06) |", "|---|---|" + "---|" * (len(ENDS) + 1)]
    for c, v in A["A1_crashes"].items():
        for run in ("C05", "C06"):
            L.append(f"| {c} | {run} | " + " | ".join(str(v[f"ended_{run}"][e]) for e in ENDS) + " | "
                     + (f"{v['crash_C05_only']} vs {v['crash_C06_only']}, p={_p(v['mcnemar_exact_p'])}" if run == "C06" else "") + " |")
    L += ["", "## A2–A3. Hint effect (O-all vs O0)", "", "| Run | Tasks | crashes | O0 resolved | O-all resolved | fail→pass | pass→fail | McNemar p |",
          "|---|---|---|---|---|---|---|---|"]
    for lab, key in (("all tasks (crash = failure)", "A2_hint_effect_all"), ("crashes removed", "A3_hint_effect_crashes_removed")):
        for run in ("C06", "C05"):
            h = A[key][run]
            L.append(f"| {run} | {h['tasks']} | {lab} | {h['resolved']['O0']} | {h['resolved']['O-all']} | {h['fail_to_pass']} | "
                     f"{h['pass_to_fail']} | {_p(h['mcnemar_exact_p'])} |")
    L += ["", "## A4. Probe predictors of fail→pass (C06 outcomes)", "", "| Predictor | Set | n | positives | AUC [95% CI] | criterion ≥ 0.65 |",
          "|---|---|---|---|---|---|"]
    for name, v in A["A4_predictors"].items():
        for setname in ("primary_unresolved_at_O0", "secondary_all_tasks"):
            b = v[setname]
            ci = f"{b['auc']} [{b['ci'][0]}, {b['ci'][1]}]" if b.get("auc") is not None and b.get("ci") else str(b.get("auc"))
            crit = v[setname].get("criterion_auc_ge_0.65", "—") if setname.startswith("primary") else "—"
            L.append(f"| {name} ({v['label']}) | {setname} | {b['n']} | {b['positives']} | {ci}{'' if b['informative'] else ' (not informative)'} | {crit} |")
    L += ["", "## A5. Compaction: where the task-prompt marker sits after compaction; summary retention", "",
          "| Condition | runs compacted | prompt event compacted | requests after | marker: original prompt | summary only | elsewhere only | original prompt present | summaries (with marker) | 5-gram recall median [IQR] (max) | word recall median |",
          "|---|---|---|---|---|---|---|---|---|---|---|"]
    for c in CONDS:
        b, r = A["A5_compaction"][c], A["A5_summary_retention"][c]
        g5, wr = r["ngram5_recall"], r["word_recall"]
        L.append(f"| {c} | {b['runs_with_compaction']}/{b['runs']} | {b['task_prompt_event_compacted']} | {b['requests_after_first_compaction']} | "
                 f"{b['marker_in_original_prompt']} | {b['marker_in_summary_only']} | {b['marker_elsewhere_only']} | {b['with_original_prompt_present']} | "
                 f"{b['summaries']} ({b['summaries_with_marker']}) | "
                 + (f"{g5['median']} [{g5['q25']}, {g5['q75']}] ({g5['max']})" if g5 else "—") + " | " + (f"{wr['median']}" if wr else "—") + " |")
    L += ["", "## A6. Context numbers", "", "| Condition | requested output tokens | max prompt tokens per run, median [IQR] (max) | context errors | prompt tokens before first error |",
          "|---|---|---|---|---|"]
    for c in CONDS:
        b = A["A6_context"][c]
        q = b["max_prompt_tokens"]
        L.append(f"| {c} | {b['requested_output_tokens_seen']} | " + (f"{q['median']} [{q['q25']}, {q['q75']}] ({q['max']})" if q else "—")
                 + f" | {b['context_errors']} | {b['prompt_tokens_before_first_context_error']} |")
    L += ["", "## A7. Common metrics", "", "| Run | Condition | runs | resolved | wall s mean | tool calls mean | repeat share | malformed share | failed edits | skill calls |",
          "|---|---|---|---|---|---|---|---|---|---|"]
    for run in ("C06", "C05_same_tasks"):
        for c in CONDS:
            m = A["A7_common_metrics"][run][c]
            L.append(f"| {run} | {c} | {m['runs']} | {m['resolved']} | {m['wall_s_mean']} | {m['tool_calls_mean']} | {m['repeat_share_mean']} | "
                     f"{m['malformed_share_mean']} | {m['failed_edits_total']} | {m['skill_calls_total']} |")
    return "\n".join(L) + "\n"


SURF, INK, INK2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e6e5e0"
END_COLORS = {"resolved": "#2a78d6", "context_crash": "#eb6834", "time_budget": "#1baf7a", "tool_call_budget": "#eda100",
              "unresolved_other": "#e87ba4"}
COND_COLORS = {"O0": "#2a78d6", "O-all": "#eb6834"}


def figures(A: dict, out_dir: Path) -> list[str]:
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
    except Exception as e:  # noqa: BLE001
        print("figures skipped (matplotlib unavailable):", type(e).__name__)
        return []
    made = []

    def style(ax):
        ax.set_facecolor(SURF)
        for sp in ("top", "right", "left"):
            ax.spines[sp].set_visible(False)
        ax.spines["bottom"].set_color(INK2)
        ax.tick_params(colors=INK2, labelsize=8, length=0)
        ax.grid(axis="x", color=GRID, linewidth=0.6)
        ax.set_axisbelow(True)

    # fig 1: how runs ended
    labels, data = [], []
    for c in CONDS:
        for run in ("C05", "C06"):
            labels.append(f"{run} {c}")
            data.append(A["A1_crashes"][c][f"ended_{run}"])
    fig, ax = plt.subplots(figsize=(8, 3.2), facecolor=SURF)
    for i, d in enumerate(data):
        left = 0
        for e in ENDS:
            if d[e]:
                ax.barh(i, d[e], left=left, color=END_COLORS[e], edgecolor=SURF, linewidth=2, height=0.6)
                ax.text(left + d[e] / 2, i, str(d[e]), ha="center", va="center", fontsize=7, color=INK)
            left += d[e]
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels)
    ax.invert_yaxis()
    ax.set_xlabel("runs (12b, 30 tasks per condition)", color=INK2, fontsize=8)
    style(ax)
    import matplotlib.patches as mp
    fig.legend(handles=[mp.Patch(color=END_COLORS[e], label=e.replace("_", " ")) for e in ENDS], loc="upper left",
               bbox_to_anchor=(0.01, 0.92), ncol=len(ENDS), frameon=False, fontsize=7)
    fig.suptitle("How runs ended: C05 (max_output_tokens 16384) vs C06 (8192)", color=INK, fontsize=10, x=0.01, y=0.99, ha="left")
    fig.tight_layout(rect=(0, 0, 1, 0.82))
    p = out_dir / "C06_fig1_how_runs_ended.png"
    fig.savefig(p, dpi=200, facecolor=SURF)
    plt.close(fig)
    made.append(str(p))

    # fig 2: hint effect, all tasks vs crashes removed
    rows = []
    for run in ("C05", "C06"):
        for lab, key in (("all tasks", "A2_hint_effect_all"), ("crashes removed", "A3_hint_effect_crashes_removed")):
            h = A[key][run]
            if h["tasks"]:
                rows.append((f"{run}, {lab} (n={h['tasks']})", {c: h["resolved"][c] / h["tasks"] for c in CONDS}))
    fig, ax = plt.subplots(figsize=(8, 2.8), facecolor=SURF)
    for i, (lab, v) in enumerate(rows):
        ax.plot([v["O0"], v["O-all"]], [i, i], color=GRID, linewidth=2, zorder=1)
        for c, dy in (("O0", -0.12), ("O-all", 0.12)):   # dodged so equal rates stay visible
            ax.plot([v[c]], [i + dy], marker="o" if c == "O0" else "D", color=COND_COLORS[c], markersize=6,
                    markeredgecolor=SURF, linestyle="none", zorder=2, label=c if i == 0 else None)
    ax.set_yticks(range(len(rows)))
    ax.set_yticklabels([r[0] for r in rows])
    ax.invert_yaxis()
    ax.set_xlim(-0.02, 1.02)
    ax.set_xlabel("share of tasks resolved", color=INK2, fontsize=8)
    style(ax)
    fig.legend(loc="upper left", bbox_to_anchor=(0.01, 0.92), ncol=2, frameon=False, fontsize=8)
    fig.suptitle("Hint effect (O-all vs O0), with and without crashed runs", color=INK, fontsize=10, x=0.01, y=0.99, ha="left")
    fig.tight_layout(rect=(0, 0, 1, 0.80))
    p = out_dir / "C06_fig2_hint_effect.png"
    fig.savefig(p, dpi=200, facecolor=SURF)
    plt.close(fig)
    made.append(str(p))

    # fig 3: summary retention per summary
    pts = A["A5_summary_points"]
    fig, ax = plt.subplots(figsize=(8, 2.6), facecolor=SURF)
    rng = np.random.default_rng(0)
    for i, c in enumerate(CONDS):
        for mk in (False, True):
            xs = [s["ngram5_recall"] for s in pts if s["condition"] == c and s["has_marker"] == mk and s["ngram5_recall"] is not None]
            ys = i + rng.uniform(-0.18, 0.18, len(xs))
            ax.scatter(xs, ys, s=18, color=COND_COLORS[c] if mk else SURF, edgecolors=COND_COLORS[c], linewidths=1.2,
                       label=f"{c}, summary {'has' if mk else 'lacks'} the header")
    ax.set_yticks(range(len(CONDS)))
    ax.set_yticklabels(CONDS)
    ax.set_xlim(-0.02, 1.02)
    ax.set_xlabel("share of the task statement's word 5-grams kept in the summary", color=INK2, fontsize=8)
    style(ax)
    fig.legend(loc="upper left", bbox_to_anchor=(0.01, 0.92), ncol=2, frameon=False, fontsize=7)
    fig.suptitle("Compaction summaries: how much of the task statement survives", color=INK, fontsize=10, x=0.01, y=0.99, ha="left")
    fig.tight_layout(rect=(0, 0, 1, 0.70))
    p = out_dir / "C06_fig3_summary_retention.png"
    fig.savefig(p, dpi=200, facecolor=SURF)
    plt.close(fig)
    made.append(str(p))
    return made


def main(c06_path: str, c05_path: str, predictors_path: str, out_dir: str = ".", summaries_glob: str | None = None) -> dict:
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    P = json.loads(Path(predictors_path).read_text())
    A = analyze(json.loads(Path(c06_path).read_text()), json.loads(Path(c05_path).read_text()),
                P.get("tasks", P), summaries_glob)
    A["figures"] = [Path(f).name for f in figures(A, out)]
    (out / "C06_analysis.json").write_text(json.dumps(A, indent=1))
    md = tables_md(A)
    (out / "C06_tables.md").write_text(md)
    print(md)
    return A


if __name__ == "__main__":
    main(*sys.argv[1:])

In [ ]:
%%writefile /kaggle/working/pilot/common/stats.py
"""Paired statistics shared across pilots (no pilot-specific logic).

All functions take plain sequences and return plain floats/dicts so results can go straight into JSON.
Conventions: Wilcoxon and McNemar are two-sided; Page's L is one-sided by construction (it tests a
predicted order); bootstrap CIs are percentile intervals over the units given (usually tasks).
Requires numpy and scipy.
"""
from __future__ import annotations

import math
from typing import Callable, Iterable, Sequence

import numpy as np
from scipy import stats as _st


def _arr(x: Iterable[float]) -> np.ndarray:
    return np.asarray([v for v in x if v is not None and not (isinstance(v, float) and math.isnan(v))], dtype=float)


def bootstrap_ci(x: Sequence[float], stat: Callable = np.median, n: int = 10_000, seed: int = 0,
                 alpha: float = 0.05) -> tuple[float, float]:
    """Percentile CI of `stat` over resampled units. `stat` must accept axis=1."""
    a = _arr(x)
    if len(a) == 0:
        return (math.nan, math.nan)
    rng = np.random.default_rng(seed)
    vals = stat(a[rng.integers(0, len(a), size=(n, len(a)))], axis=1)
    lo, hi = np.quantile(vals, [alpha / 2, 1 - alpha / 2])
    return (float(lo), float(hi))


def bootstrap_ratio_ci(num: Sequence[float], den: Sequence[float], n: int = 10_000, seed: int = 0,
                       alpha: float = 0.05) -> tuple[float, float]:
    """CI of sum(num)/sum(den) with units (e.g. tasks) resampled together (cluster bootstrap)."""
    a, b = np.asarray(num, float), np.asarray(den, float)
    if len(a) == 0 or b.sum() == 0:
        return (math.nan, math.nan)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(a), size=(n, len(a)))
    s_den = b[idx].sum(axis=1)
    ok = s_den > 0
    vals = a[idx].sum(axis=1)[ok] / s_den[ok]
    lo, hi = np.quantile(vals, [alpha / 2, 1 - alpha / 2])
    return (float(lo), float(hi))


def wilcoxon_p(x: Sequence[float]) -> float | None:
    """Two-sided Wilcoxon signed-rank p for paired differences (zeros dropped). None if not computable."""
    a = _arr(x)
    a = a[a != 0]
    if len(a) < 1:
        return None
    return float(_st.wilcoxon(a, zero_method="wilcox", alternative="two-sided").pvalue)


def mcnemar_exact(b: int, c: int) -> float | None:
    """Exact two-sided McNemar p from the discordant counts b (0->1) and c (1->0)."""
    if b + c == 0:
        return None
    return float(_st.binomtest(min(b, c), b + c, 0.5, alternative="two-sided").pvalue)


def holm(pvals: dict[str, float | None]) -> dict[str, float | None]:
    """Holm-adjusted p-values within one family. None entries stay None and are not counted."""
    items = sorted(((k, p) for k, p in pvals.items() if p is not None), key=lambda kv: kv[1])
    m, out, run = len(items), {k: None for k in pvals}, 0.0
    for i, (k, p) in enumerate(items):
        run = max(run, min(1.0, (m - i) * p))
        out[k] = run
    return out


def page_trend(blocks: Sequence[Sequence[float]]) -> dict:
    """Page's L test. Rows = blocks (e.g. tasks), columns = treatments in the PREDICTED increasing order."""
    a = np.asarray(blocks, float)
    if a.ndim != 2 or a.shape[0] < 2 or a.shape[1] < 3:
        return {"L": None, "p": None, "n": int(a.shape[0]) if a.ndim == 2 else 0}
    r = _st.page_trend_test(a)
    return {"L": float(r.statistic), "p": float(r.pvalue), "n": int(a.shape[0]), "k": int(a.shape[1])}


def cochran_q(binary: Sequence[Sequence[int]]) -> dict:
    """Cochran's Q for k related binary outcomes. Rows = blocks, columns = conditions."""
    x = np.asarray(binary, float)
    if x.ndim != 2 or x.shape[0] < 2 or x.shape[1] < 2:
        return {"Q": None, "p": None, "n": 0}
    k = x.shape[1]
    col, row, tot = x.sum(axis=0), x.sum(axis=1), x.sum()
    den = k * tot - (row ** 2).sum()
    if den == 0:
        return {"Q": 0.0, "p": 1.0, "n": int(x.shape[0]), "k": k}
    q = (k - 1) * (k * (col ** 2).sum() - tot ** 2) / den
    return {"Q": float(q), "p": float(_st.chi2.sf(q, k - 1)), "n": int(x.shape[0]), "k": k}


def spearman(x: Sequence[float], y: Sequence[float]) -> dict:
    a, b = np.asarray(x, float), np.asarray(y, float)
    if len(a) < 3:
        return {"rho": None, "p": None, "n": len(a)}
    r = _st.spearmanr(a, b)
    return {"rho": float(r.statistic), "p": float(r.pvalue), "n": len(a)}


def auc(scores: Sequence[float], labels: Sequence[int]) -> float | None:
    """ROC AUC (probability a positive outscores a negative; ties count 1/2)."""
    s, y = np.asarray(scores, float), np.asarray(labels, int)
    pos, neg = s[y == 1], s[y == 0]
    if len(pos) == 0 or len(neg) == 0:
        return None
    gt = (pos[:, None] > neg[None, :]).sum() + 0.5 * (pos[:, None] == neg[None, :]).sum()
    return float(gt / (len(pos) * len(neg)))

In [ ]:
%%writefile /kaggle/working/pilot/pilot_c/analyze_c03.py
"""Analysis of the probe run (`notebooks/C03_probe__L4x4.ipynb` -> results/C03_probe.json). CPU only.

Writes:
  results/C03_analysis.json   every number below
  results/C03_tables.md       tables (data only)
  results/figures/C03_fig*.png / .svg   draft figures (data only)

Pre-registered tests follow docs/PLAN.md §6-7 (H1-H5; H6 needs the 12b validation). Everything in the
sections marked "exploratory" is not pre-registered.
Usage: python -m pilot_c.analyze_c03 [results/C03_probe.json]
"""
from __future__ import annotations

import collections
import json
import math
import sys
from pathlib import Path

import numpy as np

from common import stats as cs

ROOT = Path(__file__).resolve().parents[1]
SIZE_ORDER = ["e2b", "e4b", "12b", "31b"]  # ascending size; the ordered factor of H2/H4
LADDER_D2 = ["L0", "L2", "L3", "L4", "L5"]  # S-use depth-2 ladder cells (L1 is not sampled, PLAN §4.6)
ALPHA = 0.05
NBOOT = 10_000


def _r(x, k=3):
    return None if x is None or (isinstance(x, float) and math.isnan(x)) else round(float(x), k)


class Data:
    def __init__(self, d: dict):
        self.raw = d
        self.sizes = [s for s in SIZE_ORDER if any(r["size"] == s for r in d["d1"])]
        self.lp: dict = collections.defaultdict(dict)        # (size, state, cond) -> {task: logp} (rep 1)
        self.lp_rep2: dict = collections.defaultdict(dict)   # (size, state) -> {task: logp} for L0 rep 2
        self.mode: dict = collections.defaultdict(collections.Counter)
        for r in d["d1"]:
            if not r.get("ok"):
                continue
            if r.get("rep", 1) == 2:
                self.lp_rep2[(r["size"], r["state"])][r["task"]] = r["logp"]
            else:
                self.lp[(r["size"], r["state"], r["cond"])][r["task"]] = r["logp"]
                self.mode[(r["size"], r["state"], r["cond"])][r.get("mode")] += 1
        self.d2: dict = collections.defaultdict(lambda: collections.defaultdict(list))  # (size,state,cond) -> task -> rows
        for r in d["d2"]:
            if r.get("ok"):
                self.d2[(r["size"], r["state"], r["cond"])][r["task"]].append(r)
        self.d2_sizes = [s for s in self.sizes if any(k[0] == s for k in self.d2)]

    def delta(self, size, state, a, b) -> dict[str, float]:
        A, B = self.lp.get((size, state, a), {}), self.lp.get((size, state, b), {})
        return {t: A[t] - B[t] for t in sorted(set(A) & set(B))}


def summ(delta: dict[str, float], seed=0) -> dict:
    v = list(delta.values())
    if not v:
        return {"n": 0}
    lo, hi = cs.bootstrap_ci(v, n=NBOOT, seed=seed)
    return {"n": len(v), "median": _r(np.median(v)), "ci": [_r(lo), _r(hi)], "mean": _r(np.mean(v)),
            "share_pos": _r(np.mean(np.asarray(v) > 0)), "p": _r(cs.wilcoxon_p(v), 6)}


# --------------------------------------------------------------------------- depth 1 tables

FIND_CONDS = ["L1", "L2", "T", "W", "G-json", "G-edges", "G-adj", "X-file", "X-func", "X-graph"]
USE_CONDS = ["L1", "L2", "L3", "L4", "L5", "T", "W", "G-json", "G-edges", "G-adj",
             "X-file", "X-func", "X-graph", "X-test", "X-words"]
MATCHED = {"X-file": "L1", "X-func": "L2", "X-graph": "G-json", "X-test": "L3", "X-words": "L4"}


def depth1_tables(D: Data) -> dict:
    out = {}
    for size in D.sizes:
        for state, conds in (("find", FIND_CONDS), ("use", USE_CONDS)):
            for c in conds:
                out[f"{size}/{state}/{c}-L0"] = summ(D.delta(size, state, c, "L0"))
                if c in MATCHED:
                    out[f"{size}/{state}/{c}-{MATCHED[c]}"] = summ(D.delta(size, state, c, MATCHED[c]))
            r1, r2 = D.lp.get((size, state, "L0"), {}), D.lp_rep2.get((size, state), {})
            diffs = [abs(r1[t] - r2[t]) for t in set(r1) & set(r2)]
            out[f"{size}/{state}/L0-rerun-absdiff"] = {"n": len(diffs), "median": _r(np.median(diffs)) if diffs else None,
                                                       "max": _r(max(diffs)) if diffs else None}
            out[f"{size}/{state}/L0-logp"] = {"n": len(r1), "median": _r(np.median(list(r1.values()))) if r1 else None}
            out[f"{size}/{state}/modes"] = {c: dict(D.mode[(size, state, c)]) for c in ["L0"] + conds if D.mode[(size, state, c)]}
    return out


# --------------------------------------------------------------------------- depth 2 helpers

def use_task_flags(D: Data, size, cond) -> dict[str, dict]:
    """Per task over its S-use samples: any on_target / any exact, and sample counts."""
    out = {}
    for t, rows in D.d2.get((size, "use", cond), {}).items():
        edits = [r for r in rows if r.get("tool") == "edit_file" and r.get("edit_gold_file")]
        out[t] = {"n": len(rows), "any_on_target": int(any(r["on_target"] for r in rows)),
                  "any_exact": int(any(r["exact"] for r in rows)),
                  "n_edit": len(edits), "n_on_target": sum(r["on_target"] for r in rows),
                  "n_exact": sum(r["exact"] for r in rows), "n_applies": sum(r["applies"] for r in rows),
                  "n_edit_on_target": sum(r["on_target"] for r in edits), "n_edit_exact": sum(r["exact"] for r in edits),
                  "tools": collections.Counter(("write_file(other)" if r.get("tool") == "write_file" and not r.get("edit_gold_file")
                                                else r.get("tool") or "none") for r in rows)}
    return out


def mcnemar_tasks(a: dict[str, int], b: dict[str, int]) -> dict:
    ts = sorted(set(a) & set(b))
    up = sum(1 for t in ts if a[t] == 0 and b[t] == 1)
    down = sum(1 for t in ts if a[t] == 1 and b[t] == 0)
    return {"n": len(ts), "b_0to1": up, "c_1to0": down, "p": _r(cs.mcnemar_exact(up, down), 6)}


def depth2_tables(D: Data) -> dict:
    out: dict = {"find": {}, "use": {}, "cochran": {}, "threshold": {}}
    for size in D.d2_sizes:
        conds_f = sorted({k[2] for k in D.d2 if k[0] == size and k[1] == "find"}, key=lambda c: (c[0] != "L", c))
        base = {t: rows[0] for t, rows in D.d2.get((size, "find", "L0"), {}).items()}
        for c in conds_f:
            rows = {t: rs[0] for t, rs in D.d2[(size, "find", c)].items()}
            ent = {"n": len(rows)}
            for m in ("tool_call", "hit_file", "hit_func", "followed_wrong"):
                v = [int(r.get(m, False)) for r in rows.values()]
                ent[m] = _r(np.mean(v))
                if c != "L0" and m in ("hit_file", "hit_func"):
                    ent[f"{m}_vs_L0"] = mcnemar_tasks({t: int(r.get(m, False)) for t, r in base.items()},
                                                       {t: int(r.get(m, False)) for t, r in rows.items()})
            ent["tools"] = dict(collections.Counter(r.get("tool") for r in rows.values()))
            out["find"][f"{size}/{c}"] = ent
        conds_u = sorted({k[2] for k in D.d2 if k[0] == size and k[1] == "use"}, key=lambda c: (c[0] != "L", c))
        flags = {c: use_task_flags(D, size, c) for c in conds_u}
        for c in conds_u:
            f = flags[c]
            ts = sorted(f)
            n_s = sum(f[t]["n"] for t in ts)
            n_edit = sum(f[t]["n_edit"] for t in ts)
            tools = collections.Counter()
            for t in ts:
                tools.update(f[t]["tools"])
            ent = {"n_tasks": len(ts), "n_samples": n_s,
                   "task_any_on_target": _r(np.mean([f[t]["any_on_target"] for t in ts])),
                   "task_any_exact": _r(np.mean([f[t]["any_exact"] for t in ts])),
                   "sample_on_target": _r(sum(f[t]["n_on_target"] for t in ts) / n_s),
                   "sample_exact": _r(sum(f[t]["n_exact"] for t in ts) / n_s),
                   "sample_applies": _r(sum(f[t]["n_applies"] for t in ts) / n_s),
                   "edit_rate": _r(n_edit / n_s),
                   "edit_rate_ci": [_r(x) for x in cs.bootstrap_ratio_ci([f[t]["n_edit"] for t in ts], [f[t]["n"] for t in ts], NBOOT)],
                   "n_edit": n_edit,
                   "on_target_given_edit": _r(sum(f[t]["n_edit_on_target"] for t in ts) / n_edit) if n_edit else None,
                   "on_target_given_edit_ci": [_r(x) for x in cs.bootstrap_ratio_ci([f[t]["n_edit_on_target"] for t in ts], [f[t]["n_edit"] for t in ts], NBOOT)] if n_edit else None,
                   "exact_given_edit": _r(sum(f[t]["n_edit_exact"] for t in ts) / n_edit) if n_edit else None,
                   "exact_given_edit_ci": [_r(x) for x in cs.bootstrap_ratio_ci([f[t]["n_edit_exact"] for t in ts], [f[t]["n_edit"] for t in ts], NBOOT)] if n_edit else None,
                   "tool_shares": {k: _r(v / n_s) for k, v in tools.most_common()}}
            if c != "L0":
                for m in ("any_on_target", "any_exact"):
                    ent[f"{m}_vs_L0"] = mcnemar_tasks({t: v[m] for t, v in flags["L0"].items()}, {t: v[m] for t, v in f.items()})
            out["use"][f"{size}/{c}"] = ent
        for m in ("any_on_target", "any_exact"):
            ts = sorted(set.intersection(*[set(flags[c]) for c in LADDER_D2 if c in flags]))
            mat = [[flags[c][t][m] for c in LADDER_D2] for t in ts]
            out["cochran"][f"{size}/{m}"] = {**cs.cochran_q(mat), "conds": LADDER_D2}
        # information threshold (PLAN §5), exact only: `solves` needs notebooks/C03v_test_verify__CPU.ipynb
        thr = collections.Counter()
        for t in sorted(flags["L0"]):
            lvl = None
            for c in LADDER_D2:
                if t in flags.get(c, {}) and flags[c][t]["any_exact"]:
                    lvl = c
                    break
            missing = any(t not in flags.get(c, {}) for c in LADDER_D2)
            thr[lvl or (">L5" if not missing else "none (L4/L5 missing)")] += 1
        out["threshold"][size] = dict(thr)
    return out


# --------------------------------------------------------------------------- pre-registered hypotheses

def hypotheses(D: Data, d2: dict) -> dict:
    H: dict = {}
    # H1: location at find vs use
    for size in D.sizes:
        df, du = D.delta(size, "find", "L2", "L0"), D.delta(size, "use", "L2", "L0")
        diff = {t: df[t] - du[t] for t in set(df) & set(du)}
        p = cs.holm({"find>0": cs.wilcoxon_p(list(df.values())), "find>use": cs.wilcoxon_p(list(diff.values()))})
        sf, su, sd = summ(df), summ(du), summ(diff)
        H[f"H1/{size}"] = {
            "d_find_L2-L0": sf, "d_use_L2-L0": su, "d_find-minus-d_use": sd,
            "p_holm": {k: _r(v, 6) for k, v in p.items()},
            "part1_supported": bool(sf["median"] > 0 and p["find>0"] is not None and p["find>0"] < ALPHA),
            "part2_supported": bool(su["median"] < sf["median"] and sd["median"] > 0 and p["find>use"] is not None and p["find>use"] < ALPHA)}
    # H2: words gain rises with size, diff gain falls with size (Page's L over the available sizes)
    fam = {}
    for name, (a, b), increasing in (("L4-L3", ("L4", "L3"), True), ("L5-L4", ("L5", "L4"), False)):
        per = {s: D.delta(s, "use", a, b) for s in D.sizes}
        ts = sorted(set.intersection(*[set(v) for v in per.values()]))
        cols = D.sizes if increasing else list(reversed(D.sizes))
        pg = cs.page_trend([[per[s][t] for s in cols] for t in ts])
        fam[name] = pg["p"]
        H[f"H2/{name}"] = {"predicted": "increases with size" if increasing else "decreases with size",
                           "sizes_used": D.sizes, "page": pg, "per_size": {s: summ(per[s]) for s in D.sizes}}
    for name, p in cs.holm(fam).items():
        pg = H[f"H2/{name}"]
        pg["p_holm"] = _r(p, 6)
        pg["supported"] = bool(p is not None and p < ALPHA)
    # H3: graph encodings
    for size in D.sizes:
        pair = {"G-edges-G-json": D.delta(size, "find", "G-edges", "G-json"),
                "G-adj-G-json": D.delta(size, "find", "G-adj", "G-json"),
                "G-adj-G-edges": D.delta(size, "find", "G-adj", "G-edges")}
        vsl2 = {g: D.delta(size, "find", g, "L2") for g in ("G-json", "G-edges", "G-adj")}
        small = size in ("e2b", "e4b")
        fam = {f"{g}<L2": cs.wilcoxon_p(list(v.values())) for g, v in vsl2.items()}
        if small:
            fam.update({"G-edges>G-json": cs.wilcoxon_p(list(pair["G-edges-G-json"].values())),
                        "G-adj>G-json": cs.wilcoxon_p(list(pair["G-adj-G-json"].values()))})
        ph = cs.holm(fam)
        ent = {"pairs": {k: summ(v) for k, v in pair.items()}, "vs_L2": {g: summ(v) for g, v in vsl2.items()},
               "p_holm": {k: _r(v, 6) for k, v in ph.items()},
               "graph_below_L2_supported": all(summ(v)["median"] < 0 and ph[f"{g}<L2"] is not None and ph[f"{g}<L2"] < ALPHA for g, v in vsl2.items()),
               "scoring_modes": {c: dict(D.mode[(size, "find", c)]) for c in ("L0", "L2", "G-json")}}
        if small:
            ent["edges_adj_beat_json_supported"] = all(
                summ(pair[k])["median"] > 0 and ph[h] is not None and ph[h] < ALPHA
                for k, h in (("G-edges-G-json", "G-edges>G-json"), ("G-adj-G-json", "G-adj>G-json")))
        if size == "31b":
            ent["encodings_within_0.3_supported"] = all(abs(summ(v)["median"]) < 0.3 for v in pair.values())
        if size == "12b":
            ent["note"] = "encoding comparison is not pre-registered for 12b (exploratory)"
        H[f"H3/{size}"] = ent
    # H4: wrong hints at find
    for size in D.sizes:
        dd = {x: D.delta(size, "find", x, "L0") for x in ("X-file", "X-func", "X-graph")}
        ph = cs.holm({x: cs.wilcoxon_p(list(v.values())) for x, v in dd.items()})
        H[f"H4/{size}"] = {"vs_L0": {x: summ(v) for x, v in dd.items()},
                           "vs_matched": {x: summ(D.delta(size, "find", x, MATCHED[x])) for x in dd},
                           "p_holm": {k: _r(v, 6) for k, v in ph.items()},
                           "supported": {x: bool(summ(v)["median"] < 0 and ph[x] is not None and ph[x] < ALPHA) for x, v in dd.items()}}
    fam, pages = {}, {}
    for x in ("X-file", "X-func", "X-graph"):
        per = {s: D.delta(s, "find", x, "L0") for s in D.sizes}
        ts = sorted(set.intersection(*[set(v) for v in per.values()]))
        pages[x] = cs.page_trend([[per[s][t] for s in D.sizes] for t in ts])  # predicted: harm shrinks with size
        fam[x] = pages[x]["p"]
    ph = cs.holm(fam)
    H["H4/trend"] = {"predicted": "Δ(X−L0) increases with size (less harm for larger sizes)", "sizes_used": D.sizes,
                     "page": pages, "p_holm": {k: _r(v, 6) for k, v in ph.items()},
                     "trend_in_predicted_direction": {x: bool(ph[x] is not None and ph[x] < ALPHA) for x in pages},
                     "harm_at_every_size": {x: all(summ(D.delta(s, "find", x, "L0"))["median"] < 0 for s in D.sizes) for x in pages}}
    H["H4/trend"]["supported"] = {x: H["H4/trend"]["trend_in_predicted_direction"][x] and H["H4/trend"]["harm_at_every_size"][x]
                                  for x in pages}
    # H5: e4b on_target at S-use, L5 vs L2
    for size in D.d2_sizes:
        u = d2["use"]
        l5, l2 = u.get(f"{size}/L5"), u.get(f"{size}/L2")
        if not (l5 and l2):
            continue
        f5, f2 = use_task_flags(D, size, "L5"), use_task_flags(D, size, "L2")
        ent = {"task_any_on_target": {"L2": l2["task_any_on_target"], "L5": l5["task_any_on_target"]},
               "sample_on_target": {"L2": l2["sample_on_target"], "L5": l5["sample_on_target"]},
               "task_any_exact": {"L2": l2["task_any_exact"], "L5": l5["task_any_exact"]},
               "mcnemar_on_target_L2_vs_L5": mcnemar_tasks({t: v["any_on_target"] for t, v in f2.items()},
                                                           {t: v["any_on_target"] for t, v in f5.items()}),
               "mcnemar_exact_L2_vs_L5": mcnemar_tasks({t: v["any_exact"] for t, v in f2.items()},
                                                       {t: v["any_exact"] for t, v in f5.items()}),
               "primary_unit": "task (at least 1 of 2 samples), as in the PLAN §5 threshold rule; samples also shown"}
        ent["L5_ge_50"] = bool(l5["task_any_on_target"] >= 0.5)
        ent["L2_le_20"] = bool(l2["task_any_on_target"] <= 0.2)
        ent["supported"] = ent["L5_ge_50"] and ent["L2_le_20"]
        ent["pre_registered"] = size == "e4b"
        H[f"H5/{size}"] = ent
    H["H6"] = {"status": "not testable: needs the 12b rollout validation (PLAN §8), not run"}
    return H


# --------------------------------------------------------------------------- exploratory checks (coordinator's questions)

def graph_check(D: Data, d2: dict) -> dict:
    out = {}
    comps = [("L1-L0", "L1", "L0"), ("L2-L1", "L2", "L1"), ("X-func-L0", "X-func", "L0"), ("L2-X-func", "L2", "X-func"),
             ("G-json-L0", "G-json", "L0"), ("X-graph-L0", "X-graph", "L0"), ("G-json-X-graph", "G-json", "X-graph"),
             ("X-graph-L1", "X-graph", "L1"), ("G-json-L2", "G-json", "L2")]
    for size in D.sizes:
        for state in ("find", "use"):
            ent = {name: summ(D.delta(size, state, a, b)) for name, a, b in comps}
            ent["modes"] = {c: dict(D.mode[(size, state, c)]) for c in ("L0", "L1", "L2", "G-json", "X-graph")}
            out[f"{size}/{state}"] = ent
    out["depth2_find"] = {k: {m: v.get(m) for m in ("n", "hit_file", "hit_func")} for k, v in d2["find"].items()}
    return out


def edit_check(d2: dict) -> dict:
    return {k: {m: v.get(m) for m in ("n_samples", "edit_rate", "edit_rate_ci", "n_edit", "on_target_given_edit",
                                       "on_target_given_edit_ci", "exact_given_edit", "exact_given_edit_ci",
                                       "sample_on_target", "task_any_on_target", "tool_shares")}
            for k, v in d2["use"].items()}


# --------------------------------------------------------------------------- tables (markdown)

def _ci(s):
    if not s or not s.get("n"):
        return "—"
    return f"{s['median']:+.1f} [{s['ci'][0]:+.1f}, {s['ci'][1]:+.1f}]"


def _p(p):
    return "—" if p is None else ("<0.001" if p < 0.001 else f"{p:.3f}")


def _pct(x):
    return "—" if x is None else f"{100 * x:.0f}%"


def tables_md(D: Data, A: dict) -> str:
    T, d1, d2, H = [], A["depth1"], A["depth2"], A["hypotheses"]
    S = D.sizes
    T.append("# C03 probe — tables (data only)\n")
    T.append(f"Source: `results/C03_probe.json` (`notebooks/C03_probe__L4x4.ipynb`, plan {D.raw['plan_commit'][:7]}). "
             "Produced by `pilot_c/analyze_c03.py`. Single run. Δ in nats, median over tasks [95% bootstrap CI, 10,000 resamples]. "
             "p: two-sided Wilcoxon signed-rank (unadjusted unless marked Holm).\n")
    # coverage
    T.append("## T0. Coverage\n")
    T.append("| Size | depth-1 cells ok | depth-2 samples ok | skipped | server start (s) | depth 1 (min) | depth 2 (min) | L0 rerun |Δ| find median / max | use median / max |")
    T.append("|---|---|---|---|---|---|---|---|---|")
    for s in D.raw["sizes"]:
        size = s.get("size") or s["model"].split("-")[2]
        n1 = sum(1 for r in D.raw["d1"] if r["size"] == size and r["ok"])
        n2 = sum(1 for r in D.raw["d2"] if r["size"] == size and r["ok"])
        rf, ru = d1.get(f"{size}/find/L0-rerun-absdiff", {}), d1.get(f"{size}/use/L0-rerun-absdiff", {})
        T.append(f"| {size} | {n1} | {n2} | {s.get('skipped') or '—'} | {s.get('startup_s', '—')} | {s.get('d1_minutes', '—')} | "
                 f"{s.get('d2_minutes', '—')} | {rf.get('median', '—')} / {rf.get('max', '—')} | {ru.get('median', '—')} / {ru.get('max', '—')} |")
    T.append(f"\nSession: {D.raw['session_elapsed_h']} h; freezes: {len(D.raw['freezes'])}; tasks: {len(D.raw['task_ids'])}; "
             f"excluded before probing: {len(D.raw['excluded'])}.\n")
    # depth 1
    for state, conds in (("find", FIND_CONDS), ("use", USE_CONDS)):
        T.append(f"## T1-{state}. Depth 1, S-{state}: Δ vs L0 (and X vs its matched correct rung)\n")
        T.append("| Condition | " + " | ".join(f"{s} Δ [CI]" for s in S) + " | " + " | ".join(f"{s} n, p" for s in S) + " |")
        T.append("|---|" + "---|" * (2 * len(S)))
        rows = [(f"{c} − L0", f"{c}-L0") for c in conds] + [(f"{x} − {m}", f"{x}-{m}") for x, m in MATCHED.items() if x in conds]
        for label, key in rows:
            cells = [d1.get(f"{s}/{state}/{key}") for s in S]
            T.append(f"| {label} | " + " | ".join(_ci(c) for c in cells) + " | " +
                     " | ".join("—" if not c or not c.get("n") else f"{c['n']}, {_p(c['p'])}" for c in cells) + " |")
        T.append("")
        T.append("Scoring mode per condition (PLAN §11 dev. 8): " + "; ".join(
            f"{s}: " + ", ".join(f"{c}={'/'.join(m)}" for c, m in d1[f"{s}/{state}/modes"].items())
            for s in S if s in ("31b", "12b")) + "\n")
    # hypotheses
    T.append("## T2. Pre-registered hypotheses (PLAN §6), per size\n")
    T.append("| Hypothesis | Size | Statistic | Value | p (Holm) | Verdict |")
    T.append("|---|---|---|---|---|---|")
    for s in S:
        h = H[f"H1/{s}"]
        T.append(f"| H1 location helps find | {s} | Δ_find(L2−L0) | {_ci(h['d_find_L2-L0'])} | {_p(h['p_holm']['find>0'])} | {'supported' if h['part1_supported'] else 'not supported'} |")
        T.append(f"| H1 find > use | {s} | Δ_use(L2−L0); Δ_find−Δ_use | {_ci(h['d_use_L2-L0'])}; {_ci(h['d_find-minus-d_use'])} | {_p(h['p_holm']['find>use'])} | {'supported' if h['part2_supported'] else 'not supported'} |")
    for k in ("L4-L3", "L5-L4"):
        h = H[f"H2/{k}"]
        T.append(f"| H2 Δ_use({k.replace('-', '−')}) {h['predicted']} | {', '.join(h['sizes_used'])} (e2b missing) | Page L (n={h['page']['n']}); medians " +
                 ", ".join(f"{s} {h['per_size'][s]['median']:+.1f}" for s in S) + f" | L={_r(h['page']['L'], 1)} | {_p(h['p_holm'])} | {'supported' if h['supported'] else 'not supported'} |")
    for s in S:
        h = H[f"H3/{s}"]
        for g in ("G-json", "G-edges", "G-adj"):
            T.append(f"| H3 graph < L2 (find) | {s} | {g} − L2 | {_ci(h['vs_L2'][g])} | {_p(h['p_holm'][g + '<L2'])} | " +
                     ("—" if g != "G-adj" else ("supported (all three)" if h["graph_below_L2_supported"] else "not supported")) + " |")
        for k in ("G-edges-G-json", "G-adj-G-json", "G-adj-G-edges"):
            hk = {"G-edges-G-json": "G-edges>G-json", "G-adj-G-json": "G-adj>G-json"}.get(k)
            verdict = "—"
            if s in ("e2b", "e4b") and k == "G-adj-G-json":
                verdict = "supported" if h.get("edges_adj_beat_json_supported") else "not supported"
            if s == "31b" and k == "G-adj-G-edges":
                verdict = "supported (|median| < 0.3 for all pairs)" if h.get("encodings_within_0.3_supported") else "not supported (some |median| ≥ 0.3)"
            if s == "12b":
                verdict = "exploratory"
            T.append(f"| H3 encodings (find) | {s} | {k.replace('-G', ' − G')} | {_ci(h['pairs'][k])} | {_p(h['p_holm'].get(hk)) if hk and hk in h['p_holm'] else _p(h['pairs'][k]['p']) + ' (unadj.)'} | {verdict} |")
    for s in S:
        h = H[f"H4/{s}"]
        for x in ("X-file", "X-func", "X-graph"):
            T.append(f"| H4 wrong hint harms (find) | {s} | {x} − L0 | {_ci(h['vs_L0'][x])} | {_p(h['p_holm'][x])} | {'supported' if h['supported'][x] else 'not supported'} |")
    h = H["H4/trend"]
    for x in ("X-file", "X-func", "X-graph"):
        v = ("supported" if h["supported"][x] else
             "not supported (trend in predicted direction, but Δ > 0 at some size: no harm to shrink)" if h["trend_in_predicted_direction"][x]
             else "not supported")
        T.append(f"| H4 harm shrinks with size | {', '.join(h['sizes_used'])} | Page L (n={h['page'][x]['n']}), {x} − L0 | L={_r(h['page'][x]['L'], 1)} | {_p(h['p_holm'][x])} | {v} |")
    for s in D.d2_sizes:
        h = H.get(f"H5/{s}")
        if not h:
            continue
        tag = "" if h["pre_registered"] else " (exploratory: H5 is about e4b)"
        T.append(f"| H5 on_target L5 ≥ 50%, L2 ≤ 20%{tag} | {s} | tasks with ≥1 of 2 on_target: L2, L5 (samples: L2, L5) | "
                 f"{_pct(h['task_any_on_target']['L2'])}, {_pct(h['task_any_on_target']['L5'])} ({_pct(h['sample_on_target']['L2'])}, {_pct(h['sample_on_target']['L5'])}) | "
                 f"McNemar {_p(h['mcnemar_on_target_L2_vs_L5']['p'])} | {'supported' if h['supported'] else 'not supported'} (L5≥50%: {h['L5_ge_50']}, L2≤20%: {h['L2_le_20']}) |")
    T.append("| H6 probe predicts rollout flips | 12b | AUC | — | — | not testable (validation not run) |")
    T.append("")
    # depth 2 find
    T.append("## T3. Depth 2, S-find (1 sample per task): share of tasks\n")
    T.append("| Size | Condition | n | hit_file | hit_func | McNemar vs L0 (file; func) | tools |")
    T.append("|---|---|---|---|---|---|---|")
    for k, v in d2["find"].items():
        s, c = k.split("/")
        mc = "—" if c == "L0" else f"{_p(v['hit_file_vs_L0']['p'])}; {_p(v['hit_func_vs_L0']['p'])}"
        T.append(f"| {s} | {c} | {v['n']} | {_pct(v['hit_file'])} | {_pct(v['hit_func'])} | {mc} | " +
                 ", ".join(f"{t} {n}" for t, n in v["tools"].items()) + " |")
    T.append("")
    # depth 2 use
    T.append("## T4. Depth 2, S-use (2 samples per task)\n")
    T.append("| Size | Condition | tasks / samples | tasks ≥1 on_target | tasks ≥1 exact | McNemar vs L0 (on_target; exact) | samples: edit_file on gold file [CI] | on_target given edit [CI] | exact given edit [CI] |")
    T.append("|---|---|---|---|---|---|---|---|---|")
    for k, v in d2["use"].items():
        s, c = k.split("/")
        mc = "—" if c == "L0" else f"{_p(v['any_on_target_vs_L0']['p'])}; {_p(v['any_exact_vs_L0']['p'])}"
        ci = lambda x: "—" if not x else f"[{_pct(x[0])}, {_pct(x[1])}]"  # noqa: E731
        T.append(f"| {s} | {c} | {v['n_tasks']} / {v['n_samples']} | {_pct(v['task_any_on_target'])} | {_pct(v['task_any_exact'])} | {mc} | "
                 f"{_pct(v['edit_rate'])} {ci(v['edit_rate_ci'])} | {_pct(v['on_target_given_edit'])} {ci(v['on_target_given_edit_ci'])} (n={v['n_edit']}) | "
                 f"{_pct(v['exact_given_edit'])} {ci(v['exact_given_edit_ci'])} |")
    T.append("")
    T.append("Cochran's Q across " + ", ".join(LADDER_D2) + " (tasks with all five cells): " + "; ".join(
        f"{k}: Q={_r(v['Q'], 1)}, p={_p(v['p'])}, n={v['n']}" for k, v in d2["cochran"].items()) + "\n")
    T.append("## T5. Information threshold per task (lowest S-use rung with ≥1 of 2 samples `exact`; `solves` pending `notebooks/C03v_test_verify__CPU.ipynb`)\n")
    T.append("| Size | " + " | ".join(LADDER_D2 + [">L5", "none (L4/L5 missing)"]) + " |")
    T.append("|---|" + "---|" * (len(LADDER_D2) + 2))
    for s, thr in d2["threshold"].items():
        T.append(f"| {s} | " + " | ".join(str(thr.get(c, 0)) for c in LADDER_D2 + [">L5", "none (L4/L5 missing)"]) + " |")
    T.append("")
    # exploratory
    T.append("## T6. Exploratory: graph rung vs text location (coordinator's question a)\n")
    G = A["exploratory"]["graph"]
    names = ["L1-L0", "L2-L1", "X-func-L0", "L2-X-func", "G-json-L0", "X-graph-L0", "G-json-X-graph", "X-graph-L1", "G-json-L2"]
    T.append("| Size / state | " + " | ".join(n.replace("-", " − ").replace("G − json", "G-json").replace("X − graph", "X-graph").replace("X − func", "X-func") for n in names) + " | scoring mode L0 / G-json |")
    T.append("|---|" + "---|" * (len(names) + 1))
    for k, v in G.items():
        if k == "depth2_find":
            continue
        md = v["modes"]
        T.append(f"| {k} | " + " | ".join(_ci(v[n]) for n in names) + f" | {'/'.join(md.get('L0', {}))} / {'/'.join(md.get('G-json', {}))} |")
    T.append("")
    T.append("## T7. Exploratory: first action at S-use (coordinator's question b), share of samples\n")
    E = A["exploratory"]["edit"]
    tools = ["edit_file", "read_file", "run_command", "write_file(other)", "get_code_neighbors", "search_similar_code"]
    T.append("| Size | Condition | n | " + " | ".join(tools) + " |")
    T.append("|---|---|---|" + "---|" * len(tools))
    for k, v in E.items():
        s, c = k.split("/")
        T.append(f"| {s} | {c} | {v['n_samples']} | " + " | ".join(_pct(v["tool_shares"].get(t, 0.0)) for t in tools) + " |")
    T.append("")
    return "\n".join(T)


# --------------------------------------------------------------------------- figures (data only)

COLORS = {"e4b": "#2a78d6", "12b": "#eb6834", "31b": "#1baf7a", "e2b": "#eda100"}  # fixed per size (slot 4 = e2b, later)
MARKERS = {"e4b": "o", "12b": "s", "31b": "D", "e2b": "^"}
INK, INK2, GRID, SURF = "#0b0b0b", "#52514e", "#e6e5e0", "#fcfcfb"
MIN_EDITS_FOR_RATE = 5


def _style(ax):
    ax.set_facecolor(SURF)
    for sp in ("top", "right", "left"):
        ax.spines[sp].set_visible(False)
    ax.spines["bottom"].set_color(INK2)
    ax.tick_params(colors=INK2, labelsize=8, length=0)
    ax.grid(axis="x", color=GRID, linewidth=0.6)
    ax.set_axisbelow(True)


def _dotrows(ax, rows, sizes, getter, xlabel, zero=True):
    k = len(sizes)
    off = {s: (i - (k - 1) / 2) * 0.22 for i, s in enumerate(sizes)}
    for yi, label in enumerate(rows):
        for s in sizes:
            v = getter(s, label)
            if not v:
                continue
            m, lo, hi = v
            y = yi + off[s]
            ax.plot([lo, hi], [y, y], color=COLORS[s], linewidth=1.2, solid_capstyle="round")
            ax.plot([m], [y], marker=MARKERS[s], color=COLORS[s], markersize=4.5, markeredgecolor=SURF, markeredgewidth=0.8, linestyle="none")
    ax.set_yticks(range(len(rows)))
    ax.set_yticklabels(rows)
    if not ax.yaxis_inverted():  # shared y axes: invert once
        ax.invert_yaxis()
    if zero:
        ax.axvline(0, color=INK2, linewidth=0.8)
    ax.set_xlabel(xlabel, color=INK2, fontsize=8)
    _style(ax)


def _legend(fig, sizes):
    import matplotlib.lines as ml
    h = [ml.Line2D([], [], color=COLORS[s], marker=MARKERS[s], linestyle="-", linewidth=1.2, markersize=4.5, label=s) for s in sizes]
    fig.legend(handles=h, loc="upper right", bbox_to_anchor=(1.0, 1.0), ncol=len(sizes), frameon=False, fontsize=8, labelcolor=INK)


def figures(D: Data, A: dict, outdir: Path, stem: str = "C03") -> list[str]:
    import matplotlib
    matplotlib.use("Agg")
    matplotlib.rcParams["svg.hashsalt"] = "pilot-c"   # deterministic SVG ids, so reruns reproduce the committed files
    import matplotlib.pyplot as plt
    outdir.mkdir(parents=True, exist_ok=True)
    d1, made = A["depth1"], []

    def g1(state):
        def f(s, c):
            v = d1.get(f"{s}/{state}/{c}-L0")
            return (v["median"], *v["ci"]) if v and v.get("n") else None
        return f

    fig, axes = plt.subplots(1, 2, figsize=(9, 5.2), gridspec_kw={"width_ratios": [1, 1.25]}, facecolor=SURF)
    _dotrows(axes[0], FIND_CONDS, D.sizes, g1("find"), "Δ log p(target) vs L0, nats (median, 95% CI)")
    axes[0].set_title("S-find", color=INK, fontsize=10, loc="left")
    _dotrows(axes[1], USE_CONDS, D.sizes, g1("use"), "Δ log p(target) vs L0, nats (median, 95% CI)")
    axes[1].set_title("S-use", color=INK, fontsize=10, loc="left")
    fig.suptitle("Depth 1: gain in target log-likelihood over L0, per condition and size", color=INK, fontsize=10, x=0.01, y=0.985, ha="left")
    _legend(fig, D.sizes)
    fig.tight_layout(rect=(0, 0, 1, 0.955))
    for ext in ("png", "svg"):
        p = outdir / f"{stem}_fig1_depth1_deltas.{ext}"
        fig.savefig(p, dpi=200, facecolor=SURF, metadata={"Date": None} if ext == "svg" else None)
        made.append(p.name)
    plt.close(fig)

    u = A["depth2"]["use"]
    conds = [c for c in ("L0", "L2", "L3", "L4", "L5", "T", "W", "G-json", "X-func", "X-test") if any(f"{s}/{c}" in u for s in D.d2_sizes)]
    panels = [("edit_rate", "edit_rate_ci", "edit_file on the gold file, share of samples"),
              ("on_target_given_edit", "on_target_given_edit_ci", "on_target among those edits"),
              ("exact_given_edit", "exact_given_edit_ci", "exact among those edits")]  # shown only with >= 5 edits
    fig, axes = plt.subplots(1, 3, figsize=(10, 4.2), sharey=True, facecolor=SURF)
    for ax, (m, mci, lab) in zip(axes, panels):
        def g(s, c, m=m, mci=mci):
            v = u.get(f"{s}/{c}")
            if m != "edit_rate" and v and (v.get("n_edit") or 0) < MIN_EDITS_FOR_RATE:
                return None  # conditional rate on fewer than MIN_EDITS_FOR_RATE edits: not shown (n in T4)
            return (v[m], *v[mci]) if v and v.get(m) is not None and v.get(mci) else None
        _dotrows(ax, conds, D.d2_sizes, g, lab + " (95% CI)", zero=False)
        ax.set_xlim(-0.03, 1.03)
    fig.suptitle("Depth 2, S-use: does the model edit now, and how good is the edit when it does", color=INK, fontsize=10, x=0.01, y=0.985, ha="left")
    _legend(fig, D.d2_sizes)
    fig.tight_layout(rect=(0, 0, 1, 0.94))
    for ext in ("png", "svg"):
        p = outdir / f"{stem}_fig2_depth2_use_edit.{ext}"
        fig.savefig(p, dpi=200, facecolor=SURF, metadata={"Date": None} if ext == "svg" else None)
        made.append(p.name)
    plt.close(fig)

    G = A["exploratory"]["graph"]
    comps = [("L1-L0", "text: file (L1 − L0)"), ("X-func-L0", "text: file + wrong function (X-func − L0)"),
             ("L2-X-func", "text: right vs wrong function (L2 − X-func)"), ("X-graph-L0", "graph: wrong focus, same file (X-graph − L0)"),
             ("G-json-X-graph", "graph: right vs wrong focus (G-json − X-graph)"), ("X-graph-L1", "graph wrong focus vs text file (X-graph − L1)")]
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True, facecolor=SURF)
    for ax, state in zip(axes, ("find", "use")):
        def g(s, lab, state=state):
            key = dict((b, a) for a, b in comps)[lab]
            v = G.get(f"{s}/{state}", {}).get(key)
            return (v["median"], *v["ci"]) if v and v.get("n") else None
        _dotrows(ax, [b for _, b in comps], D.sizes, g, "Δ log p(target), nats (median, 95% CI)")
        ax.set_title(f"S-{state}", color=INK, fontsize=10, loc="left")
    fig.suptitle("Location as text vs as graph-tool output (exploratory decomposition)", color=INK, fontsize=10, x=0.01, y=0.985, ha="left")
    _legend(fig, D.sizes)
    fig.tight_layout(rect=(0, 0, 1, 0.94))
    for ext in ("png", "svg"):
        p = outdir / f"{stem}_fig3_graph_vs_text.{ext}"
        fig.savefig(p, dpi=200, facecolor=SURF, metadata={"Date": None} if ext == "svg" else None)
        made.append(p.name)
    plt.close(fig)
    return made


def analyze_all(d: dict, source: str, out_dir: Path, fig_dir: Path, stem: str = "C03") -> dict:
    """Every table and figure of this module for a probe result dict `d` (C03's, or C03 merged with C04's design
    cells); writes <stem>_analysis.json, <stem>_tables.md and <stem>_fig*.png/svg."""
    D = Data(d)
    A = {"source": source, "plan_commit": d["plan_commit"], "sizes_with_depth1": D.sizes, "sizes_with_depth2": D.d2_sizes,
         "depth1": depth1_tables(D)}
    A["depth2"] = depth2_tables(D)
    A["hypotheses"] = hypotheses(D, A["depth2"])
    A["exploratory"] = {"graph": graph_check(D, A["depth2"]), "edit": edit_check(A["depth2"])}
    try:
        A["figures"] = figures(D, A, fig_dir, stem)
    except ImportError as e:   # matplotlib missing: tables still written
        A["figures"] = [f"skipped: {e}"]
    out_dir.mkdir(parents=True, exist_ok=True)
    (out_dir / f"{stem}_analysis.json").write_text(json.dumps(A, indent=1, ensure_ascii=False, default=str))
    (out_dir / f"{stem}_tables.md").write_text(tables_md(D, A).replace("Source: `results/C03_probe.json`", f"Source: `{source}`"))
    print(f"wrote {out_dir}/{stem}_analysis.json, {stem}_tables.md,", ", ".join(A["figures"]))
    return A


def main(path: str = "results/C03_probe.json") -> None:
    analyze_all(json.load(open(ROOT / path)), path, ROOT / "results", ROOT / "results" / "figures", "C03")


if __name__ == "__main__":
    main(*sys.argv[1:])

In [ ]:
%%writefile /kaggle/working/pilot/pilot_c/analyze_d1.py
"""Frozen analysis of notebooks/D1_heldout_probe__L4x4.ipynb, as pre-registered in docs/PLAN_D1_heldout_probe.md §5.
Standalone: it runs as D1's last cell on Kaggle and offline, and is tested on synthetic data (tests/test_analyze_d1.py).
Needs numpy, scipy (common/stats.py) and, for figures, matplotlib (all preinstalled on Kaggle).

The same functions run on the dev probe (C03 + C04, merged as in pilot_c/analyze_c04.py) to give the dev column of every
table: results/D1_dev_reference.json, built offline by `python -m pilot_c.analyze_d1 --dev-reference`.

Inputs : D1_heldout_probe.json (the notebook's output; rows d1 / d2 in C03's format), D1_dev_reference.json.
Outputs: D1_analysis.json, D1_tables.md, D1_fig1_criteria.png, D1_fig2_depth2.png (numbers only; no task text).
Usage  : python -m pilot_c.analyze_d1 <D1_heldout_probe.json> <D1_dev_reference.json> <out_dir>
         python -m pilot_c.analyze_d1 --dev-reference <C03_probe.json> <C04_probe_followup.json> <out.json>
"""
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np

from common import stats as cs
from pilot_c import analyze_c03 as a3

SIZES = ("e4b", "12b", "31b")
FIND = ("L0", "L1", "L2", "W", "G-json", "G-null", "X-func", "X-graph")     # depth 1, S-find (PLAN_D1 §2)
USE = ("L0", "L2", "W", "T", "L5", "G-json", "G-null")                      # depth 1, S-use
D2_FIND = ("L0", "L2", "G-json", "X-graph", "G-null")                       # depth 2, 1 sample each
D2_USE = ("L0", "L2", "L5")                                                 # depth 2, 2 samples each
ALPHA = 0.05
GAP_NATS = 2.0          # R4 tolerance
MIN_TASKS = 12          # a size is evaluable for a criterion only with >= 12 tasks having every cell it needs
# Depth-1 contrasts shown in the descriptive table (state, a, b)
CONTRASTS = [("find", c, "L0") for c in FIND[1:]] + [("find", "X-func", "L2"), ("find", "X-graph", "G-json"),
             ("find", "G-json", "G-null"), ("find", "L2", "X-func"), ("find", "G-json", "X-graph")] + \
            [("use", c, "L0") for c in USE[1:]] + [("use", "L5", "W"), ("use", "G-json", "G-null")]


def _r(x, k=3):
    return a3._r(x, k)


def _ok(s: dict) -> bool:
    return s.get("n", 0) >= MIN_TASKS


# --------------------------------------------------------------------------- criteria R1-R4 (PLAN_D1 §4)

def r1(D: a3.Data, size: str) -> dict:
    """Claim 1 / H1, the dev test unchanged: Holm within size over {find>0, find>use}, two-sided Wilcoxon."""
    df, du = D.delta(size, "find", "L2", "L0"), D.delta(size, "use", "L2", "L0")
    diff = {t: df[t] - du[t] for t in set(df) & set(du)}
    p = cs.holm({"find>0": cs.wilcoxon_p(list(df.values())), "find>use": cs.wilcoxon_p(list(diff.values()))})
    sf, su, sd = a3.summ(df), a3.summ(du), a3.summ(diff)
    ev = _ok(sd)
    part1 = ev and sf["median"] > 0 and p["find>0"] is not None and p["find>0"] < ALPHA
    part2 = ev and su["median"] < sf["median"] and sd["median"] > 0 and p["find>use"] is not None and p["find>use"] < ALPHA
    return {"d_find_L2-L0": sf, "d_use_L2-L0": su, "d_find-minus-d_use": sd, "p_holm": {k: _r(v, 6) for k, v in p.items()},
            "part1_location_helps_find": bool(part1) if ev else None, "holds": bool(part2) if ev else None}


def r2(D: a3.Data, size: str) -> dict:
    """Claim 1: words help the edit, the full diff helps more: median Δ_use(W−L0) > 0 and median Δ_use(L5−L0) >
    median Δ_use(W−L0), on the tasks with all three cells."""
    common = set(D.lp.get((size, "use", "L0"), {})) & set(D.lp.get((size, "use", "W"), {})) & set(D.lp.get((size, "use", "L5"), {}))
    sub = lambda d: {t: v for t, v in d.items() if t in common}  # noqa: E731
    w, l5, l5w = (a3.summ(sub(D.delta(size, "use", a, b))) for a, b in (("W", "L0"), ("L5", "L0"), ("L5", "W")))
    ev = len(common) >= MIN_TASKS
    holds = ev and w["median"] > 0 and l5["median"] > w["median"]
    return {"n_tasks": len(common), "d_use_W-L0": w, "d_use_L5-L0": l5, "d_use_L5-W": l5w, "holds": bool(holds) if ev else None}


def r3(D: a3.Data, size: str) -> dict:
    """Claim 3, find: graph content helps (median Δ_find(G-json−G-null) > 0) and which symbol the graph is about matters
    less than which location is named: |median Δ_find(G-json−X-graph)| < median Δ_find(L2−X-func)."""
    gn, gx, lx = (a3.summ(D.delta(size, "find", a, b)) for a, b in (("G-json", "G-null"), ("G-json", "X-graph"), ("L2", "X-func")))
    ev = _ok(gn) and _ok(gx) and _ok(lx)
    holds = ev and gn["median"] > 0 and abs(gx["median"]) < lx["median"]
    return {"d_find_G-json-G-null": gn, "d_find_G-json-X-graph": gx, "d_find_L2-X-func": lx, "holds": bool(holds) if ev else None}


def r4(D: a3.Data, size: str) -> dict:
    """Claim 3, use: graph content adds nothing at the edit: median Δ_use(G-null−L0) >= median Δ_use(G-json−L0) − 2."""
    a, b, c = (a3.summ(D.delta(size, "use", x, y)) for x, y in (("G-null", "L0"), ("G-json", "L0"), ("G-json", "G-null")))
    ev = _ok(a) and _ok(b)
    holds = ev and a["median"] >= b["median"] - GAP_NATS
    return {"d_use_G-null-L0": a, "d_use_G-json-L0": b, "d_use_G-json-G-null": c, "holds": bool(holds) if ev else None}


def verdict(per_size: dict) -> str:
    v = [per_size.get(s, {}).get("holds") for s in SIZES]
    if all(x is True for x in v):
        return "replicated"
    if any(x is False for x in v):
        return f"not replicated ({sum(x is True for x in v)} of 3 sizes hold)"
    return f"incomplete ({sum(x is True for x in v)} of 3 sizes hold, {sum(x is None for x in v)} not evaluable)"


# --------------------------------------------------------------------------- depth 2 (R5, descriptive)

def depth2(D: a3.Data) -> dict:
    out: dict = {"find": {}, "use": {}}
    for size in SIZES:
        base = {t: rs[0] for t, rs in D.d2.get((size, "find", "L0"), {}).items()}
        for c in D2_FIND:
            rows = {t: rs[0] for t, rs in D.d2.get((size, "find", c), {}).items()}
            if not rows:
                continue
            e = {"tasks": len(rows), **{m: _r(np.mean([bool(r.get(m)) for r in rows.values()])) for m in
                                        ("tool_call", "hit_file", "hit_func", "followed_wrong")}}
            if c != "L0":
                for m in ("hit_file", "hit_func"):
                    e[f"{m}_vs_L0"] = a3.mcnemar_tasks({t: int(bool(r.get(m))) for t, r in base.items()},
                                                       {t: int(bool(r.get(m))) for t, r in rows.items()})
            out["find"][f"{size}/{c}"] = e
        for c in D2_USE:
            per = D.d2.get((size, "use", c), {})
            rows = [r for rs in per.values() for r in rs]
            if not rows:
                continue
            mean = lambda k, rr=rows: _r(np.mean([bool(r.get(k)) for r in rr]))  # noqa: E731
            edits = [r for r in rows if r.get("edit_gold_file")]
            out["use"][f"{size}/{c}"] = {
                "tasks": len(per), "samples": len(rows), "tool_call": mean("tool_call"),
                "edit_now": mean("edit_gold_file"), "applies": mean("applies"), "on_target": mean("on_target"),
                "exact": mean("exact"), "exact_given_edit": _r(np.mean([bool(r.get("exact")) for r in edits])) if edits else None,
                "tasks_any_exact": _r(np.mean([any(r.get("exact") for r in rs) for rs in per.values()])),
                "first_tools": dict(sorted(__import__("collections").Counter(r.get("tool") or "none" for r in rows).items()))}
    return out


def depth1_table(D: a3.Data) -> dict:
    return {f"{s}/{st}/{a}-{b}": a3.summ(D.delta(s, st, a, b)) for s in SIZES for (st, a, b) in CONTRASTS}


def health(d: dict) -> dict:
    import collections
    out = {}
    for key in ("d1", "d2"):
        c = collections.Counter((r.get("size"), "ok" if r.get("ok") else (r.get("why") or "?").split(":")[0]) for r in d.get(key, []))
        out[key] = {f"{s}/{w}": n for (s, w), n in sorted(c.items(), key=str)}
    out["score_modes"] = dict(collections.Counter(r.get("mode") for r in d.get("d1", []) if r.get("ok")))
    return out


def analyze_rows(d: dict) -> dict:
    D = a3.Data(d)
    A = {"R1_claim1_find_vs_use": {s: r1(D, s) for s in SIZES}, "R2_claim1_words_and_diff": {s: r2(D, s) for s in SIZES},
         "R3_claim3_graph_find": {s: r3(D, s) for s in SIZES}, "R4_claim3_graph_use": {s: r4(D, s) for s in SIZES}}
    A["verdicts"] = {k.split("_")[0]: verdict(v) for k, v in A.items()}
    A["R5_depth2"] = depth2(D)
    A["depth1"] = depth1_table(D)
    A["sizes_with_depth1"] = D.sizes
    A["sizes_with_depth2"] = D.d2_sizes
    A["health"] = health(d)
    return A


# --------------------------------------------------------------------------- tables and figures

def _ci(s: dict | None) -> str:
    if not s or not s.get("n"):
        return "—"
    return f"{s['median']:+.1f} [{s['ci'][0]:+.1f}, {s['ci'][1]:+.1f}] (n={s['n']})"


def _p(p):
    return "—" if p is None else ("<0.001" if p < 0.001 else f"{p:.3f}")


def _yn(x):
    return {True: "holds", False: "fails", None: "not evaluable"}[x]


def _pct(x):
    return "—" if x is None else f"{100 * x:.0f}%"


def tables_md(A: dict, dev: dict | None, meta: dict) -> str:
    dev = dev or {}
    g = lambda k, s, f: (dev.get(k, {}).get(s) or {}).get(f)  # noqa: E731
    T = ["# D1 held-out probe: frozen criteria (docs/PLAN_D1_heldout_probe.md)", "",
         f"Tasks: {meta.get('n_tasks')} held-out ({meta.get('by_repo')}); sizes with depth 1: {A['sizes_with_depth1']}; "
         f"with depth 2: {A['sizes_with_depth2']}. Medians in nats with 95% bootstrap CI over tasks. Dev = C03 + C04 "
         "(30 dev tasks), same code.", "",
         "## Verdicts", "", "| criterion | held-out verdict | dev |", "|---|---|---|"]
    for k, v in A["verdicts"].items():
        T.append(f"| {k} | {v} | {dev.get('verdicts', {}).get(k, '—')} |")
    T += ["", "## R1 — claim 1 / H1: location helps find more than use (Holm within size, Wilcoxon)", "",
          "| size | Δ_find(L2−L0) | Δ_use(L2−L0) | Δ_find − Δ_use | Holm p (find>use) | held-out | dev Δ_find − Δ_use |", "|---|---|---|---|---|---|---|"]
    for s in SIZES:
        h = A["R1_claim1_find_vs_use"][s]
        T.append(f"| {s} | {_ci(h['d_find_L2-L0'])} | {_ci(h['d_use_L2-L0'])} | {_ci(h['d_find-minus-d_use'])} | "
                 f"{_p(h['p_holm']['find>use'])} | {_yn(h['holds'])} | {_ci(g('R1_claim1_find_vs_use', s, 'd_find-minus-d_use'))} |")
    T += ["", "## R2 — claim 1: words help the edit, the diff helps more (tasks with L0, W and L5)", "",
          "| size | Δ_use(W−L0) | Δ_use(L5−L0) | Δ_use(L5−W) | held-out | dev W−L0 / L5−L0 |", "|---|---|---|---|---|---|"]
    for s in SIZES:
        h = A["R2_claim1_words_and_diff"][s]
        T.append(f"| {s} | {_ci(h['d_use_W-L0'])} | {_ci(h['d_use_L5-L0'])} | {_ci(h['d_use_L5-W'])} | {_yn(h['holds'])} | "
                 f"{_ci(g('R2_claim1_words_and_diff', s, 'd_use_W-L0'))} / {_ci(g('R2_claim1_words_and_diff', s, 'd_use_L5-L0'))} |")
    T += ["", "## R3 — claim 3 at find: graph content helps; the symbol matters less than the named location", "",
          "| size | Δ_find(G-json−G-null) | Δ_find(G-json−X-graph) | Δ_find(L2−X-func) | held-out | dev G-json−G-null |", "|---|---|---|---|---|---|"]
    for s in SIZES:
        h = A["R3_claim3_graph_find"][s]
        T.append(f"| {s} | {_ci(h['d_find_G-json-G-null'])} | {_ci(h['d_find_G-json-X-graph'])} | {_ci(h['d_find_L2-X-func'])} | "
                 f"{_yn(h['holds'])} | {_ci(g('R3_claim3_graph_find', s, 'd_find_G-json-G-null'))} |")
    T += ["", f"## R4 — claim 3 at use: graph content adds nothing (G-null within {GAP_NATS:.0f} nats of G-json)", "",
          "| size | Δ_use(G-null−L0) | Δ_use(G-json−L0) | Δ_use(G-json−G-null) | held-out | dev G-json−G-null |", "|---|---|---|---|---|---|"]
    for s in SIZES:
        h = A["R4_claim3_graph_use"][s]
        T.append(f"| {s} | {_ci(h['d_use_G-null-L0'])} | {_ci(h['d_use_G-json-L0'])} | {_ci(h['d_use_G-json-G-null'])} | "
                 f"{_yn(h['holds'])} | {_ci(g('R4_claim3_graph_use', s, 'd_use_G-json-G-null'))} |")
    T += ["", "## R5 — depth 2 (sampled first action), descriptive", "",
          "S-find, share of tasks (1 sample each); McNemar vs L0 on the same tasks.", "",
          "| size | cond | reads gold file | reads gold function | follows wrong | dev gold file / function |", "|---|---|---|---|---|---|"]
    df, du = dev.get("R5_depth2", {}).get("find", {}), dev.get("R5_depth2", {}).get("use", {})
    for k, e in A["R5_depth2"]["find"].items():
        mc = e.get("hit_file_vs_L0", {})
        T.append(f"| {k.split('/')[0]} | {k.split('/')[1]} | {_pct(e['hit_file'])}" + (f" (p {_p(mc.get('p'))})" if mc else "")
                 + f" | {_pct(e['hit_func'])} | {_pct(e['followed_wrong'])} | {_pct((df.get(k) or {}).get('hit_file'))} / {_pct((df.get(k) or {}).get('hit_func'))} |")
    T += ["", "S-use, share of samples (2 per task).", "",
          "| size | cond | edits gold file now | applies | on target | exact | exact given edit | tasks with an exact edit | dev edit now / exact |",
          "|---|---|---|---|---|---|---|---|---|"]
    for k, e in A["R5_depth2"]["use"].items():
        T.append(f"| {k.split('/')[0]} | {k.split('/')[1]} | {_pct(e['edit_now'])} | {_pct(e['applies'])} | {_pct(e['on_target'])} | "
                 f"{_pct(e['exact'])} | {_pct(e['exact_given_edit'])} | {_pct(e['tasks_any_exact'])} | "
                 f"{_pct((du.get(k) or {}).get('edit_now'))} / {_pct((du.get(k) or {}).get('exact'))} |")
    T += ["", "## Depth 1, every contrast (descriptive)", "", "| contrast | " + " | ".join(SIZES) + " | dev " + " | dev ".join(SIZES) + " |",
          "|---|" + "---|" * (2 * len(SIZES))]
    for (st, a, b) in CONTRASTS:
        k = f"{st}/{a}-{b}"
        T.append(f"| {st} {a}−{b} | " + " | ".join(_ci(A["depth1"].get(f"{s}/{k}")) for s in SIZES) + " | "
                 + " | ".join(_ci(dev.get("depth1", {}).get(f"{s}/{k}")) for s in SIZES) + " |")
    T += ["", f"Cell health: {json.dumps(A['health'])}", ""]
    return "\n".join(T)


def figures(A: dict, dev: dict | None, out: Path) -> list[str]:
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
    except Exception:  # noqa: BLE001
        return []
    dev = dev or {}
    rows = [("R1", "R1_claim1_find_vs_use", "d_find-minus-d_use", "Δ_find(L2−L0) − Δ_use(L2−L0)"),
            ("R2", "R2_claim1_words_and_diff", "d_use_W-L0", "Δ_use(W−L0)"),
            ("R2", "R2_claim1_words_and_diff", "d_use_L5-L0", "Δ_use(L5−L0)"),
            ("R3", "R3_claim3_graph_find", "d_find_G-json-G-null", "Δ_find(G-json−G-null)"),
            ("R3", "R3_claim3_graph_find", "d_find_L2-X-func", "Δ_find(L2−X-func)"),
            ("R4", "R4_claim3_graph_use", "d_use_G-json-G-null", "Δ_use(G-json−G-null)")]
    colors = {"e4b": "#4C78A8", "12b": "#F58518", "31b": "#54A24B"}
    fig, ax = plt.subplots(figsize=(8.5, 5.2))
    for i, (_, key, field, label) in enumerate(rows):
        for j, s in enumerate(SIZES):
            for src, mk, off in ((A, "o", -0.12), (dev, "s", 0.12)):
                e = ((src.get(key) or {}).get(s) or {}).get(field) or {}
                if not e.get("n"):
                    continue
                y = i + (j - 1) * 0.25 + off / 2
                ax.errorbar(e["median"], y, xerr=[[e["median"] - e["ci"][0]], [e["ci"][1] - e["median"]]], fmt=mk,
                            color=colors[s], mfc=colors[s] if src is A else "white", ms=5, capsize=2, lw=1)
    ax.axvline(0, color="#888", lw=0.8)
    ax.set_yticks(range(len(rows)))
    ax.set_yticklabels([f"{r[0]}  {r[3]}" for r in rows])
    ax.invert_yaxis()
    ax.set_xlabel("median Δ log p (nats), 95% bootstrap CI")
    ax.set_title("D1 held-out (filled) vs dev (open), per size")
    for s, c in colors.items():
        ax.plot([], [], "o", color=c, label=s)
    ax.legend(frameon=False, loc="lower right")
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
    fig.tight_layout()
    fig.savefig(out / "D1_fig1_criteria.png", dpi=150)
    plt.close(fig)
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
    for ax, (state, metric, conds, title) in zip(axes, (("find", "hit_file", D2_FIND, "S-find: reads the gold file"),
                                                         ("use", "exact", D2_USE, "S-use: exact edit (share of samples)"))):
        w = 0.13
        for j, s in enumerate(SIZES):
            for k, (src, hatch) in enumerate(((A, None), (dev, "//"))):
                vals = [((src.get("R5_depth2") or {}).get(state, {}).get(f"{s}/{c}") or {}).get(metric) for c in conds]
                xs = np.arange(len(conds)) + (2 * j + k - 2.5) * w
                ax.bar(xs, [v if v is not None else 0 for v in vals], w, color=colors[s], alpha=1.0 if k == 0 else 0.45,
                       hatch=hatch, edgecolor="white", label=f"{s} {'held-out' if k == 0 else 'dev'}")
        ax.set_xticks(range(len(conds)))
        ax.set_xticklabels(conds)
        ax.set_ylim(0, 1)
        ax.set_title(title, fontsize=10)
        for sp in ("top", "right"):
            ax.spines[sp].set_visible(False)
    axes[1].legend(frameon=False, fontsize=7, ncol=2)
    fig.tight_layout()
    fig.savefig(out / "D1_fig2_depth2.png", dpi=150)
    plt.close(fig)
    return ["D1_fig1_criteria.png", "D1_fig2_depth2.png"]


def main(d1_path: str, dev_path: str | None, out_dir: str = ".") -> dict:
    d = json.loads(Path(d1_path).read_text())
    dev = json.loads(Path(dev_path).read_text()) if dev_path and Path(dev_path).exists() else None
    A = analyze_rows(d)
    sel = d.get("selection") or {}
    meta = {"n_tasks": len(d.get("task_ids", [])), "by_repo": sel.get("selected_by_repo")}
    A.update({"plan": "docs/PLAN_D1_heldout_probe.md", "notebook": d.get("notebook"), "plan_commit": d.get("plan_commit"),
              "task_ids": d.get("task_ids"), "selection": sel, "session_elapsed_h": d.get("session_elapsed_h"),
              "stopped": d.get("stopped"), "depth2_run": d.get("depth2"), "dev_reference": bool(dev)})
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    A["figures"] = figures(A, dev, out)
    (out / "D1_analysis.json").write_text(json.dumps(A, indent=1))
    (out / "D1_tables.md").write_text(tables_md(A, dev, meta))
    print("D1 verdicts:", json.dumps(A["verdicts"], indent=1))
    return A


def dev_reference(c03_path: str, c04_path: str, out_path: str) -> dict:
    from pilot_c import analyze_c04 as a4
    c03, c04 = json.loads(Path(c03_path).read_text()), json.loads(Path(c04_path).read_text())
    d = a4.with_controls(a4.merge_design(c03, c04), c04)
    A = analyze_rows(d)
    A.pop("health", None)
    A.update({"source": "dev: results/C03_probe.json + results/C04_probe_followup.json (merged as analyze_c04)",
              "n_tasks": len(c03["task_ids"])})
    Path(out_path).write_text(json.dumps(A, indent=1))
    print("dev verdicts:", json.dumps(A["verdicts"], indent=1))
    return A


if __name__ == "__main__":
    if sys.argv[1] == "--dev-reference":
        dev_reference(*sys.argv[2:5])
    else:
        main(*sys.argv[1:4])

In [ ]:
%%writefile /kaggle/working/pilot/pilot_c/heldout.py
"""Held-out tasks for notebooks/D1_heldout_probe__L4x4.ipynb and notebooks/D2_heldout_rollouts_12b__L4x4.ipynb.

Reads pilot D's export (the output of notebooks/D0b_heldout_export__CPU.ipynb on pilot/d-heldout: `heldout_export/`,
docs/PROBE_INPUT.md Level A, plus `D0_summary.json`) and:
  1. merges it with the competition's agent and sandbox into one data directory, so pilot C's runner
     (pilot_c/kaggle_runner.py, which reads everything from one data directory) runs unchanged on held-out tasks;
  2. installs the export's test-only wheels that this image lacks (the subprocess sandbox uses the host's packages);
  3. selects the D1/D2 task set by the rule frozen in docs/D_SPECS.md (paper/main), rules 1-6.

The selection is pilot D's `pilot_d/select_d_tasks.py` (pilot/d-heldout 04dd8d1), vendored so the notebooks can inline it.
`select()` is that script's sampling code, unchanged. Two changes in what reaches it (coordinator, 2026-11-02):
  - repos in EXCLUDE_REPOS (sympy) are dropped by name, not only through the graph-size rule: sympy's graph stalls the
    harness's graph tools (O(N^2) load above ~25k nodes);
  - rule 4 (no 300 s hang in the no-patch run): D0b's output has no validation rows (it exports valid tasks only), so the
    hang tasks come from a validation file if one is attached, else from HANG_IDS, the hang tasks pilot D found in its
    local validation (results/heldout_pool_local.json on pilot/d-heldout).
Rule 3 is unchanged: the problem statement must come from an issue (`problem_statement_source` starts with "issue#").
PR-body statements often describe the fix, which would leak "where" and "what" into the baseline (L0) of the ladder.
Those tasks still count in pilot D's validity audit (claim 5); they are only kept out of D1/D2.

Outputs: D_selection.json (our own task ids and counts; safe to commit). Task content stays in memory / private/.
"""
from __future__ import annotations

import collections
import glob
import importlib.metadata as md
import json
import os
import random
import re
import subprocess
import sys
from pathlib import Path
from typing import Any

SEED = 20261102
MAX_NODES = 20000
EXCLUDE_REPOS = ("sympy/sympy",)
HANG_IDS = ("sqlglot_df5ec1ecd4", "sympy_30515")
RULE = "docs/D_SPECS.md (paper/main 02381e4) rules 1-6; selection code pilot_d/select_d_tasks.py (pilot/d-heldout 04dd8d1)"


# --------------------------------------------------------------------------- locating the inputs

def _input_root() -> str:
    return os.environ.get("PILOT_INPUT_ROOT", "/kaggle/input")


def find_export() -> Path:
    """The attached `heldout_export/` (found by its EXPORT.json, wherever Kaggle mounts D0b's output)."""
    hits = sorted(glob.glob(f"{_input_root()}/**/heldout_export/EXPORT.json", recursive=True))
    if not hits:
        raise FileNotFoundError("heldout_export/EXPORT.json not found under /kaggle/input: attach the output of "
                                "notebooks/D0b_heldout_export__CPU.ipynb (pilot/d-heldout) as an input first")
    return Path(hits[0]).parent


def find_optional(name: str) -> Path | None:
    hits = sorted(glob.glob(f"{_input_root()}/**/{name}", recursive=True))
    return Path(hits[0]) if hits else None


def competition_dir() -> Path:
    for c in ("/kaggle/input/competitions/gemma-4-developer-agent", "/kaggle/input/gemma-4-developer-agent"):
        if Path(c).is_dir():
            return Path(c)
    raise FileNotFoundError("competition gemma-4-developer-agent is not attached")


def merged_data_dir(export: Path, comp: Path, dst: Path) -> Path:
    """One data directory: task data from the export, agent and sandbox from the competition. Sets PILOT_DATA_DIR, so
    call it before kaggle_runner.set_env()."""
    dst.mkdir(parents=True, exist_ok=True)
    links = {"tasks.jsonl": export / "tasks.jsonl", "snapshots": export / "snapshots", "graphs": export / "graphs",
             "embeddings": export / "embeddings", "wheels": export / "wheels",
             "sample_submission": comp / "sample_submission", "sandbox": comp / "sandbox"}
    for name, src in links.items():
        p = dst / name
        if p.is_symlink() or p.exists():
            p.unlink()
        if src.exists():
            p.symlink_to(src.resolve())
    os.environ["PILOT_DATA_DIR"] = str(dst)
    return dst


def _norm(name: str) -> str:
    return re.sub(r"[-_.]+", "_", name).lower()


def install_test_wheels(export: Path) -> dict[str, Any]:
    """Install the export's test-only wheels whose distribution this image lacks (as D0b did), offline and without
    dependencies. Already-installed distributions are never replaced: the model-serving stack keeps its versions."""
    from pilot_c.kaggle_runner import _dist_of, _wheel_supported
    have = {_norm(d.metadata["Name"] or "") for d in md.distributions()}
    wheels = sorted((export / "wheels").glob("*.whl"))
    todo, seen = [], set()
    for w in wheels:
        dist = _dist_of(w.name)
        if dist in have or dist in seen or not _wheel_supported(w.name):
            continue
        seen.add(dist)
        todo.append(str(w))
    if todo:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--no-deps", *todo], check=True)
    return {"wheels_in_export": len(wheels), "installed": [Path(t).name for t in todo]}


# --------------------------------------------------------------------------- selection

def _jl(p: Path) -> list[dict]:
    return [json.loads(l) for l in open(p) if l.strip()] if p and Path(p).exists() else []


def graph_nodes(export: Path, task: dict) -> int:
    name = task["repo"].split("/")[1]
    for stem in (name, name.lower()):
        g = export / "graphs" / f"{stem}_{task['base_commit']}.json"
        if g.exists():
            return len(json.load(open(g))["nodes"])
    return 10 ** 9


def cheap_rules(export: Path, validation: list[dict] | None = None, hang_ids=HANG_IDS,
                exclude_repos=EXCLUDE_REPOS) -> tuple[list[str], dict[str, str], dict]:
    """Rules 1, 3, 4, 6 and the repo exclusion (no snapshot access). Returns (candidate ids, reason per excluded id,
    counts). Rule 1: the export holds valid tasks only; a validation file, if given, must also say valid."""
    tasks = {t["instance_id"]: t for t in _jl(export / "tasks.jsonl")}
    meta = {m["instance_id"]: m for m in _jl(export / "heldout_meta.jsonl")}
    val = {v["instance_id"]: v for v in (validation or [])}
    hang = set(hang_ids) | {i for i, v in val.items() if v.get("base_exit") == 124}
    cand, why = [], {}
    for i in sorted(tasks):
        src = (meta.get(i, {}).get("problem_statement_source") or "pending")
        if tasks[i]["repo"] in exclude_repos:
            why[i] = "excluded repo"
        elif val and not val.get(i, {}).get("valid"):
            why[i] = "not valid"
        elif not src.startswith("issue#"):
            why[i] = "statement not from an issue (" + src.split("#")[0] + ")"
        elif i in hang:
            why[i] = "300 s hang without a patch"
        elif graph_nodes(export, tasks[i]) > MAX_NODES:
            why[i] = f"graph > {MAX_NODES} nodes"
        else:
            cand.append(i)
    src_counts = collections.Counter((meta.get(i, {}).get("problem_statement_source") or "pending").split("#")[0] for i in tasks)
    counts = {"exported": len(tasks), "statement_source": dict(src_counts),
              "exported_by_repo": dict(collections.Counter(t["repo"] for t in tasks.values())),
              "excluded_before_records": dict(collections.Counter(why.values())), "candidates": len(cand)}
    return cand, why, counts


def eligible_any_repo(g: dict) -> bool:
    """pilot_c/select_tasks.eligible, the dev probe rule, without its dev-repo filter (D_SPECS rule 2). The rule itself is
    reused unchanged: only the repo field is set to a dev repo for the check."""
    from pilot_c.select_tasks import QUOTA, eligible
    return eligible({**g, "repo": next(iter(QUOTA))})


def probe_ready(ids: list[str]) -> tuple[dict[str, dict], dict[str, str]]:
    """Rule 2: C's eligibility rule (minus the dev-repo filter) and probe.build_probe_record == ok. Uses the merged data
    directory (PILOT_DATA_DIR). Returns ({id: probe record}, {id: reason})."""
    from pilot_c import kaggle_runner as kr
    from pilot_c import probe as pb
    d = kr.data_dir()
    tasks = {t["instance_id"]: t for t in kr.load_tasks(d / "tasks.jsonl")}
    gold = kr.build_gold(ids)
    recs, why = {}, {}
    for i in ids:
        if not eligible_any_repo(gold[i]):
            why[i] = "not eligible"
            continue
        rec, reason = pb.build_probe_record(tasks[i], gold[i], d / "snapshots" / f"{i}.tgz")
        if rec is None:
            why[i] = reason
        else:
            recs[i] = rec
    return recs, why


def select(pool: list[str], repo_of: dict[str, str], n: int = 24, seed: int = SEED) -> dict[str, Any]:
    """pilot_d/select_d_tasks.py's sampling, unchanged: rule 5 (>= 2 pool tasks per repo), then a seeded,
    repo-stratified round-robin sample of n, at most n // 2 per repo, from >= 3 repos."""
    by = collections.defaultdict(list)
    for i in sorted(pool):
        by[repo_of[i]].append(i)
    by = {r: ids for r, ids in by.items() if len(ids) >= 2}           # rule 5
    rng = random.Random(seed)
    for r in sorted(by):
        rng.shuffle(by[r])
    cap = n // 2
    chosen, repos = [], sorted(by, key=lambda r: (-len(by[r]), r))
    while len(chosen) < n and any(by[r] for r in repos):            # round-robin over repos, respecting the cap
        for r in repos:
            if by[r] and sum(repo_of[c] == r for c in chosen) < cap and len(chosen) < n:
                chosen.append(by[r].pop(0))
        if all(not by[r] or sum(repo_of[c] == r for c in chosen) >= cap for r in repos):
            break
    out = {"seed": seed, "n_target": n, "cap_per_repo": cap, "pool": len(pool),
           "pool_by_repo": dict(collections.Counter(repo_of[i] for i in pool)),
           "selected": sorted(chosen), "selected_by_repo": dict(collections.Counter(repo_of[i] for i in chosen)),
           "n_repos": len({repo_of[i] for i in chosen})}
    if out["n_repos"] < 3:
        out["warning"] = "fewer than 3 repos: the spec requires >= 3"
    return out


def select_tasks(export: Path, n: int = 24, validation_path: Path | None = None) -> tuple[dict[str, Any], list[dict]]:
    """Rules 1-6 and the sample. Returns (D_selection dict, probe records of the selected tasks, in id order)."""
    validation = _jl(validation_path) if validation_path else None
    cand, why0, counts = cheap_rules(export, validation)
    recs, why2 = probe_ready(cand)
    repo_of = {i: r["repo"] for i, r in recs.items()}
    sel = select(sorted(recs), repo_of, n)
    out = {"rule": RULE, "issue_rule": "problem_statement_source startswith 'issue#'",
           "exclude_repos": list(EXCLUDE_REPOS), "hang_ids": sorted(set(HANG_IDS)),
           "validation_file": str(validation_path) if validation_path else None, **counts,
           "not_probe_ready": dict(collections.Counter(why2.values())), "probe_ready": len(recs), **sel}
    return out, [recs[i] for i in sel["selected"]]

In [ ]:
%%writefile /kaggle/working/pilot/results/D1_dev_reference.json
{
 "R1_claim1_find_vs_use": {
  "e4b": {
   "d_find_L2-L0": {
    "n": 30,
    "median": 19.364,
    "ci": [
     17.485,
     24.087
    ],
    "mean": 20.835,
    "share_pos": 1.0,
    "p": 0.0
   },
   "d_use_L2-L0": {
    "n": 30,
    "median": 1.855,
    "ci": [
     1.006,
     3.372
    ],
    "mean": 2.334,
    "share_pos": 0.9,
    "p": 7e-06
   },
   "d_find-minus-d_use": {
    "n": 30,
    "median": 17.229,
    "ci": [
     13.739,
     19.771
    ],
    "mean": 18.501,
    "share_pos": 1.0,
    "p": 0.0
   },
   "p_holm": {
    "find>0": 0.0,
    "find>use": 0.0
   },
   "part1_location_helps_find": true,
   "holds": true
  },
  "12b": {
   "d_find_L2-L0": {
    "n": 30,
    "median": 25.279,
    "ci": [
     21.33,
     29.316
    ],
    "mean": 26.37,
    "share_pos": 1.0,
    "p": 0.0
   },
   "d_use_L2-L0": {
    "n": 30,
    "median": -1.767,
    "ci": [
     -3.347,
     1.945
    ],
    "mean": -1.123,
    "share_pos": 0.4,
    "p": 0.228553
   },
   "d_find-minus-d_use": {
    "n": 30,
    "median": 25.529,
    "ci": [
     23.822,
     30.672
    ],
    "mean": 27.493,
    "share_pos": 1.0,
    "p": 0.0
   },
   "p_holm": {
    "find>0": 0.0,
    "find>use": 0.0
   },
   "part1_location_helps_find": true,
   "holds": true
  },
  "31b": {
   "d_find_L2-L0": {
    "n": 30,
    "median": 33.406,
    "ci": [
     26.591,
     45.559
    ],
    "mean": 36.315,
    "share_pos": 1.0,
    "p": 0.0
   },
   "d_use_L2-L0": {
    "n": 30,
    "median": -1.529,
    "ci": [
     -3.012,
     0.637
    ],
    "mean": -1.741,
    "share_pos": 0.367,
    "p": 0.024786
   },
   "d_find-minus-d_use": {
    "n": 30,
    "median": 37.881,
    "ci": [
     27.318,
     46.943
    ],
    "mean": 38.055,
    "share_pos": 1.0,
    "p": 0.0
   },
   "p_holm": {
    "find>0": 0.0,
    "find>use": 0.0
   },
   "part1_location_helps_find": true,
   "holds": true
  }
 },
 "R2_claim1_words_and_diff": {
  "e4b": {
   "n_tasks": 27,
   "d_use_W-L0": {
    "n": 27,
    "median": 11.416,
    "ci": [
     7.244,
     15.375
    ],
    "mean": 13.087,
    "share_pos": 1.0,
    "p": 0.0
   },
   "d_use_L5-L0": {
    "n": 27,
    "median": 29.179,
    "ci": [
     22.552,
     35.661
    ],
    "mean": 44.08,
    "share_pos": 1.0,
    "p": 0.0
   },
   "d_use_L5-W": {
    "n": 27,
    "median": 18.818,
    "ci": [
     11.952,
     24.494
    ],
    "mean": 30.993,
    "share_pos": 0.963,
    "p": 0.0
   },
   "holds": true
  },
  "12b": {
   "n_tasks": 27,
   "d_use_W-L0": {
    "n": 27,
    "median": 14.424,
    "ci": [
     6.945,
     22.222
    ],
    "mean": 18.007,
    "share_pos": 0.926,
    "p": 0.0
   },
   "d_use_L5-L0": {
    "n": 27,
    "median": 42.398,
    "ci": [
     27.014,
     56.248
    ],
    "mean": 51.907,
    "share_pos": 0.963,
    "p": 0.0
   },
   "d_use_L5-W": {
    "n": 27,
    "median": 20.721,
    "ci": [
     12.874,
     33.369
    ],
    "mean": 33.9,
    "share_pos": 0.926,
    "p": 0.0
   },
   "holds": true
  },
  "31b": {
   "n_tasks": 27,
   "d_use_W-L0": {
    "n": 27,
    "median": 11.638,
    "ci": [
     5.408,
     18.497
    ],
    "mean": 14.517,
    "share_pos": 0.926,
    "p": 7e-06
   },
   "d_use_L5-L0": {
    "n": 27,
    "median": 33.754,
    "ci": [
     22.63,
     48.877
    ],
    "mean": 50.138,
    "share_pos": 0.963,
    "p": 0.0
   },
   "d_use_L5-W": {
    "n": 27,
    "median": 21.43,
    "ci": [
     14.345,
     26.847
    ],
    "mean": 35.621,
    "share_pos": 0.889,
    "p": 1e-06
   },
   "holds": true
  }
 },
 "R3_claim3_graph_find": {
  "e4b": {
   "d_find_G-json-G-null": {
    "n": 30,
    "median": 6.488,
    "ci": [
     4.146,
     8.894
    ],
    "mean": 7.726,
    "share_pos": 0.967,
    "p": 0.0
   },
   "d_find_G-json-X-graph": {
    "n": 30,
    "median": 1.022,
    "ci": [
     0.242,
     1.831
    ],
    "mean": 1.253,
    "share_pos": 0.767,
    "p": 0.007111
   },
   "d_find_L2-X-func": {
    "n": 30,
    "median": 11.41,
    "ci": [
     8.948,
     14.279
    ],
    "mean": 12.079,
    "share_pos": 1.0,
    "p": 0.0
   },
   "holds": true
  },
  "12b": {
   "d_find_G-json-G-null": {
    "n": 30,
    "median": 9.832,
    "ci": [
     6.424,
     12.738
    ],
    "mean": 9.82,
    "share_pos": 0.867,
    "p": 2e-06
   },
   "d_find_G-json-X-graph": {
    "n": 30,
    "median": 1.507,
    "ci": [
     -0.232,
     2.54
    ],
    "mean": 1.211,
    "share_pos": 0.633,
    "p": 0.029326
   },
   "d_find_L2-X-func": {
    "n": 30,
    "median": 18.6,
    "ci": [
     15.673,
     23.322
    ],
    "mean": 19.906,
    "share_pos": 1.0,
    "p": 0.0
   },
   "holds": true
  },
  "31b": {
   "d_find_G-json-G-null": {
    "n": 30,
    "median": 4.612,
    "ci": [
     1.954,
     9.646
    ],
    "mean": 6.887,
    "share_pos": 0.833,
    "p": 5.6e-05
   },
   "d_find_G-json-X-graph": {
    "n": 30,
    "median": 1.158,
    "ci": [
     -0.509,
     4.047
    ],
    "mean": 3.054,
    "share_pos": 0.567,
    "p": 0.030985
   },
   "d_find_L2-X-func": {
    "n": 30,
    "median": 20.129,
    "ci": [
     17.56,
     25.982
    ],
    "mean": 20.62,
    "share_pos": 1.0,
    "p": 0.0
   },
   "holds": true
  }
 },
 "R4_claim3_graph_use": {
  "e4b": {
   "d_use_G-null-L0": {
    "n": 30,
    "median": 0.489,
    "ci": [
     -0.365,
     1.763
    ],
    "mean": 0.696,
    "share_pos": 0.633,
    "p": 0.073244
   },
   "d_use_G-json-L0": {
    "n": 30,
    "median": 1.173,
    "ci": [
     0.04,
     3.787
    ],
    "mean": 1.512,
    "share_pos": 0.7,
    "p": 0.011303
   },
   "d_use_G-json-G-null": {
    "n": 30,
    "median": 1.319,
    "ci": [
     0.352,
     2.326
    ],
    "mean": 0.816,
    "share_pos": 0.7,
    "p": 0.038418
   },
   "holds": true
  },
  "12b": {
   "d_use_G-null-L0": {
    "n": 30,
    "median": 9.343,
    "ci": [
     7.342,
     11.485
    ],
    "mean": 10.53,
    "share_pos": 0.967,
    "p": 0.0
   },
   "d_use_G-json-L0": {
    "n": 30,
    "median": 7.513,
    "ci": [
     4.572,
     9.339
    ],
    "mean": 7.419,
    "share_pos": 0.933,
    "p": 0.0
   },
   "d_use_G-json-G-null": {
    "n": 30,
    "median": -1.871,
    "ci": [
     -4.575,
     -0.695
    ],
    "mean": -3.111,
    "share_pos": 0.233,
    "p": 0.001038
   },
   "holds": true
  },
  "31b": {
   "d_use_G-null-L0": {
    "n": 30,
    "median": 8.678,
    "ci": [
     7.042,
     11.123
    ],
    "mean": 9.674,
    "share_pos": 0.967,
    "p": 0.0
   },
   "d_use_G-json-L0": {
    "n": 30,
    "median": 5.317,
    "ci": [
     1.836,
     6.383
    ],
    "mean": 4.936,
    "share_pos": 0.767,
    "p": 1.6e-05
   },
   "d_use_G-json-G-null": {
    "n": 30,
    "median": -4.393,
    "ci": [
     -6.484,
     -2.435
    ],
    "mean": -4.738,
    "share_pos": 0.233,
    "p": 7.1e-05
   },
   "holds": true
  }
 },
 "verdicts": {
  "R1": "replicated",
  "R2": "replicated",
  "R3": "replicated",
  "R4": "replicated"
 },
 "R5_depth2": {
  "find": {
   "e4b/L0": {
    "tasks": 30,
    "tool_call": 1.0,
    "hit_file": 0.367,
    "hit_func": 0.133,
    "followed_wrong": 0.0
   },
   "e4b/L2": {
    "tasks": 30,
    "tool_call": 1.0,
    "hit_file": 1.0,
    "hit_func": 0.967,
    "followed_wrong": 0.0,
    "hit_file_vs_L0": {
     "n": 30,
     "b_0to1": 19,
     "c_1to0": 0,
     "p": 4e-06
    },
    "hit_func_vs_L0": {
     "n": 30,
     "b_0to1": 25,
     "c_1to0": 0,
     "p": 0.0
    }
   },
   "e4b/G-json": {
    "tasks": 30,
    "tool_call": 1.0,
    "hit_file": 0.833,
    "hit_func": 0.333,
    "followed_wrong": 0.0,
    "hit_file_vs_L0": {
     "n": 30,
     "b_0to1": 14,
     "c_1to0": 0,
     "p": 0.000122
    },
    "hit_func_vs_L0": {
     "n": 30,
     "b_0to1": 6,
     "c_1to0": 0,
     "p": 0.03125
    }
   },
   "e4b/X-graph": {
    "tasks": 30,
    "tool_call": 1.0,
    "hit_file": 0.633,
    "hit_func": 0.333,
    "followed_wrong": 0.0,
    "hit_file_vs_L0": {
     "n": 30,
     "b_0to1": 8,
     "c_1to0": 0,
     "p": 0.007812
    },
    "hit_func_vs_L0": {
     "n": 30,
     "b_0to1": 6,
     "c_1to0": 0,
     "p": 0.03125
    }
   },
   "e4b/G-null": {
    "tasks": 30,
    "tool_call": 1.0,
    "hit_file": 0.2,
    "hit_func": 0.133,
    "followed_wrong": 0.2,
    "hit_file_vs_L0": {
     "n": 30,
     "b_0to1": 0,
     "c_1to0": 5,
     "p": 0.0625
    },
    "hit_func_vs_L0": {
     "n": 30,
     "b_0to1": 2,
     "c_1to0": 2,
     "p": 1.0
    }
   },
   "12b/L0": {
    "tasks": 30,
    "tool_call": 1.0,
    "hit_file": 0.2,
    "hit_func": 0.033,
    "followed_wrong": 0.0
   },
   "12b/L2": {
    "tasks": 30,
    "tool_call": 1.0,
    "hit_file": 1.0,
    "hit_func": 0.933,
    "followed_wrong": 0.0,
    "hit_file_vs_L0": {
     "n": 30,
     "b_0to1": 24,
     "c_1to0": 0,
     "p": 0.0
    },
    "hit_func_vs_L0": {
     "n": 30,
     "b_0to1": 27,
     "c_1to0": 0,
     "p": 0.0
    }
   },
   "12b/G-json": {
    "tasks": 30,
    "tool_call": 1.0,
    "hit_file": 0.7,
    "hit_func": 0.233,
    "followed_wrong": 0.0,
    "hit_file_vs_L0": {
     "n": 30,
     "b_0to1": 16,
     "c_1to0": 1,
     "p": 0.000275
    },
    "hit_func_vs_L0": {
     "n": 30,
     "b_0to1": 7,
     "c_1to0": 1,
     "p": 0.070312
    }
   },
   "12b/X-graph": {
    "tasks": 30,
    "tool_call": 1.0,
    "hit_file": 0.5,
    "hit_func": 0.233,
    "followed_wrong": 0.0,
    "hit_file_vs_L0": {
     "n": 30,
     "b_0to1": 9,
     "c_1to0": 0,
     "p": 0.003906
    },
    "hit_func_vs_L0": {
     "n": 30,
     "b_0to1": 6,
     "c_1to0": 0,
     "p": 0.03125
    }
   },
   "12b/G-null": {
    "tasks": 30,
    "tool_call": 0.933,
    "hit_file": 0.133,
    "hit_func": 0.133,
    "followed_wrong": 0.167,
    "hit_file_vs_L0": {
     "n": 30,
     "b_0to1": 1,
     "c_1to0": 3,
     "p": 0.625
    },
    "hit_func_vs_L0": {
     "n": 30,
     "b_0to1": 4,
     "c_1to0": 1,
     "p": 0.375
    }
   },
   "31b/L0": {
    "tasks": 30,
    "tool_call": 1.0,
    "hit_file": 0.433,
    "hit_func": 0.1,
    "followed_wrong": 0.0
   },
   "31b/L2": {
    "tasks": 30,
    "tool_call": 1.0,
    "hit_file": 0.933,
    "hit_func": 0.933,
    "followed_wrong": 0.0,
    "hit_file_vs_L0": {
     "n": 30,
     "b_0to1": 15,
     "c_1to0": 0,
     "p": 6.1e-05
    },
    "hit_func_vs_L0": {
     "n": 30,
     "b_0to1": 25,
     "c_1to0": 0,
     "p": 0.0
    }
   },
   "31b/G-json": {
    "tasks": 30,
    "tool_call": 1.0,
    "hit_file": 0.7,
    "hit_func": 0.2,
    "followed_wrong": 0.0,
    "hit_file_vs_L0": {
     "n": 30,
     "b_0to1": 8,
     "c_1to0": 0,
     "p": 0.007812
    },
    "hit_func_vs_L0": {
     "n": 30,
     "b_0to1": 3,
     "c_1to0": 0,
     "p": 0.25
    }
   },
   "31b/X-graph": {
    "tasks": 30,
    "tool_call": 1.0,
    "hit_file": 0.467,
    "hit_func": 0.167,
    "followed_wrong": 0.0,
    "hit_file_vs_L0": {
     "n": 30,
     "b_0to1": 1,
     "c_1to0": 0,
     "p": 1.0
    },
    "hit_func_vs_L0": {
     "n": 30,
     "b_0to1": 2,
     "c_1to0": 0,
     "p": 0.5
    }
   },
   "31b/G-null": {
    "tasks": 30,
    "tool_call": 1.0,
    "hit_file": 0.267,
    "hit_func": 0.1,
    "followed_wrong": 0.033,
    "hit_file_vs_L0": {
     "n": 30,
     "b_0to1": 0,
     "c_1to0": 5,
     "p": 0.0625
    },
    "hit_func_vs_L0": {
     "n": 30,
     "b_0to1": 0,
     "c_1to0": 0,
     "p": null
    }
   }
  },
  "use": {
   "e4b/L0": {
    "tasks": 30,
    "samples": 60,
    "tool_call": 1.0,
    "edit_now": 0.583,
    "applies": 0.517,
    "on_target": 0.267,
    "exact": 0.017,
    "exact_given_edit": 0.029,
    "tasks_any_exact": 0.033,
    "first_tools": {
     "edit_file": 35,
     "read_file": 20,
     "run_command": 5
    }
   },
   "e4b/L2": {
    "tasks": 30,
    "samples": 60,
    "tool_call": 1.0,
    "edit_now": 0.883,
    "applies": 0.817,
    "on_target": 0.55,
    "exact": 0.033,
    "exact_given_edit": 0.038,
    "tasks_any_exact": 0.067,
    "first_tools": {
     "edit_file": 53,
     "read_file": 7
    }
   },
   "e4b/L5": {
    "tasks": 27,
    "samples": 54,
    "tool_call": 1.0,
    "edit_now": 0.852,
    "applies": 0.722,
    "on_target": 0.704,
    "exact": 0.463,
    "exact_given_edit": 0.543,
    "tasks_any_exact": 0.519,
    "first_tools": {
     "edit_file": 46,
     "read_file": 8
    }
   },
   "12b/L0": {
    "tasks": 30,
    "samples": 60,
    "tool_call": 0.95,
    "edit_now": 0.033,
    "applies": 0.033,
    "on_target": 0.0,
    "exact": 0.0,
    "exact_given_edit": 0.0,
    "tasks_any_exact": 0.0,
    "first_tools": {
     "edit_file": 2,
     "get_code_neighbors": 4,
     "none": 3,
     "read_file": 33,
     "run_command": 18
    }
   },
   "12b/L2": {
    "tasks": 30,
    "samples": 60,
    "tool_call": 1.0,
    "edit_now": 0.017,
    "applies": 0.017,
    "on_target": 0.0,
    "exact": 0.0,
    "exact_given_edit": 0.0,
    "tasks_any_exact": 0.0,
    "first_tools": {
     "edit_file": 1,
     "get_code_neighbors": 1,
     "read_file": 34,
     "run_command": 24
    }
   },
   "12b/L5": {
    "tasks": 27,
    "samples": 54,
    "tool_call": 0.944,
    "edit_now": 0.296,
    "applies": 0.241,
    "on_target": 0.241,
    "exact": 0.185,
    "exact_given_edit": 0.625,
    "tasks_any_exact": 0.259,
    "first_tools": {
     "edit_file": 16,
     "none": 3,
     "read_file": 25,
     "run_command": 5,
     "write_file": 5
    }
   },
   "31b/L0": {
    "tasks": 30,
    "samples": 60,
    "tool_call": 1.0,
    "edit_now": 0.0,
    "applies": 0.0,
    "on_target": 0.0,
    "exact": 0.0,
    "exact_given_edit": null,
    "tasks_any_exact": 0.0,
    "first_tools": {
     "read_file": 32,
     "run_command": 21,
     "search_similar_code": 2,
     "write_file": 5
    }
   },
   "31b/L2": {
    "tasks": 30,
    "samples": 60,
    "tool_call": 1.0,
    "edit_now": 0.017,
    "applies": 0.017,
    "on_target": 0.017,
    "exact": 0.0,
    "exact_given_edit": 0.0,
    "tasks_any_exact": 0.0,
    "first_tools": {
     "edit_file": 1,
     "get_code_neighbors": 2,
     "read_file": 28,
     "run_command": 18,
     "write_file": 11
    }
   },
   "31b/L5": {
    "tasks": 27,
    "samples": 54,
    "tool_call": 1.0,
    "edit_now": 0.463,
    "applies": 0.463,
    "on_target": 0.407,
    "exact": 0.315,
    "exact_given_edit": 0.68,
    "tasks_any_exact": 0.407,
    "first_tools": {
     "edit_file": 25,
     "read_file": 18,
     "run_command": 6,
     "write_file": 5
    }
   }
  }
 },
 "depth1": {
  "e4b/find/L1-L0": {
   "n": 30,
   "median": 6.328,
   "ci": [
    3.246,
    8.647
   ],
   "mean": 7.71,
   "share_pos": 1.0,
   "p": 0.0
  },
  "e4b/find/L2-L0": {
   "n": 30,
   "median": 19.364,
   "ci": [
    17.485,
    24.087
   ],
   "mean": 20.835,
   "share_pos": 1.0,
   "p": 0.0
  },
  "e4b/find/W-L0": {
   "n": 27,
   "median": 2.752,
   "ci": [
    1.24,
    5.488
   ],
   "mean": 3.291,
   "share_pos": 0.778,
   "p": 1.1e-05
  },
  "e4b/find/G-json-L0": {
   "n": 30,
   "median": 11.011,
   "ci": [
    9.425,
    14.294
   ],
   "mean": 12.526,
   "share_pos": 1.0,
   "p": 0.0
  },
  "e4b/find/G-null-L0": {
   "n": 30,
   "median": 4.883,
   "ci": [
    3.684,
    5.742
   ],
   "mean": 4.8,
   "share_pos": 0.967,
   "p": 0.0
  },
  "e4b/find/X-func-L0": {
   "n": 30,
   "median": 7.428,
   "ci": [
    5.065,
    9.759
   ],
   "mean": 8.756,
   "share_pos": 0.867,
   "p": 0.0
  },
  "e4b/find/X-graph-L0": {
   "n": 30,
   "median": 9.541,
   "ci": [
    8.564,
    10.907
   ],
   "mean": 11.273,
   "share_pos": 1.0,
   "p": 0.0
  },
  "e4b/find/X-func-L2": {
   "n": 30,
   "median": -11.41,
   "ci": [
    -14.279,
    -8.948
   ],
   "mean": -12.079,
   "share_pos": 0.0,
   "p": 0.0
  },
  "e4b/find/X-graph-G-json": {
   "n": 30,
   "median": -1.022,
   "ci": [
    -1.831,
    -0.242
   ],
   "mean": -1.253,
   "share_pos": 0.233,
   "p": 0.007111
  },
  "e4b/find/G-json-G-null": {
   "n": 30,
   "median": 6.488,
   "ci": [
    4.146,
    8.894
   ],
   "mean": 7.726,
   "share_pos": 0.967,
   "p": 0.0
  },
  "e4b/find/L2-X-func": {
   "n": 30,
   "median": 11.41,
   "ci": [
    8.948,
    14.279
   ],
   "mean": 12.079,
   "share_pos": 1.0,
   "p": 0.0
  },
  "e4b/find/G-json-X-graph": {
   "n": 30,
   "median": 1.022,
   "ci": [
    0.242,
    1.831
   ],
   "mean": 1.253,
   "share_pos": 0.767,
   "p": 0.007111
  },
  "e4b/use/L2-L0": {
   "n": 30,
   "median": 1.855,
   "ci": [
    1.006,
    3.372
   ],
   "mean": 2.334,
   "share_pos": 0.9,
   "p": 7e-06
  },
  "e4b/use/W-L0": {
   "n": 27,
   "median": 11.416,
   "ci": [
    7.244,
    15.375
   ],
   "mean": 13.087,
   "share_pos": 1.0,
   "p": 0.0
  },
  "e4b/use/T-L0": {
   "n": 30,
   "median": 1.512,
   "ci": [
    0.33,
    3.459
   ],
   "mean": 4.065,
   "share_pos": 0.7,
   "p": 0.000123
  },
  "e4b/use/L5-L0": {
   "n": 27,
   "median": 29.179,
   "ci": [
    22.552,
    35.661
   ],
   "mean": 44.08,
   "share_pos": 1.0,
   "p": 0.0
  },
  "e4b/use/G-json-L0": {
   "n": 30,
   "median": 1.173,
   "ci": [
    0.04,
    3.787
   ],
   "mean": 1.512,
   "share_pos": 0.7,
   "p": 0.011303
  },
  "e4b/use/G-null-L0": {
   "n": 30,
   "median": 0.489,
   "ci": [
    -0.365,
    1.763
   ],
   "mean": 0.696,
   "share_pos": 0.633,
   "p": 0.073244
  },
  "e4b/use/L5-W": {
   "n": 27,
   "median": 18.818,
   "ci": [
    11.952,
    24.494
   ],
   "mean": 30.993,
   "share_pos": 0.963,
   "p": 0.0
  },
  "e4b/use/G-json-G-null": {
   "n": 30,
   "median": 1.319,
   "ci": [
    0.352,
    2.326
   ],
   "mean": 0.816,
   "share_pos": 0.7,
   "p": 0.038418
  },
  "12b/find/L1-L0": {
   "n": 30,
   "median": 8.199,
   "ci": [
    5.357,
    10.66
   ],
   "mean": 9.413,
   "share_pos": 0.933,
   "p": 0.0
  },
  "12b/find/L2-L0": {
   "n": 30,
   "median": 25.279,
   "ci": [
    21.33,
    29.316
   ],
   "mean": 26.37,
   "share_pos": 1.0,
   "p": 0.0
  },
  "12b/find/W-L0": {
   "n": 27,
   "median": 1.908,
   "ci": [
    0.714,
    4.808
   ],
   "mean": 2.994,
   "share_pos": 0.889,
   "p": 0.000301
  },
  "12b/find/G-json-L0": {
   "n": 30,
   "median": -1.604,
   "ci": [
    -5.667,
    0.049
   ],
   "mean": -1.111,
   "share_pos": 0.333,
   "p": 0.271006
  },
  "12b/find/G-null-L0": {
   "n": 30,
   "median": -9.497,
   "ci": [
    -13.996,
    -7.334
   ],
   "mean": -10.931,
   "share_pos": 0.033,
   "p": 0.0
  },
  "12b/find/X-func-L0": {
   "n": 30,
   "median": 4.557,
   "ci": [
    0.736,
    9.974
   ],
   "mean": 6.464,
   "share_pos": 0.7,
   "p": 0.004665
  },
  "12b/find/X-graph-L0": {
   "n": 30,
   "median": -3.514,
   "ci": [
    -7.336,
    1.514
   ],
   "mean": -2.322,
   "share_pos": 0.367,
   "p": 0.087939
  },
  "12b/find/X-func-L2": {
   "n": 30,
   "median": -18.6,
   "ci": [
    -23.322,
    -15.673
   ],
   "mean": -19.906,
   "share_pos": 0.0,
   "p": 0.0
  },
  "12b/find/X-graph-G-json": {
   "n": 30,
   "median": -1.507,
   "ci": [
    -2.54,
    0.232
   ],
   "mean": -1.211,
   "share_pos": 0.367,
   "p": 0.029326
  },
  "12b/find/G-json-G-null": {
   "n": 30,
   "median": 9.832,
   "ci": [
    6.424,
    12.738
   ],
   "mean": 9.82,
   "share_pos": 0.867,
   "p": 2e-06
  },
  "12b/find/L2-X-func": {
   "n": 30,
   "median": 18.6,
   "ci": [
    15.673,
    23.322
   ],
   "mean": 19.906,
   "share_pos": 1.0,
   "p": 0.0
  },
  "12b/find/G-json-X-graph": {
   "n": 30,
   "median": 1.507,
   "ci": [
    -0.232,
    2.54
   ],
   "mean": 1.211,
   "share_pos": 0.633,
   "p": 0.029326
  },
  "12b/use/L2-L0": {
   "n": 30,
   "median": -1.767,
   "ci": [
    -3.347,
    1.945
   ],
   "mean": -1.123,
   "share_pos": 0.4,
   "p": 0.228553
  },
  "12b/use/W-L0": {
   "n": 27,
   "median": 14.424,
   "ci": [
    6.945,
    22.222
   ],
   "mean": 18.007,
   "share_pos": 0.926,
   "p": 0.0
  },
  "12b/use/T-L0": {
   "n": 30,
   "median": 2.169,
   "ci": [
    -0.267,
    4.219
   ],
   "mean": 4.266,
   "share_pos": 0.633,
   "p": 0.004338
  },
  "12b/use/L5-L0": {
   "n": 27,
   "median": 42.398,
   "ci": [
    27.014,
    56.248
   ],
   "mean": 51.907,
   "share_pos": 0.963,
   "p": 0.0
  },
  "12b/use/G-json-L0": {
   "n": 30,
   "median": 7.513,
   "ci": [
    4.572,
    9.339
   ],
   "mean": 7.419,
   "share_pos": 0.933,
   "p": 0.0
  },
  "12b/use/G-null-L0": {
   "n": 30,
   "median": 9.343,
   "ci": [
    7.342,
    11.485
   ],
   "mean": 10.53,
   "share_pos": 0.967,
   "p": 0.0
  },
  "12b/use/L5-W": {
   "n": 27,
   "median": 20.721,
   "ci": [
    12.874,
    33.369
   ],
   "mean": 33.9,
   "share_pos": 0.926,
   "p": 0.0
  },
  "12b/use/G-json-G-null": {
   "n": 30,
   "median": -1.871,
   "ci": [
    -4.575,
    -0.695
   ],
   "mean": -3.111,
   "share_pos": 0.233,
   "p": 0.001038
  },
  "31b/find/L1-L0": {
   "n": 30,
   "median": 6.892,
   "ci": [
    3.639,
    14.919
   ],
   "mean": 11.513,
   "share_pos": 1.0,
   "p": 0.0
  },
  "31b/find/L2-L0": {
   "n": 30,
   "median": 33.406,
   "ci": [
    26.591,
    45.559
   ],
   "mean": 36.315,
   "share_pos": 1.0,
   "p": 0.0
  },
  "31b/find/W-L0": {
   "n": 27,
   "median": 1.046,
   "ci": [
    -0.704,
    6.565
   ],
   "mean": 4.033,
   "share_pos": 0.593,
   "p": 0.052081
  },
  "31b/find/G-json-L0": {
   "n": 30,
   "median": 7.409,
   "ci": [
    4.675,
    12.91
   ],
   "mean": 11.681,
   "share_pos": 0.967,
   "p": 0.0
  },
  "31b/find/G-null-L0": {
   "n": 30,
   "median": 4.234,
   "ci": [
    -0.197,
    8.497
   ],
   "mean": 4.794,
   "share_pos": 0.667,
   "p": 0.008143
  },
  "31b/find/X-func-L0": {
   "n": 30,
   "median": 10.649,
   "ci": [
    7.486,
    20.552
   ],
   "mean": 15.695,
   "share_pos": 0.967,
   "p": 0.0
  },
  "31b/find/X-graph-L0": {
   "n": 30,
   "median": 5.487,
   "ci": [
    1.288,
    12.278
   ],
   "mean": 8.627,
   "share_pos": 0.8,
   "p": 0.000232
  },
  "31b/find/X-func-L2": {
   "n": 30,
   "median": -20.129,
   "ci": [
    -25.982,
    -17.56
   ],
   "mean": -20.62,
   "share_pos": 0.0,
   "p": 0.0
  },
  "31b/find/X-graph-G-json": {
   "n": 30,
   "median": -1.158,
   "ci": [
    -4.047,
    0.509
   ],
   "mean": -3.054,
   "share_pos": 0.433,
   "p": 0.030985
  },
  "31b/find/G-json-G-null": {
   "n": 30,
   "median": 4.612,
   "ci": [
    1.954,
    9.646
   ],
   "mean": 6.887,
   "share_pos": 0.833,
   "p": 5.6e-05
  },
  "31b/find/L2-X-func": {
   "n": 30,
   "median": 20.129,
   "ci": [
    17.56,
    25.982
   ],
   "mean": 20.62,
   "share_pos": 1.0,
   "p": 0.0
  },
  "31b/find/G-json-X-graph": {
   "n": 30,
   "median": 1.158,
   "ci": [
    -0.509,
    4.047
   ],
   "mean": 3.054,
   "share_pos": 0.567,
   "p": 0.030985
  },
  "31b/use/L2-L0": {
   "n": 30,
   "median": -1.529,
   "ci": [
    -3.012,
    0.637
   ],
   "mean": -1.741,
   "share_pos": 0.367,
   "p": 0.024786
  },
  "31b/use/W-L0": {
   "n": 27,
   "median": 11.638,
   "ci": [
    5.408,
    18.497
   ],
   "mean": 14.517,
   "share_pos": 0.926,
   "p": 7e-06
  },
  "31b/use/T-L0": {
   "n": 30,
   "median": 0.403,
   "ci": [
    -0.489,
    2.26
   ],
   "mean": 4.005,
   "share_pos": 0.6,
   "p": 0.20541
  },
  "31b/use/L5-L0": {
   "n": 27,
   "median": 33.754,
   "ci": [
    22.63,
    48.877
   ],
   "mean": 50.138,
   "share_pos": 0.963,
   "p": 0.0
  },
  "31b/use/G-json-L0": {
   "n": 30,
   "median": 5.317,
   "ci": [
    1.836,
    6.383
   ],
   "mean": 4.936,
   "share_pos": 0.767,
   "p": 1.6e-05
  },
  "31b/use/G-null-L0": {
   "n": 30,
   "median": 8.678,
   "ci": [
    7.042,
    11.123
   ],
   "mean": 9.674,
   "share_pos": 0.967,
   "p": 0.0
  },
  "31b/use/L5-W": {
   "n": 27,
   "median": 21.43,
   "ci": [
    14.345,
    26.847
   ],
   "mean": 35.621,
   "share_pos": 0.889,
   "p": 1e-06
  },
  "31b/use/G-json-G-null": {
   "n": 30,
   "median": -4.393,
   "ci": [
    -6.484,
    -2.435
   ],
   "mean": -4.738,
   "share_pos": 0.233,
   "p": 7.1e-05
  }
 },
 "sizes_with_depth1": [
  "e4b",
  "12b",
  "31b"
 ],
 "sizes_with_depth2": [
  "e4b",
  "12b",
  "31b"
 ],
 "source": "dev: results/C03_probe.json + results/C04_probe_followup.json (merged as analyze_c04)",
 "n_tasks": 30
}

In [ ]:
# Held-out data (fails fast if D0b's output is not attached). Task data come from pilot D's export; the agent and the
# sandbox from the competition. The merged directory must exist before kr.set_env() in the next cell.
import os, sys, json
from pathlib import Path
sys.path.insert(0, '/kaggle/working/pilot')
from pilot_c import heldout as ho
EXPORT = ho.find_export()
_ex = json.loads((EXPORT / 'EXPORT.json').read_text())
print('held-out export:', EXPORT, {k: _ex.get(k) for k in ('tasks', 'repos', 'probe_eligible', 'needs_issue_text')})
_d0 = ho.find_optional('D0_summary.json')
print('D0_summary.json:', _d0, (json.loads(_d0.read_text()).get('valid_kaggle') if _d0 else 'not found'))
DATA = ho.merged_data_dir(EXPORT, ho.competition_dir(), Path('/kaggle/working/heldout_data'))
print('data dir:', DATA, sorted(p.name for p in DATA.iterdir()))

In [ ]:
import os, sys, time, json
from pathlib import Path
sys.path.insert(0, '/kaggle/working/pilot')
SESSION_T0 = time.time()
from pilot_c import kaggle_runner as kr
kr.set_env()
kr.install_wheelhouse(gpu=True, only=None)
kr.quiet_logs()
IDS = json.loads(Path('/kaggle/working/pilot/pilot_c/task_ids.json').read_text())
print(kr.env_info())

In [ ]:
# ---- configuration (only edit here) — docs/PLAN_D1_heldout_probe.md (frozen at 682a34f01fc90908e786b75845de382d9d2cce49)
PLAN_COMMIT = '682a34f01fc90908e786b75845de382d9d2cce49'
DEPTH2 = True                # main version. False = fallback version (depth 1 only); change nothing else
N_TASKS = int(os.environ.get('PILOT_D_N', 24))                # PILOT_D_N: local dry runs only
SIZES = ['gemma-4-31b-it-qat-w4a16-ct', 'gemma-4-e4b-it-qat-w4a16-ct', 'gemma-4-12b-it-qat-w4a16-ct']   # 31b writes the words
FIND = ('L0', 'L1', 'L2', 'W', 'G-json', 'G-null', 'X-func', 'X-graph')   # depth 1, S-find
USE = ('L0', 'L2', 'W', 'T', 'L5', 'G-json', 'G-null')                    # depth 1, S-use
D2_FIND = ('L0', 'L2', 'G-json', 'X-graph', 'G-null')                     # depth 2, 1 sample each
D2_USE = ('L0', 'L2', 'L5')                                               # depth 2, 2 samples each
RESERVE_MIN_PER_SIZE = 20    # depth 2 of a size starts only if this many minutes are left per remaining size
PROBE_GPU_UTIL = 0.75        # probe-server settings of C03/C04 (scheduling/memory only)
PROBE_VLLM_ARGS = ('--max-num-batched-tokens', '1024')
BUDGET = dict(timeout_seconds=120, max_tool_calls=40, max_time_minutes=15)   # capture runs the real agent config
MAX_SESSION_HOURS = 2.2 if DEPTH2 else 1.6
WORKERS = 16
from pilot_c import analyze_d1 as ad
assert (FIND, USE, D2_FIND, D2_USE) == (ad.FIND, ad.USE, ad.D2_FIND, ad.D2_USE), 'config differs from the frozen analysis'
print('version:', 'main (depth 1 + depth 2)' if DEPTH2 else 'fallback (depth 1 only)', '| target tasks:', N_TASKS)

In [ ]:
from pilot_c import probe_runner as pr
from pilot_c import probe as pb
from common import server_watchdog as wd
OUT = Path('/kaggle/working/D1_heldout_probe.json')
summary = {'pilot': 'C', 'notebook': 'notebooks/D1_heldout_probe__L4x4.ipynb', 'plan': 'docs/PLAN_D1_heldout_probe.md',
           'plan_commit': PLAN_COMMIT, 'env': kr.env_info(), 'depth2': DEPTH2,
           'export': {k: _ex.get(k) for k in ('format', 'tasks', 'repos', 'graph_generator', 'embeddings')},
           'sizes': [], 'd1': [], 'd2': [], 'freezes': [], 'stopped': None}
def save():
    summary['summary'] = pr.summarize(summary['d1'], summary['d2'])
    summary['session_elapsed_h'] = round((time.time() - SESSION_T0) / 3600, 3)
    OUT.write_text(json.dumps(summary, indent=1)); print('saved', OUT, summary['session_elapsed_h'], 'h')

# 1: task selection (CPU): docs/D_SPECS.md rules 1-6 + the seeded sample; the records are C's probe records
SEL, recs = ho.select_tasks(EXPORT, N_TASKS, ho.find_optional('val_subprocess.jsonl'))
Path('/kaggle/working/D_selection.json').write_text(json.dumps(SEL, indent=1))
print(json.dumps({k: v for k, v in SEL.items() if k != 'selected'}, indent=1))
summary['selection'] = SEL
assert len(recs) >= int(os.environ.get('PILOT_MIN_TASKS', 12)), f'only {len(recs)} held-out tasks qualify: stopping before any GPU work'
# 2: harness graph output (CPU) and the unrelated-graph control
pr.attach_graphs(recs)
pb.assign_null_graphs(recs)
summary['controls'] = {r['instance_id']: {k: v for k, v in r['wrong'].items() if k.endswith('ratio')} for r in recs}
summary['null_controls'] = {r['instance_id']: {k: r['null_graph'][k] for k in ('from', 'len_ratio', 'within_tol')}
                            for r in recs if r.get('null_graph')}
summary['outside_edits'] = {r['instance_id']: r['outside_edits'] for r in recs}
# 3: the agent's exact contexts with the real harness loop (CPU, no model)
ctxs = pr.capture_contexts(recs, BUDGET)
summary['context_checks'] = {r['instance_id']: pr.context_check(r, ctxs[r['instance_id']]) for r in recs if r['instance_id'] in ctxs}
summary['capture_missing'] = [r['instance_id'] for r in recs if r['instance_id'] not in ctxs]
recs = [r for r in recs if r['instance_id'] in ctxs]
summary['task_ids'] = [r['instance_id'] for r in recs]
kr.PRIVATE.mkdir(parents=True, exist_ok=True)
(kr.PRIVATE / 'probe_contexts.json').write_text(json.dumps(ctxs))
save()
assert recs, 'no contexts captured: stopping before any GPU work'

# 4: one vLLM server per size: depth 1, then depth 2 if time is left for the remaining sizes' depth 1.
# Each size is isolated (own try/finally, own output file). Freeze protection as in C03/C04.
deadline = SESSION_T0 + MAX_SESSION_HOURS * 3600
MAX_FREEZES, selfcheck_fail_streak = 2, 0
d1_cells = [(r, st, c, 1, {}) for r in recs for st, conds in (('find', FIND), ('use', USE)) for c in conds]
d2_cells = ([(r, 'find', c, 1) for r in recs for c in D2_FIND] + [(r, 'use', c, 2) for r in recs for c in D2_USE]) if DEPTH2 else []
print('cells per size: depth 1', len(d1_cells), '| depth-2 cells', len(d2_cells))

def retry_failed(rows, cells, depth, server, model, dl, samples_file=None):
    """One retry of the cells that failed with a request error (not unavailable, time guard or a frozen server)."""
    keep = [r for r in rows if r.get('ok') or r.get('why') in ('unavailable', 'time_guard')]
    ck = (lambda c: (c[0]['instance_id'], c[1], c[2], c[3])) if depth == 1 else (lambda c: (c[0]['instance_id'], c[1], c[2]))
    rk = (lambda r: (r['task'], r['state'], r['cond'], r.get('rep', 1))) if depth == 1 else (lambda r: (r['task'], r['state'], r['cond']))
    done = {rk(r) for r in keep}
    miss = [c for c in cells if ck(c) not in done]
    if not miss or wd.TRIPPED.is_set() or time.time() > dl - 120:
        return rows, 0
    print(f'retrying {len(miss)} depth-{depth} cells that failed with a request error')
    rr = pr.run_cells(server, model, recs, ctxs, dl, *((miss, []) if depth == 1 else ([], miss)), WORKERS, samples_file=samples_file)
    return keep + rr['d1' if depth == 1 else 'd2'], len(miss)

for k, model in enumerate(SIZES):
    size = kr.SHORT[model]
    if summary['stopped'] or time.time() > deadline - 600:
        summary['sizes'].append({'model': model, 'size': size, 'skipped': summary['stopped'] or 'session time guard'}); continue
    res, startup, server, dog = {'size': size, 'd1': [], 'd2': [], 'skipped': []}, None, None, None
    try:
        wd.reset()
        agent_dir = kr.prepare_agent_dir(model, BUDGET)
        server, models, adapters, startup = kr.start_server(agent_dir, gpu_memory_utilization=PROBE_GPU_UTIL,
                                                            extra_args=PROBE_VLLM_ARGS)
        dog = wd.Watchdog(kr.server_root(server), lambda srv=server, tag=size: kr.stop_server(srv, tag=tag))
        dog.start()
        res = pr.run_cells(server, model, recs, ctxs, deadline, d1_cells, [], WORKERS, write_words_here=(size == '31b'))
        res['d1'], res['retried_d1'] = retry_failed(res['d1'], d1_cells, 1, server, model, deadline)
        if d2_cells:
            d2_deadline = deadline - RESERVE_MIN_PER_SIZE * 60 * (len(SIZES) - 1 - k)
            if wd.TRIPPED.is_set():
                res['skipped'].append('depth2 (server frozen)')
            elif time.time() > d2_deadline - 300:
                res['skipped'].append('depth2 (time reserved for the remaining sizes)')
            else:
                r2 = pr.run_cells(server, model, recs, ctxs, d2_deadline, [], d2_cells, WORKERS,
                                  samples_file=f'probe_samples_D1_{size}.jsonl')
                res['d2'], res['retried_d2'] = retry_failed(r2['d2'], d2_cells, 2, server, model, d2_deadline,
                                                            f'probe_samples_D1_{size}_retry.jsonl')
                res['d2_minutes'], res['skipped'] = r2.get('d2_minutes'), res['skipped'] + r2['skipped']
    except kr.GpuMemoryNotReleased as e:
        res['error'] = f'GpuMemoryNotReleased: {e}'
        summary['stopped'] = 'GPU memory was not released by a stopped server'
    except Exception as e:
        res['error'] = f'{type(e).__name__}: {str(e)[:600]}'
    finally:
        if dog is not None:
            dog.stop()
        if server is not None:
            res['stop'] = kr.stop_server(server, tag=size)
        res['server_error_kinds'] = kr.error_kinds(kr.LAST_SERVER_ERRORS)
        if kr.LAST_SERVER_ERRORS and res.get('error'):
            print(f'[{size}] server log errors (full log in private/server_logs/):', *kr.LAST_SERVER_ERRORS[-8:], sep='\n  ')
    if wd.TRIPPED.is_set():
        res['server_frozen'] = True
        summary['freezes'].append({'model': model, **wd.state()})
        if len(summary['freezes']) >= MAX_FREEZES:
            summary['stopped'] = 'second inference-server freeze'
    selfcheck_fail_streak = selfcheck_fail_streak + 1 if 'SELFCHECK' in str(res.get('error', '')) else 0
    if selfcheck_fail_streak >= 2:
        summary['stopped'] = 'API self-check failed on two sizes in a row'
    d1, d2 = res.pop('d1'), res.pop('d2')
    entry = {'model': model, 'startup_s': round(startup, 1) if startup else None, **res}
    Path(f'/kaggle/working/D1_heldout_probe_{size}.json').write_text(json.dumps(
        {'pilot': 'C', 'notebook': 'notebooks/D1_heldout_probe__L4x4.ipynb', 'plan_commit': PLAN_COMMIT, 'size': size,
         'task_ids': summary['task_ids'], 'entry': entry, 'd1': d1, 'd2': d2, 'summary': pr.summarize(d1, d2)}, indent=1))
    summary['d1'] += d1; summary['d2'] += d2
    summary['sizes'].append(entry)
    save()
    if res.get('error'):
        print(f'[{size}] FAILED: {res["error"][:300]}')
save()
summary['words_written'] = sum(bool(r.get('words')) for r in recs)
with open(kr.PRIVATE / 'probe_records.jsonl', 'w') as f:   # with the words: input of notebooks/D1v_test_verify__CPU.ipynb
    for r in recs: f.write(json.dumps(pb.public_record(r)) + '\n')
save()
print('L4x4 hours used by this session (wall clock):', summary['session_elapsed_h'], '| freezes:', len(summary['freezes']),
      '| stopped:', summary['stopped'])

In [ ]:
# ---- standalone analysis: docs/PLAN_D1_heldout_probe.md §5, code pilot_c/analyze_d1.py (frozen with the plan; tested on
# synthetic data). Re-runnable offline: python -m pilot_c.analyze_d1 D1_heldout_probe.json results/D1_dev_reference.json <out>
from pilot_c import analyze_d1 as ad
AD = ad.main('/kaggle/working/D1_heldout_probe.json', '/kaggle/working/pilot/results/D1_dev_reference.json', '/kaggle/working')
print('figures:', AD['figures'])

**When it finishes:** download `D1_heldout_probe.json`, the three `D1_heldout_probe_<size>.json`, `D_selection.json`, `D1_analysis.json`, `D1_tables.md` and the two `D1_fig*.png` (numbers and our own task ids only) and commit them to `results/` on branch
`pilot/c-bottleneck`. Everything under `/kaggle/working/private/` (contexts, records, sampled edits, summaries) and
`/kaggle/working/heldout_data/` (links to the export) is task-derived: keep it on Kaggle, never commit it.